Plot the exported highres pickle files of NenuFAR

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import astropy.units as u
from astropy.time import Time
from datetime import datetime, timedelta
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm
from matplotlib.ticker import AutoMinorLocator
import matplotlib as mpl
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00' # use precise epoch
try: mdates.set_epoch('1970-01-01T00:00:00')
except: pass


data_dir    = '/home/mnedal/data/ORFEES'
folder_path = '/home/mnedal/data/pkl_files'
outputs     = '/home/mnedal/data/png'

In [ ]:
mydate = '20250326'

nenufar_files = sorted(glob.glob(f'{folder_path}/nenufar/*'))
for f in enumerate(nenufar_files):
    print(f)

In [ ]:
SRB_groupname = 'typeII' # typeIII_G1, typeIII_G2, typeIII_G3, typeIII_G4, typeII

g_files = [f for f in nenufar_files if mydate in f and f.endswith(f'{SRB_groupname}.pkl')] # find the filenames that end with G1
print(*g_files, sep='\n')

stokesI_filename  = [f for f in g_files if 'stokesI' in f][0]
stokesVI_filename = [f for f in g_files if 'stokesV_over_I' in f][0]

df_int = pd.read_pickle(stokesI_filename)
df_pol = pd.read_pickle(stokesVI_filename)

# Convert arb. unit to dB for Stokes I
df_int = 10 * np.log10(df_int) # Convert the amplitude to decibels

<b>Original cadence:</b>
* Frequency resolution: 6.10 kHz
* Time resolution: 20.97 ms

In [ ]:
dyspec_subtracted = df_int - np.tile(np.nanmedian(df_int,0), (df_int.shape[0],1))

print('Stokes I:')
print(f'Frequency resolution: {np.nanmedian(np.diff(df_int.columns)*1e3):.2f} kHz')
print(f'Time resolution: {np.nanmedian(np.diff(df_int.index)/np.timedelta64(1,"ms")):.2f} ms')
print(f'Data shape: {df_int.shape}\n')

print('Stokes V/I:')
print(f'Frequency resolution: {np.nanmedian(np.diff(df_pol.columns)*1e3):.2f} kHz')
print(f'Time resolution: {np.nanmedian(np.diff(df_pol.index)/np.timedelta64(1,"ms")):.2f} ms')
print(f'Data shape: {df_pol.shape}')

## Plot the highres dynamic spectra

In [ ]:
fig = plt.figure(figsize=[15,6])

ax = fig.add_subplot(121)
pc = ax.pcolormesh(dyspec_subtracted.index, dyspec_subtracted.columns, dyspec_subtracted.T,
                   vmin=0, vmax=np.percentile(dyspec_subtracted, 99),
                   cmap='Spectral_r')
fig.colorbar(pc, ax=ax, pad=0.02, label='dB (background subtracted)')
ax.set_xlabel(f'Time (UT) on {df_int.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_int.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_int.index[0].date()} 09:41:00'))

ax = fig.add_subplot(122)
pc = ax.pcolormesh(df_pol.index, df_pol.columns, df_pol.T,
                   vmin=-1, vmax=1,
                   cmap='seismic')
fig.colorbar(pc, ax=ax, pad=0.02, label='Polarization fraction')
ax.set_xlabel(f'Time (UT) on {df_pol.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_pol.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_pol.index[0].date()} 09:41:00'))
fig.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=[15,5])
ax = fig.add_subplot()
ax.pcolormesh(dyspec_subtracted.index, dyspec_subtracted.columns, dyspec_subtracted.T,
              vmin=0, vmax=np.nanpercentile(dyspec_subtracted, 98), cmap='Spectral_r')
ax.set_xlabel(f'Time (UT) on {df_int.index[0].date()}')
ax.set_ylabel('Frequency (MHz)')
ax.set_ylim(ax.get_ylim()[::-1])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# ax.set_xlim(left=pd.Timestamp(f'{df_int.index[0].date()} 09:36:00'),
#             right=pd.Timestamp(f'{df_int.index[0].date()} 09:41:00'))
fig.tight_layout()
plt.show()

---

# Full shock analysis of the type II radio burst &mdash; 26 March 2025 (NenuFAR)

Everything above loads and displays the NenuFAR high-resolution Stokes&nbsp;I and Stokes&nbsp;V/I
dynamic spectra. Everything below turns the type II burst in those spectra into a set of shock
characteristics.

**The burst.** One type II from one shock, seen in both plasma-emission harmonics. The
**fundamental** ($s=1$) enters the NenuFAR window at ~09:20&nbsp;UT near 73&nbsp;MHz and drifts
down until it leaves the bottom of the band around 09:33. The **harmonic** ($s=2$) sits at twice
the fundamental frequency, so it only enters the top of the band at ~09:26, once
$2f_F\lesssim85$&nbsp;MHz, and then follows the same shock out to ~09:50 at ~33&nbsp;MHz. The two
bands therefore appear offset along the time axis even though they are simultaneous &mdash; each is
visible only over the interval where its own frequency falls inside the observing window, and they
overlap around 09:26&ndash;09:33 where $f_H/f_F=2$ can be checked directly. Each band shows its own
band splitting and fine structure.

**Tracing.** There is **one interactive plot of the whole dynamic spectrum** and nothing is split
or windowed. You click along a lane, press **End trace**, and it is recorded under the current
label &mdash; `F lane 1`, `F lane 2`, `H lane 1`, and so on. Trace as many lanes per band as the
burst shows; the analysis works out which is which afterwards by sorting them in frequency, so the
lowest-frequency lane of a band is taken as the upstream branch of its split and the next as the
downstream branch. Nothing about the split has to be declared in advance.

**What the chain does**

1. Takes `dyspec_subtracted` as the tracing layer, restricted to the type II window and decimated
   in time so one interactive figure stays responsive.
2. You trace the lanes on that single figure; each lane can be traced more than once, and the
   scatter between repeats becomes an error bar.
3. Fits each lane $f(t)$ with a low-order polynomial in $\log_{10}f$, keeping the coefficient
   covariance, and Monte-Carlo samples it so the **fit error and the repeat error propagate
   together**.
4. Per band the split gives the density jump and the perpendicular-shock Alfv&eacute;n Mach number
$$X=\left(\frac{f_U}{f_L}\right)^2,\qquad
  M_A=\sqrt{\frac{X\,(X+5)}{2\,(4-X)}},$$
   both of which are **independent of the coronal density model** and are measured twice over, once
   from each band.
5. Each lane frequency is converted to a source height with its own harmonic number through five
   electron-density models at folds 1&ndash;4. Because $f_H=2f_F$ maps to the same local density,
   the two bands give **one continuous height-time track** covering the whole burst &mdash; longer
   than either band alone spans. Differentiating $r(t)$ gives $v_{\rm sh}$ and $a$, and
$$v_A=\frac{v_{\rm sh}}{M_A},\qquad B=v_A\sqrt{\mu_0\rho},\qquad \rho=\mu\, m_p\, n_e .$$
6. &sect;A.5 checks the fundamental/harmonic identification against the data: the frequency ratio
   over the overlap, the relative drift rates, the agreement of the two height tracks, and the
   contrast in circular polarisation between the bands from the **Stokes V/I** frame.
7. The joint height-time track is refit three ways &mdash; polynomial, Gallagher et&nbsp;al. (2003)
   and Byrne et&nbsp;al. (2013) &mdash; with bootstrap error bands, to show the kinematics are not
   an artefact of one fitting choice.
8. Everything is exported as CSV and LaTeX tables, with a summary paragraph of the
   measured values.

## A.0 &middot; Imports, constants and configuration

In [ ]:
# only what the shock analysis adds; everything else comes from the import cell at the top
import pickle
import textwrap
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import CubicSpline
from matplotlib.colors import Normalize, PowerNorm
from astropy.constants import R_sun, m_e, m_p, e, eps0, mu0, c

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

# physical constants taken from astropy rather than hardcoded
R_SUN_M    = R_sun.to('m').value
C_MS       = c.to('m/s').value
E_CHARGE_J = e.si.value
M_E        = m_e.value
M_P        = m_p.value
EPS0       = eps0.value
MU0        = mu0.value

# electron plasma frequency: f_p [Hz] = PLASMA_CONST * sqrt(n_e [cm^-3])
PLASMA_CONST = (1 / (2 * np.pi)) * np.sqrt(1e6 * E_CHARGE_J**2 / (EPS0 * M_E))
print(f'PLASMA_CONST = {PLASMA_CONST:.4g}   (f_pe [Hz] = PLASMA_CONST * sqrt(n_e [cm^-3]))')

In [ ]:
# ============================== CONFIGURATION ==============================
EVENT_DATE = str(df_int.index[0].date())
OUTDIR     = os.path.join(outputs, 'type2_nenufar')
os.makedirs(OUTDIR, exist_ok=True)
# a marker file stamped now: A.10 compares file times against it with the same clock and the
# same call, so the stale-output check cannot be thrown off by timezone or epoch conventions
_marker = os.path.join(OUTDIR, '.run_marker')
open(_marker, 'w').close()
RUN_START = os.path.getmtime(_marker)

# --- emission physics ---------------------------------------------------------
HARM = {'F': 1, 'H': 2}          # harmonic number s per band
BAND_NAME = {'F': 'Fundamental', 'H': 'Harmonic'}
BANDS = ['F', 'H']               # bands offered by the tracer, in this order
MU   = 1.27                      # mass per electron in units of m_p, for rho = MU m_p n_e.
                                 # 1.27 = 1.4/1.1 is the common choice in the type II literature.
                                 # Note it is NOT the fully-ionised 10% He value: with He doubly
                                 # ionised, n_e = 1.2 n_H and rho = 1.4 m_p n_H give 1.4/1.2 =
                                 # 1.167, which lowers B by 4.2% (B scales as sqrt(MU)). Quote
                                 # which convention was used; do not present 1.27 as the
                                 # fully-ionised value

# --- tracing ------------------------------------------------------------------
# Lanes are not declared in advance: you trace them on one plot and each finished trace is
# recorded as 'F lane 1', 'F lane 2', 'H lane 1', ... Within a band the lanes are sorted by
# frequency afterwards, so the lowest is taken as the upstream branch of the split and the next
# as the downstream branch. Trace each lane N_REPS times to get a reproducibility error bar.
N_REPS = 3

TRACE_METHOD = 'click'           # which method the tracer opens with; switch it live in the
                                 # widget at any time. 'click'  -> click points along the lane
                                 #                     'bezier' -> place a Bezier curve on it

# --- Bezier tracing -----------------------------------------------------------
BEZIER_ANCHORS    = 2            # 1 -> quadratic (1 anchor), 2 -> cubic (2 anchors). Cubic by
                                 # default: these lanes span close to an octave and a quadratic
                                 # cannot follow them closely enough to recover the band split
BEZIER_NUM_POINTS = 80           # samples taken along the curve when it is recorded

# The auto-repeats displace the control points of the curve you placed. Set the size of that
# displacement as a FREQUENCY, not as a pixel count: a pixel means nothing physical and its value
# in MHz changes with LAYER_MAX_F and with the frequency window. BEZIER_JITTER (in channels) is
# derived from this in A.1, once the channel width of the tracing layer is known.
#
# 0.4 MHz is the scale on which a curve can be placed ambiguously on one of these band-split
# branches - comparable to the trace-to-ridge offset that A.3 measures - so the repeats span a
# plausible range of curves rather than an arbitrary one, and the spread is large enough to see.
#
# READ THIS BEFORE QUOTING AN ERROR BAR: the spread between jittered repeats is how far this knob
# displaced the curve. It is not a measurement of how reproducibly a curve can be placed on the
# ridge, and it is a lower bound on the tracing error, nothing more. The frequency error the lane
# fits use is LANE_SIGMA, measured from the trace-to-ridge offset in A.3. For a genuine
# reproducibility error, switch auto-repeats off and place the curve from scratch N_REPS times.
BEZIER_JITTER_MHZ = 0.4          # target 1-sigma frequency displacement of the auto-repeats
BEZIER_JITTER     = None         # channels; None -> derived from BEZIER_JITTER_MHZ in A.1
BEZIER_SEED       = 0

TYPEII_WINDOW = (pd.Timestamp(f'{EVENT_DATE} 09:18:00'), pd.Timestamp(f'{EVENT_DATE} 09:52:00'))
TYPEII_FLIM   = [25, 85]         # MHz, the part of the NenuFAR band the burst occupies

# --- decimation of the tracing layer -----------------------------------------
# the exported grid (~123k x 640 for this event) is far finer in time than the tracing needs and
# too large to percentile-scale and pcolormesh interactively, so the layer is block-averaged down
# to at most this size. Leave LAYER_MAX_F generous: the band split and the fine structure live
# in frequency, so nothing should be averaged there
LAYER_MAX_T = 4000               # time samples kept in the tracing layer
LAYER_MAX_F = 1200               # frequency channels kept

# --- display stretch of the tracing layer -------------------------------------
# Keep LAYER_PLO low: clipping the bottom of the distribution is what erases the faint
# background and the fine structure. LAYER_GAMMA < 1 applies a power-law stretch that brightens
# the faint end without throwing it away, which is the gentler way to bring detail up.
LAYER_MODE   = 'db_sub'          # 'db_sub' -> dyspec_subtracted, i.e. dB above the background
                                 # 'ratio'  -> linear ratio to the background, log colour scale
LAYER_CMAP   = 'Spectral_r'
LAYER_PLO    = 2                 # lower percentile of the colour scale
LAYER_PHI    = 98                # upper percentile of the colour scale
LAYER_GAMMA  = 0.6               # <1 brightens faint features, 1 = plain linear stretch
INVERT_FREQ  = True              # low frequency at the top, as in the figures above
PREVIEW_MAX_TCOLS = 1600         # further time down-sampling for the previews only

# --- density-model grid -------------------------------------------------------
FOLDS = [1, 2, 3, 4]
REF_MODEL_NAME = 'Newkirk x2'    # model x fold used for the reference figures and tables
R_BOUNDS = (1, 5)            # heliocentric range over which n_e(r) is inverted

# --- what is analysed ----------------------------------------------------------
# Every traced lane is analysed on its own. MAKE_JOINT additionally builds an 'F+H joint' track
# that averages the two bands' UPSTREAM lanes where they overlap and takes whichever exists
# elsewhere. That buys time coverage (a second derivative needs a long baseline) but it mixes two
# independent measurements into one curve, so it is off by default.
MAKE_JOINT = False
REF_LANE   = None                # lane the height-time fits and the model sweep use.
                                 # None -> the upstream lane of the first band traced

# --- Monte-Carlo and height-time fitting ---------------------------------------
N_MC      = 100                  # MC draws per pass for the fit error
POLY_DEG  = 2                    # polynomial degree for the HEIGHT-TIME fits in A.7. Kept at 2:
                                 # a cubic in r(t) can bend a height track into an S and then
                                 # report a spurious 1000 m/s^2 near the ends
LANE_DEG  = 3                    # polynomial degree for the LANE fit, log10 f against t.
                                 # A quadratic is not flexible enough here, and the shortfall is
                                 # not cosmetic: push a shock moving at EXACTLY constant speed
                                 # through this chain with LANE_DEG = 2 and it comes back with a
                                 # spurious deceleration of -5 to -54 m/s^2, because the misfit
                                 # between a quadratic and the true log f(t) of a constant-speed
                                 # shock in a structured corona turns into curvature in r(t).
                                 # At 3 that bias drops to ~1 m/s^2. A.5 measures what is left
                                 # for your actual lanes rather than trusting this comment
KIN_MIN_PTS = 6                  # minimum finite height points before v and a are computed
KIN_MIN_BASELINE_S = 600         # ADVISORY ONLY. A short traced baseline constrains curvature
                                 # weakly, and this is the span below which that is worth saying
                                 # out loud. It is NOT a gate: a fixed cut here turns a four-second
                                 # difference in traced length into the presence or absence of a
                                 # measurement, which is indefensible. The lane is always fitted
                                 # at KIN_DEG when it has the points for it, and whether the
                                 # curvature means anything is judged by KIN_A_SIGMA below
KIN_A_SIGMA = 2                  # |a| must exceed this many times its own uncertainty to count as
                                 # measured. A degenerate quadratic over a short baseline does not
                                 # need to be forbidden - it reports a large error bar and fails
                                 # this test on its own, which is both more honest and more
                                 # informative than refusing to fit it
KIN_N_DENSE = 200                # samples along a lane's OWN span used to fit r(t). The shared
                                 # N_GRID grid spans the whole burst, so a short lane gets only a
                                 # handful of its points; the lane fit is analytic, so it can be
                                 # sampled as finely as the fit needs
KIN_MIN_FRAC = 0.5               # of a lane's OWN traced span the density model must resolve.
                                 # Judged against the lane's span, not the shared grid: a short
                                 # lane is a legitimate measurement, it just covers less time

# --- how the frequency uncertainty of a traced lane is set -----------------------
# A Bezier is sampled at BEZIER_NUM_POINTS along a curve with only ~8 free parameters, so those
# samples are nowhere near independent and np.polyfit's covariance comes out far too small.
# Thinning to one point per FIT_MIN_DT_S seconds removes the fake sample count, and giving each
# point an explicit frequency error (from the QC offset from the ridge) makes the covariance
# mean something rather than just tracking how smooth the curve happens to be.
FIT_MIN_DT_S    = 10             # s, minimum spacing of the points a lane fit actually uses
# 'auto' -> per-lane rms offset from the ridge (A.3 QC); a number -> use that in MHz; None -> no
# per-point weighting. Read the window-dependence test A.3 prints before quoting any error bar
# derived from 'auto'. A brightest-pixel search has no intrinsic scale, so on a spectrum as
# structured as this one the rms offset keeps growing as the search window is widened and the
# resulting sigma_f is set largely by that window, not by a measured trace-to-ridge distance.
# It is a defensible order of magnitude - about one channel to a few - and it is NOT a
# reproducibility error. Every statistical error bar in this notebook scales linearly with it,
# so a factor-of-two view of sigma_f is a factor-of-two view of every sigma quoted here. What
# the A.3 control test does establish is that the traces follow emission at all.
LANE_SIGMA_MHZ  = 'auto'
LANE_SIGMA_FLOOR = 0.2           # MHz, floor on that uncertainty

# --- sanity thresholds used by the consistency audit in A.5 ----------------------
QC_MIN_SNR_DB   = 3              # a trace sitting on less than this above background is not on
                                 # the burst. The offset-from-ridge check alone cannot catch that:
                                 # over empty sky the brightest pixel within the search window is
                                 # noise, which lands near zero offset by chance
QC_CONTROL_SHIFT = 40            # channels. A.3 repeats the ridge measurement with every lane
                                 # displaced this far in frequency, onto blank spectrum, as a
                                 # control: if the on-lane rms is not clearly lower than the
                                 # displaced one, the traces are not following the burst
CONSIST_SIGMA   = 3              # flag a disagreement beyond this many combined sigma
A_PLAUSIBLE_MS2 = 200            # |a| above this is worth questioning for a coronal shock
LANE_FIT_MAX_RESID = 3           # flag a lane whose polynomial leaves an rms residual more than
                                 # this many LANE_SIGMA from the traced points. A Bezier that the
                                 # fit cannot follow makes every derived quantity meaningless,
                                 # and nothing else in the chain notices
REPORT_EXCITER_ENERGY = False    # (gamma-1) m_e c^2 at the shock speed is ~1 eV, far below the
                                 # coronal thermal energy, and is not a property of the emitting
                                 # electrons. Off by default: it can only mislead in a table
INFLATE_BY_CHI2 = False          # if True, widen the height-time bands by sqrt(chi2_red) when the
                                 # fit is worse than the errors imply. OFF by default: for
                                 # comparing models you want to SEE that a band fails to cover the
                                 # data, not have it inflated until it does. chi2_red is reported
                                 # separately either way
FIT_CLIP_TO_TRACED_F = True      # blank any fitted sample whose FREQUENCY falls outside the
                                 # range actually traced on that lane, not just outside its time
                                 # span. Without it the polynomial extrapolates at the lane ends -
                                 # far enough on the harmonic here to leave the analysis band and
                                 # send the height track backwards
FIT_F_TOL_MHZ = 0.05             # tolerance on that clip, so a legitimate endpoint is not lost to
                                 # rounding. Well under LANE_SIGMA
FIT_IN_LOGF = True               # fit log10 f(t) rather than f(t). A type II lane spans close to
                                 # an octave here, over which a quadratic in f is far too stiff -
                                 # the drift is much nearer log-linear. Set False for the plain fit
KIN_DEG   = 2                    # degree of the r(t) fit the speed and acceleration come from.
                                 # 2 by default: a cubic can bend a height track into an S and
                                 # then reports a spurious 1000 m/s^2 near the ends
N_BOOT    = 300                  # bootstrap refits for the height-time error bands
N_GRID    = 60                   # points on the shared time grid

# --- Stokes V/I sampling along the lanes ----------------------------------------
POL_DT_S   = 2                   # +/- seconds of the box the V/I median is taken over
POL_DF_MHZ = 0.5                 # +/- MHz of that box

JOINT = 'F+H joint'              # key of the combined fundamental + harmonic track

# --- export of the full model grid for comparison with EUV / coronagraph tracks ----------------
EXPORT_DT_S  = 10                # cadence of the exported height-time tracks [s]
EXPORT_N_MC  = 40                # MC draws per pass for the exported error column. Lower than
                                 # N_MC because this runs over all 20 model x fold combinations
                                 # and the export is for overplotting, not for the headline numbers
R_LIMB_RSUN  = 1                 # heliocentric radius of the limb. Coronagraph and EUV heights are
                                 # often quoted as height ABOVE THE LIMB, radio heights almost
                                 # always as heliocentric r. The export carries both columns
                                 # explicitly so the two can never be silently mixed - a 1 Rsun
                                 # offset is larger than the entire spread between density models


def save_fig(fig, name, dpi=300):
    path = os.path.join(OUTDIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('saved', path)


print(f'event {EVENT_DATE}')
print('bands offered: ' + ', '.join(f'{b} = {BAND_NAME[b]} (s={HARM[b]})' for b in BANDS))
print(f'{N_REPS} repeat(s) per lane')
print('outputs ->', OUTDIR)

## A.1 &middot; The tracing layer and its display stretch

`dyspec_subtracted` computed above is already the right quantity: `df_int` in dB minus the
time-median of every frequency channel, which removes the instrumental bandpass and the quiet-Sun
level and leaves the burst as an excess in dB. It is reused directly rather than recomputed.
Setting `LAYER_MODE = 'ratio'` instead undoes the dB conversion and forms the linear ratio to that
background, displayed on a log colour scale.

The layer is **block-averaged in time** down to at most `LAYER_MAX_T` samples, in chunks so the
full-resolution frame is never copied whole. At the exported 20.97&nbsp;ms cadence the type II
window alone holds of order $10^8$ samples, which no interactive figure can redraw at a usable
rate. `LAYER_MAX_F` is deliberately larger than the 640 channels of this export, so **nothing is
averaged in frequency** &mdash; the band splitting and the fine structure are measured there.

**On the colour stretch.** Two things control how much faint detail survives. `LAYER_PLO` sets the
bottom of the colour scale as a percentile: anything below it is clipped flat, so a high value
erases the background and with it the fine structure. `LAYER_GAMMA` applies a power-law stretch
($\gamma<1$ brightens the faint end without discarding it), which is the gentler instrument for
this. The cell below shows several combinations side by side so you can pick one; the defaults are
`LAYER_PLO = 2`, `LAYER_PHI = 98`, `LAYER_GAMMA = 0.6`.

In [ ]:
def decimate(df, max_t=LAYER_MAX_T, max_f=LAYER_MAX_F, chunk=20000):
    """Block-average a (time x frequency) frame down to at most max_t x max_f samples, using the
    block centres as the new time and frequency coordinates. Averaged in time chunks so the
    full-resolution frame is never copied whole - the type II window holds ~1e8 samples at the
    native NenuFAR cadence. Returns the frame unchanged when it is already small enough."""
    nt, nf = df.shape
    kt = max(1, int(np.ceil(nt / max_t)))
    kf = max(1, int(np.ceil(nf / max_f)))
    if kt == 1 and kf == 1:
        return df, (1, 1)
    nt2, nf2 = (nt // kt) * kt, (nf // kf) * kf
    rows = max(kt, (chunk // kt) * kt)
    blocks = []
    for i0 in tqdm(range(0, nt2, rows), desc='decimating', leave=False):
        i1 = min(i0 + rows, nt2)
        v = df.iloc[i0:i1, :nf2].to_numpy(float)
        blocks.append(np.nanmean(v.reshape((i1 - i0) // kt, kt, nf2 // kf, kf), axis=(1, 3)))
    ti = df.index[:nt2].to_numpy().astype('int64').reshape(-1, kt).mean(axis=1)
    fi = np.asarray(df.columns, float)[:nf2].reshape(-1, kf).mean(axis=1)
    return pd.DataFrame(np.vstack(blocks), index=pd.to_datetime(ti.astype('int64')),
                        columns=fi), (kt, kf)


def window(df):
    """Slice a (time x frequency) frame to the type II window and frequency limits."""
    f = np.asarray(df.columns, float)
    keep = (f >= TYPEII_FLIM[0]) & (f <= TYPEII_FLIM[1])
    return df.loc[max(TYPEII_WINDOW[0], df.index[0]):min(TYPEII_WINDOW[1], df.index[-1]),
                  df.columns[keep]]


# Stokes I tracing layer, straight from the background-subtracted frame computed above
if LAYER_MODE == 'db_sub':
    src_I = dyspec_subtracted
else:
    lin = 10 ** (df_int / 10)
    src_I = lin / np.nanmedian(lin.to_numpy(), axis=0)

LAYER, (kt_I, kf_I) = decimate(window(src_I))
POL, (kt_P, kf_P) = decimate(window(df_pol))
LAYER_T = LAYER.index
LAYER_F = np.asarray(LAYER.columns, float)
LAYER_D = LAYER.to_numpy().T                       # (nfreq, ntime)

print(f'Stokes I layer: {LAYER.shape[0]} time x {LAYER.shape[1]} freq '
      f'(decimated {kt_I}x in time, {kf_I}x in frequency)')
print(f'  {LAYER_T[0].time()}-{LAYER_T[-1].time()} UT, {LAYER_F.min():.1f}-{LAYER_F.max():.1f} MHz,'
      f' cadence {np.nanmedian(np.diff(LAYER_T) / np.timedelta64(1, "ms")):.0f} ms,'
      f' channel {np.nanmedian(np.diff(LAYER_F)) * 1e3:.0f} kHz')
print(f'Stokes V/I layer: {POL.shape[0]} time x {POL.shape[1]} freq '
      f'(decimated {kt_P}x in time, {kf_P}x in frequency)')

# --- convert the target Bezier jitter from MHz into channels of this layer -------------------
# Jittering all four control points of a cubic independently by sigma does NOT move the curve by
# sigma: the curve is a weighted average of them, so the displacement is attenuated. For a cubic
# the rms attenuation over the curve is sqrt(int[(1-t)^6 + 9(1-t)^4 t^2 + 9(1-t)^2 t^4 + t^6] dt)
# = sqrt(0.4571) = 0.676. Divide it out so the achieved spread lands on the target.
CHAN_MHZ = float(np.nanmedian(np.diff(LAYER_F)))
BEZIER_CURVE_ATTEN = 0.676
if BEZIER_JITTER is None:
    BEZIER_JITTER = round(BEZIER_JITTER_MHZ / (CHAN_MHZ * BEZIER_CURVE_ATTEN), 1)
    print(f'\nBEZIER_JITTER = {BEZIER_JITTER:.1f} channels, derived from '
          f'BEZIER_JITTER_MHZ = {BEZIER_JITTER_MHZ} MHz')
    print(f'  ({CHAN_MHZ * 1e3:.0f} kHz per channel, divided by the {BEZIER_CURVE_ATTEN} cubic '
          'attenuation). The traced-lanes figure reports the spread actually achieved.')

In [ ]:
def layer_norm(D, plo=None, phi=None, gamma=None):
    """Colour normalisation for the tracing layer. A low `plo` keeps the faint background instead
    of clipping it flat, and gamma < 1 stretches the faint end up; the two together are what
    decide how much fine structure is visible."""
    plo = LAYER_PLO if plo is None else plo
    phi = LAYER_PHI if phi is None else phi
    gamma = LAYER_GAMMA if gamma is None else gamma
    vmin, vmax = np.nanpercentile(D, plo), np.nanpercentile(D, phi)
    if LAYER_MODE == 'ratio':
        return LogNorm(vmin=max(vmin, 1e-3), vmax=vmax)
    if gamma != 1:
        return PowerNorm(gamma, vmin=vmin, vmax=vmax)
    return Normalize(vmin=vmin, vmax=vmax)


LAST_STRETCH = {}


def draw_layer(ax, plo=2, phi=None, gamma=1, cmap=LAYER_CMAP, tstep=None):
    """Draw the whole tracing layer with pcolormesh in physical (UT, MHz) axes, down-sampled in
    time only for display speed. This is the one figure everything is traced on.

    plo and gamma default to 2 and 1: keep the faint end of the distribution, apply no power-law
    stretch. Pass None for either to fall back to LAYER_PLO / LAYER_GAMMA from the configuration
    cell, or pass explicit values (the stretch-comparison figure does exactly that)."""
    step = max(1, LAYER_D.shape[1] // PREVIEW_MAX_TCOLS) if tstep is None else tstep
    LAST_STRETCH.update(plo=(LAYER_PLO if plo is None else plo),
                        phi=(LAYER_PHI if phi is None else phi),
                        gamma=(LAYER_GAMMA if gamma is None else gamma))
    pm = ax.pcolormesh(LAYER_T[::step], LAYER_F, LAYER_D[:, ::step],
                       norm=layer_norm(LAYER_D, plo, phi, gamma), cmap=cmap, rasterized=True)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    if INVERT_FREQ:
        ax.invert_yaxis()
    ax.yaxis.set_minor_locator(AutoMinorLocator(n=5))
    ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
    ax.set_ylabel('Frequency (MHz)')
    return pm


CBAR_LABEL = 'dB above background' if LAYER_MODE == 'db_sub' else 'ratio to background'

# --- compare a few stretches: pick one and set LAYER_PLO / LAYER_PHI / LAYER_GAMMA above ---
TRIALS = [(60, 99.5, 1, 'plo=60, gamma=1  (harsh: faint detail clipped)'),
          (2, 98, 1, 'plo=2, gamma=1'),
          (2, 98, 0.6, 'plo=2, gamma=0.6  (default)'),
          (1, 99, 0.4, 'plo=1, gamma=0.4  (softest)')]
fig = plt.figure(figsize=[15, 12])
for i, (plo, phi, gam, ttl) in enumerate(TRIALS, start=1):
    ax = fig.add_subplot(len(TRIALS), 1, i)
    pm = draw_layer(ax, plo=plo, phi=phi, gamma=gam)
    fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)
    ax.set_title(ttl, fontsize=10)
    if i < len(TRIALS):
        ax.set_xlabel('')
fig.suptitle('Choosing the display stretch', y=1.005, fontsize=13)
fig.tight_layout()
save_fig(fig, 'display_stretch_comparison')
plt.show()

> **Figure 1 &mdash; Choosing the display stretch.** `display_stretch_comparison.png`
>
> The same tracing layer rendered with four colour scalings, from the harshest at the top to the
> softest at the bottom. `plo` is the percentile placed at the bottom of the colour bar, so
> everything below it is clipped flat; `gamma` is a power-law stretch applied after normalisation
> ($\gamma<1$ brightens the faint end without discarding it). Only the display changes &mdash; no
> panel here alters a single stored value.
>
> *Why it is here:* how much fine structure you can see is how much you can trace, and a high
> `plo` silently erases the faint band-split branches. Pick a row and set `LAYER_PLO`, `LAYER_PHI`
> and `LAYER_GAMMA` accordingly before tracing.

In [ ]:
fig = plt.figure(figsize=[15, 5])
ax = fig.add_subplot(111)
pm = draw_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)
ax.set_title(f'NenuFAR type II tracing layer  (plo={LAST_STRETCH["plo"]}, '
             f'phi={LAST_STRETCH["phi"]}, gamma={LAST_STRETCH["gamma"]})')
fig.tight_layout()
save_fig(fig, 'nenufar_typeii_window')
plt.show()

> **Figure 2 &mdash; The tracing layer.** `nenufar_typeii_window.png`
>
> The NenuFAR Stokes&nbsp;I dynamic spectrum over the type II window, as `dyspec_subtracted`
> (each frequency channel minus its own time median, so the instrumental bandpass and the quiet-Sun
> level are removed and the burst appears as an excess in dB). Frequency increases downwards, as in
> the overview panels above. This is the single image every lane is traced on.
>
> *How it was made:* the frame is sliced to `TYPEII_WINDOW` and `TYPEII_FLIM`, then block-averaged
> **in time only** to at most `LAYER_MAX_T` samples so one interactive figure stays responsive.
> Nothing is averaged in frequency &mdash; the band splitting and the fine structure live there.

## A.2 &middot; Electron-density models &times; fold numbers

Each model returns $n_e(r)$ in cm$^{-3}$ for heliocentric distance $r$ in solar radii. The **fold**
multiplies the whole profile (fold 1 = quiet Sun; 2&ndash;4 brackets a streamer or an active-region
corona), so the model&times;fold grid is the systematic uncertainty on every height-dependent
quantity. A frequency only has a height solution where the implied plasma density falls inside the
model's range; outside it the height is returned as NaN rather than extrapolated.

The panel on the right shows why the two bands are seen at different times. The emission at
$f=s\,f_{pe}$ leaves the observing window as soon as $s\,f_{pe}$ falls outside it, so for a shock
climbing through the corona the fundamental sweeps out through the bottom of the NenuFAR band while
the harmonic is still entering through the top &mdash; the same source, twice, offset in time.

**Where the Mann et&nbsp;al. (2023) constant comes from.** Their Eq.&nbsp;11 is written symbolically,
$N_i(r)=N_{i,\odot}\exp\!\left[\frac{2r_c}{R_\odot}\left(\frac{R_\odot}{r}-1\right)\right]$, so the
exponent has to be built from their Table&nbsp;5, which lists $r_c$ separately for each of the five
solutions A&ndash;D and M. Point&nbsp;M gives $r_c=3877$&nbsp;Mm and hence
$2r_c/R_\odot=11.146$, which is the value used here; it reproduces that paper's own quoted
$n_e(3\,R_\odot)=4.267\times10^5$ cm$^{-3}$ to better than 0.1%. Point&nbsp;D gives 3943&nbsp;Mm and
so 11.34, with a different base density, and the two sets must not be mixed.

In [ ]:
def newkirk(r, fold=1):
    """Newkirk (1961), exponential quiet-corona model."""
    r = np.asarray(r, float)
    return fold * 4.2e4 * 10 ** (4.32 / r)

def saito(r, fold=1):
    """Saito (1970), equatorial two-term fit."""
    r = np.asarray(r, float)
    return fold * (1.36e6 * r ** -2.14 + 1.68e8 * r ** -6.13)

def leblanc(r, fold=1):
    """Leblanc, Dulk & Bougeret (1998), normalised to 7.2 cm^-3 at 1 AU.

    Published as valid for r > 1.8 Rsun. Inside that it is an extrapolation, and at fold 1 it
    cannot represent a fundamental above ~82 MHz at all without pushing the source onto the
    photosphere - so the low end of any Leblanc x1 height range here should be read as a bound
    rather than a measurement."""
    r = np.asarray(r, float)
    return fold * (3.3e5 * r ** -2 + 4.1e6 * r ** -4 + 8e7 * r ** -6)

def baumbach_allen(r, fold=1):
    """Baumbach (1937) eclipse fit with the Allen (1947) correction, three-term."""
    r = np.asarray(r, float)
    return fold * 1e8 * (0.036 * r ** -1.5 + 1.55 * r ** -6 + 2.99 * r ** -16)

def mann2023(r, fold=1):
    """Mann, Warmuth, Vocks & Rouillard (2023), A&A 679, A64, barometric model.

    Their Eq. 11, N(r) = N_sun exp[(2 r_c / Rsun)(Rsun/r - 1)], is printed symbolically, so the
    exponent has to be taken from their Table 5, which gives r_c per solution. Point M has
    r_c = 3877 Mm, hence 2 r_c / Rsun = 11.146, and that value reproduces their own quoted
    n_e(3 Rsun) = 4.267e5 cm^-3 to better than 0.1% - which is the check that fixes it. Point D
    (r_c = 3943 Mm) gives 11.34 with a different base density, so the two sets must not be mixed.
    Valid to about 3 Rsun."""
    r = np.asarray(r, float)
    return fold * 7.17e8 * np.exp(11.14 * (1 / r - 1))


BASE_MODELS = {'Newkirk': newkirk, 'Saito': saito, 'Leblanc': leblanc,
               'Baumbach-Allen': baumbach_allen, 'Mann 2023': mann2023}

MODEL_GRID = {f'{name} x{fold}': (lambda r, _f=fun, _k=fold: _f(r, fold=_k))
              for name, fun in BASE_MODELS.items() for fold in FOLDS}
print(f'{len(BASE_MODELS)} models x {len(FOLDS)} folds = {len(MODEL_GRID)} combinations')


def freq_to_density(f_hz, harmonic=1):
    """Plasma frequency -> local electron density [cm^-3] for emission at harmonic s. Because the
    harmonic is emitted at 2 f_pe, dividing by s recovers the same local density from either band -
    which is what lets the two bands be joined into one height-time track."""
    return (np.asarray(f_hz, float) / harmonic / PLASMA_CONST) ** 2

def freq_to_radius(f_hz, model, harmonic=1, r_bounds=R_BOUNDS, nres=6000):
    """Invert a monotonically decreasing n_e(r) for the source height; out-of-range -> NaN."""
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    ne = model(rr)
    ne_t = freq_to_density(np.atleast_1d(f_hz), harmonic=harmonic)
    r = np.interp(ne_t, ne[::-1], rr[::-1])
    r[(ne_t > np.nanmax(ne)) | (ne_t < np.nanmin(ne))] = np.nan
    return r if r.size > 1 else float(r[0])

def alfven_mach_from_X(X, gamma=5/3, beta=0):
    """Perpendicular-shock Alfven Mach number from the density jump X (Rankine-Hugoniot).

    M_A^2 = X (X + 5 + 5 beta) / [2 (4 - X)] at gamma = 5/3, valid for 1 <= X < 4; NaN outside.
    The upper bound is (gamma+1)/(gamma-1), where M_A diverges: X = 3.99 already gives M_A = 42.

    gamma and beta are NOT free here. The coefficients 5 and 4 below are the gamma = 5/3 values
    and beta = 0 (cold upstream) is assumed, so anything else raises rather than silently
    returning the gamma = 5/3, beta = 0 answer - which is what this signature previously did."""
    if abs(gamma - 5 / 3) > 1e-12:
        raise ValueError(f'gamma = {gamma} is not supported: the closed form here is the '
                         'gamma = 5/3 case. Re-derive the coefficients for another gamma.')
    if beta != 0:
        raise ValueError(f'beta = {beta} is not supported: this is the cold-upstream form. '
                         'Finite beta lowers M_A; see the caveats in A.12.')
    X = np.asarray(X, float)
    out = np.full(X.shape, np.nan)
    ok = (X >= 1) & (X < 4)
    out[ok] = np.sqrt(X[ok] * (X[ok] + 5) / (2 * (4 - X[ok])))
    return out

def electron_energy_from_speed(v_kms):
    """Kinetic energy [keV] of an electron moving at v: E = (gamma - 1) m_e c^2. At a type II
    shock speed this is the bulk exciter energy and is firmly non-relativistic; the relativistic
    form is kept so the same estimator also serves fast type III beams."""
    v = np.asarray(v_kms, float) * 1e3
    beta = np.clip(v / C_MS, 0, 0.999999)
    gamma = 1 / np.sqrt(1 - beta ** 2)
    return (gamma - 1) * M_E * C_MS ** 2 / E_CHARGE_J / 1e3

In [ ]:
rr = np.linspace(1, 3, 400)
fig = plt.figure(figsize=[14, 5])

ax = fig.add_subplot(121)
for name, fun in BASE_MODELS.items():
    ax.plot(rr, fun(rr), lw=1.8, label=name)
ax.set_yscale('log')
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$n_e$ [cm$^{-3}$]')
ax.set_title('(a) Electron-density models (fold 1)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

ax = fig.add_subplot(122)
for b in BANDS:
    for name, fun in BASE_MODELS.items():
        ax.plot(rr, HARM[b] * PLASMA_CONST * np.sqrt(fun(rr)) / 1e6, lw=1.6,
                ls=('-' if b == 'F' else '--'), label=(name if b == 'F' else None))
ax.axhspan(TYPEII_FLIM[0], TYPEII_FLIM[1], color='0.8', alpha=0.5, zorder=0)
ax.text(2.85, np.sqrt(TYPEII_FLIM[0] * TYPEII_FLIM[1]), 'NenuFAR band', ha='right', fontsize=9)
ax.set_yscale('log')
ax.set_ylim(8, 400)
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$s\,f_{pe}$ [MHz]')
ax.set_title(r'(b) Emission frequency vs height: $s=1$ solid, $s=2$ dashed')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')

fig.tight_layout()
save_fig(fig, 'density_models')
plt.show()

> **Figure 3 &mdash; Coronal electron-density models.** `density_models.png`
>
> *(a)* The five analytic $n_e(r)$ profiles at fold&nbsp;1. *(b)* The emission frequency
> $s\,f_{pe}(r)$ each of them predicts, solid for the fundamental ($s=1$) and dashed for the
> harmonic ($s=2$), with the observing band shaded.
>
> *What to take from it:* panel (b) is the reason the two bands are seen at different times. As the
> shock climbs, the fundamental sweeps down out of the bottom of the band while the harmonic, at
> twice the frequency, is still entering through the top &mdash; the same source, twice, offset in
> time. The vertical spread between the curves at a fixed frequency is the height systematic that
> §A.8 quantifies.

## A.3 &middot; Trace the lanes

**One figure, two ways of laying down a lane.** The whole dynamic spectrum is drawn once and
nothing is split or windowed. A **Method** toggle switches between:

- **Click points** &mdash; left-click along the lane, one point at a time. Best for fine structure
  and for lanes that wander.
- **B&eacute;zier curve** &mdash; place a quadratic or cubic B&eacute;zier on the lane with sliders
  (or type an exact index in the box beside each). The curve updates live and is sampled at
  `BEZIER_NUM_POINTS` points when you record it. Best for the smooth drifting envelope of a band,
  and much quicker than clicking a long lane.

Either way you press **End trace** to record what you have placed, under the current label.

| control | what it does |
|---|---|
| **Band** | whether the next trace is a fundamental or a harmonic lane |
| **Method** | click points, or place a B&eacute;zier curve |
| **End trace** (&#10003;) | record the trace in progress under the current label. In click mode a right-click does the same; in B&eacute;zier mode clicks on the plot are ignored, so this button is the only way to commit |
| **Undo point** (&#8630;) | remove the last clicked point (click mode only) |
| **New lane** (+) | move on to the next free lane number of this band |
| **Delete last** | remove the most recently recorded repeat |
| **auto-repeats** | B&eacute;zier mode only: record all `N_REPS` repeats at once, the first being the curve you placed and the rest jittered by `BEZIER_JITTER` pixels |

Traces are labelled **`F lane 1`, `F lane 2`, `H lane 1`** and so on. Each label is traced
`N_REPS` times &mdash; the spread between repeats is the reproducibility error &mdash; and the
tracer moves to the next lane automatically once a label has its repeats. Trace as many lanes per
band as the burst actually shows; you do not have to say which is the lower or upper branch of a
split, because &sect;A.4 sorts the lanes of each band in frequency and takes the lowest as the
upstream branch and the next as the downstream one.

**Your tracing is safe against re-runs.** The recorded traces live in a module-level `TRACE_STORE`,
not on the tracer object, so re-running the cell that opens the tracer picks up exactly what you
had (it prints what it restored). Call `LaneTracer(reset=True)` when you actually want to start
over. The lane number is derived from what is recorded rather than from a counter, so deleting
traces walks the numbering back down instead of leaving it stranded above the real lanes. Run
`tracer.summary()` at any point to see precisely what is stored.

The figure needs the widget backend (`%matplotlib widget`, i.e. `ipympl`). Zoom and pan with the
toolbar as usual; clicks are ignored while a toolbar tool is active, so you can zoom in on the fine
structure and then trace it.

In [ ]:
def draw_bezier(x1=0, y1=0, x2=0, y2=0, controls=[[0, 0]], n=2, num_points=30):
    """Bezier curve of degree n from (x1,y1) to (x2,y2) with n-1 control points.
    n=1 linear, n=2 quadratic (1 anchor), n=3 cubic (2 anchors). Returns (num_points, 2)."""
    P0 = np.array([x1, y1], dtype=float)
    P3 = np.array([x2, y2], dtype=float)
    t = np.linspace(0, 1, num_points)
    if n == 1:
        curve = (1 - t)[:, None] * P0 + t[:, None] * P3
    elif n == 2:
        P1 = np.array(controls[0], dtype=float)
        curve = ((1 - t)[:, None] ** 2 * P0
                 + 2 * (1 - t)[:, None] * t[:, None] * P1
                 + t[:, None] ** 2 * P3)
    elif n == 3:
        P1 = np.array(controls[0], dtype=float)
        P2 = np.array(controls[1], dtype=float)
        curve = ((1 - t)[:, None] ** 3 * P0
                 + 3 * (1 - t)[:, None] ** 2 * t[:, None] * P1
                 + 3 * (1 - t)[:, None] * t[:, None] ** 2 * P2
                 + t[:, None] ** 3 * P3)
    else:
        raise ValueError('n must be 1 (linear), 2 (quadratic) or 3 (cubic).')
    return curve


def bezier_indices(x1, y1, x2, y2, controls, n, num_points=BEZIER_NUM_POINTS):
    """Integer (time-index, frequency-index) samples along a Bezier placed in the index space of
    the tracing layer, clipped to its bounds."""
    curve = draw_bezier(x1, y1, x2, y2, controls, n, num_points)
    xi = np.clip(np.round(curve[:, 0]).astype(int), 0, len(LAYER_T) - 1)
    yi = np.clip(np.round(curve[:, 1]).astype(int), 0, len(LAYER_F) - 1)
    return xi, yi


def bezier_freq_lane(x_idx, y_idx):
    """Map integer (time-idx, freq-idx) curve samples onto a {'t': [...], 'f': [...]} lane in MHz,
    sorted in time with duplicate times collapsed so the polynomial fits stay well posed.

    Duplicates are AVERAGED, not decided by whichever sample sorted first. Where the curve is
    steep several samples round to one time column and their frequencies can differ by several
    MHz; keeping the first is then an arbitrary pick from that range, and it biases the lane
    systematically towards one end of the sweep."""
    t_sel = pd.to_datetime([LAYER_T[i] for i in x_idx])
    f_val = np.asarray(LAYER_F[y_idx], float)
    x_num = mdates.date2num(t_sel)
    uniq_x, inv = np.unique(x_num, return_inverse=True)
    f_mean = np.bincount(inv, weights=f_val) / np.bincount(inv)
    t_uniq = pd.to_datetime(mdates.num2date(uniq_x)).tz_localize(None)
    return {'t': list(t_uniq), 'f': [float(v) for v in f_mean]}

In [ ]:
# Recorded traces live in a module-level store, NOT on the tracer object, so re-running this
# cell or the one that opens the tracer never throws your tracing away. Pass reset=True to
# LaneTracer when you really do want to start from scratch.
TRACE_STORE = {}         # label -> list of repeats, each {'t': [...], 'f': [...]}
TRACE_HISTORY = []       # labels in the order they were actually recorded, for Delete last
TRACE_KIND = {}          # label -> how its repeats were produced. This matters: jittered copies
                         # of one Bezier are NOT independent re-tracings and their spread is not a
                         # reproducibility error


class LaneTracer:
    """Lane tracer on a single plot of the whole dynamic spectrum, with two interchangeable
    methods: clicking points, or placing a Bezier curve with sliders. Both record under the same
    labels ('F lane 1', ...) and feed the identical `passes` structure. The spectrogram is drawn
    once and only the overlays are redrawn, through blitting.

    The lane number is always derived from what is actually recorded rather than from a counter,
    so deleting traces walks the numbering back down instead of leaving it stranded above the
    real lanes."""

    SLIDERS = [('x_start', 'start x (time)', 'x'), ('y_start', 'start y (freq)', 'y'),
               ('x_end', 'end x (time)', 'x'), ('y_end', 'end y (freq)', 'y'),
               ('cx1', 'anchor 1 x', 'x'), ('cy1', 'anchor 1 y', 'y'),
               ('cx2', 'anchor 2 x', 'x'), ('cy2', 'anchor 2 y', 'y')]

    def __init__(self, n_reps=N_REPS, bands=None, method=TRACE_METHOD, anchors=BEZIER_ANCHORS,
                 reset=False):
        import ipywidgets as widgets
        from IPython.display import display
        if reset:
            TRACE_STORE.clear()
            TRACE_HISTORY.clear()
            TRACE_KIND.clear()
        self.n_reps = n_reps
        self.bands = list(bands) if bands else list(BANDS)
        self.band = self.bands[0]
        self.method = method
        self.n_anchors = anchors
        self.traces = TRACE_STORE            # bound, not copied: survives re-running the cell
        self.history = TRACE_HISTORY
        self.lane_no = {b: 1 for b in self.bands}
        self.current = {'t': [], 'f': []}
        self.rng = np.random.default_rng(BEZIER_SEED)
        for b in self.bands:
            self._sync_label(b)
        if self.traces:
            print(f'restored {len(self.traces)} lane(s) already recorded: '
                  + ', '.join(f'{l} ({len(v)} rep)' for l, v in sorted(self.traces.items())))
            print('pass reset=True to LaneTracer if you want to start over')
        self._build(widgets, display)

    # ---------------------------------------------------------------- labels and state
    @property
    def label(self):
        return f'{self.band} lane {self.lane_no[self.band]}'

    def _used(self, band):
        return sorted(int(l.split()[-1]) for l in self.traces if l.startswith(band + ' lane '))

    def _sync_label(self, band=None):
        """Point the current label at a lane that is still being filled, otherwise at the next
        free number. Derived from `self.traces`, never from a free-running counter."""
        b = band or self.band
        used = self._used(b)
        n_cur = len(self.traces.get(f'{b} lane {self.lane_no[b]}', []))
        if 0 < n_cur < self.n_reps:
            return
        self.lane_no[b] = (max(used) + 1) if used else 1

    @property
    def labels(self):
        """Every recorded label, ordered by band then lane number."""
        return sorted(self.traces, key=lambda l: (self.bands.index(l.split()[0]),
                                                  int(l.split()[-1])))

    @property
    def order(self):
        return self.labels

    def colour(self, label):
        b = label.split()[0]
        i = int(label.split()[-1]) - 1
        cmap = plt.cm.winter if b == 'F' else plt.cm.autumn
        return cmap(min(i, 3) / 3.5)

    @property
    def passes(self):
        """The k-th repeat of every label assembled into pass k, so the lanes of a pass are
        mutually consistent. Labels with fewer repeats cycle through the ones they have."""
        if not self.traces:
            return []
        n = max(len(v) for v in self.traces.values())
        out = []
        for k in range(n):
            pas = {}
            for lab in self.labels:
                reps = self.traces[lab]
                if reps:
                    r = reps[k % len(reps)]
                    pas[lab] = {'t': list(r['t']), 'f': list(r['f'])}
            out.append(pas)
        return out

    def summary(self):
        """What is currently recorded. Run `tracer.summary()` at any point to check."""
        rows = []
        for lab in self.labels:
            reps = self.traces[lab]
            f = np.concatenate([np.asarray(r['f'], float) for r in reps])
            t = pd.to_datetime(np.concatenate([np.asarray(r['t']) for r in reps]))
            rows.append({'lane': lab, 'band': lab.split()[0], 'repeats': len(reps),
                         'repeat_kind': TRACE_KIND.get(lab, 'independent traces'),
                         'points_per_repeat': int(np.median([len(r['f']) for r in reps])),
                         'start_UT': t.min().strftime('%H:%M:%S'),
                         'end_UT': t.max().strftime('%H:%M:%S'),
                         'f_min_MHz': round(float(f.min()), 2),
                         'f_max_MHz': round(float(f.max()), 2)})
        return pd.DataFrame(rows)

    # ---------------------------------------------------------------- figure and widgets
    def _build(self, widgets, display):
        from ipywidgets import IntSlider, BoundedIntText, HBox, VBox, Label, Layout, jslink
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=[13, 6])
        plt.ion()
        try:
            self.fig.canvas.header_visible = False
        except AttributeError:
            pass
        pm = draw_layer(self.ax)
        self.fig.colorbar(pm, ax=self.ax, pad=0.01, label=CBAR_LABEL)

        (self.live,) = self.ax.plot([], [], 'o-', color='k', ms=6, lw=1.4, mfc='white',
                                    mew=1.4, animated=True)
        (self.bez_line,) = self.ax.plot([], [], '-', color='k', lw=2.2, animated=True)
        (self.bez_ends,) = self.ax.plot([], [], 'o', mfc='white', mec='black', mew=1.5, ms=9,
                                        ls='none', animated=True)
        (self.bez_anch,) = self.ax.plot([], [], '^', mfc='white', mec='black', mew=1.3, ms=10,
                                        ls='none', animated=True)
        self.done_lines = {}
        self._bg = None
        self.fig.canvas.mpl_connect('draw_event', self._on_draw)
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)

        self.sel = widgets.ToggleButtons(
            options=[(f'{BAND_NAME[b]} ({b})', b) for b in self.bands], value=self.band,
            description='Band:', style={'description_width': 'initial'})
        self.meth = widgets.ToggleButtons(
            options=[('Click points', 'click'), ('Bezier curve', 'bezier')], value=self.method,
            description='Method:', style={'description_width': 'initial'})
        self.deg = widgets.Dropdown(options=[('quadratic', 1), ('cubic', 2)], value=self.n_anchors,
                                    description='Bezier:', layout=Layout(width='170px'),
                                    style={'description_width': 'initial'})
        self.jit = widgets.Checkbox(value=True, description=f'auto-repeats ({self.n_reps})',
                                    indent=False, layout=Layout(width='170px'))
        self.btn_end = widgets.Button(description='End trace', icon='check',
                                      button_style='success',
                                      tooltip='record the trace in progress')
        self.btn_undo = widgets.Button(description='Undo point', icon='rotate-left')
        self.btn_new = widgets.Button(description='New lane', icon='plus',
                                      tooltip='start the next lane of this band')
        self.btn_del = widgets.Button(description='Delete last', icon='trash',
                                      button_style='danger',
                                      tooltip='drop the most recently recorded trace')
        self.status = widgets.HTML()

        d = self._defaults()
        self._sliders, rows = {}, []
        for name, lab, axis in self.SLIDERS:
            hi = (len(LAYER_T) if axis == 'x' else len(LAYER_F)) - 1
            sld = IntSlider(value=d[name], min=0, max=hi, step=1, readout=False,
                            continuous_update=True, layout=Layout(width='520px'))
            box = BoundedIntText(value=d[name], min=0, max=hi, step=1, layout=Layout(width='95px'))
            jslink((sld, 'value'), (box, 'value'))
            sld.observe(self._on_slider, names='value')
            self._sliders[name] = sld
            rows.append(HBox([Label(lab, layout=Layout(width='120px')), sld, box]))
        self._rows = rows
        self.bez_box = VBox(rows)

        self.sel.observe(self._on_band, names='value')
        self.meth.observe(self._on_method, names='value')
        self.deg.observe(self._on_deg, names='value')
        self.btn_end.on_click(lambda _: self.end_trace())
        self.btn_undo.on_click(lambda _: self.undo())
        self.btn_new.on_click(lambda _: self.new_lane())
        self.btn_del.on_click(lambda _: self.delete_last())
        display(VBox([HBox([self.sel, self.meth]),
                      HBox([self.btn_end, self.btn_undo, self.btn_new, self.btn_del,
                            self.deg, self.jit]),
                      self.bez_box, self.status, self.fig.canvas]))
        for lab in self.labels:
            self._redraw_label(lab)
        self._apply_method()
        self._title()
        self._refresh()

    def _defaults(self):
        """Seed the Bezier on the brightest channel near the start, middle and end of the window,
        so the curve already lies near an emission lane before you touch it."""
        nt, nf = len(LAYER_T), len(LAYER_F)
        xs = [int(f * (nt - 1)) for f in (0.10, 0.35, 0.65, 0.90)]
        w = max(1, nt // 40)
        ys = []
        for x in xs:
            col = np.nanmean(LAYER_D[:, max(0, x - w):min(nt, x + w) + 1], axis=1)
            ys.append(int(np.nanargmax(col)) if np.isfinite(col).any() else nf // 2)
        return dict(x_start=xs[0], y_start=ys[0], x_end=xs[3], y_end=ys[3],
                    cx1=xs[1], cy1=ys[1], cx2=xs[2], cy2=ys[2])

    def _title(self):
        n = len(self.traces.get(self.label, []))
        how = 'click points' if self.method == 'click' else 'place the Bezier with the sliders'
        self.ax.set_title(f'tracing {self.label}  -  repeat {min(n + 1, self.n_reps)} of '
                          f'{self.n_reps}   |   {how}, then End trace')

    # ---------------------------------------------------------------- blitting
    def _on_draw(self, event=None):
        self._bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self._blit()

    def _blit(self):
        for ln in self.done_lines.values():
            self.ax.draw_artist(ln)
        for art in (self.live, self.bez_line, self.bez_ends, self.bez_anch):
            self.ax.draw_artist(art)

    def _refresh(self):
        if self.method == 'click':
            if self.current['f']:
                self.live.set_data(mdates.date2num(pd.to_datetime(self.current['t'])),
                                   self.current['f'])
            else:
                self.live.set_data([], [])
        else:
            v = {k: s.value for k, s in self._sliders.items()}
            anch = self._anchors(v)
            curve = draw_bezier(v['x_start'], v['y_start'], v['x_end'], v['y_end'],
                                anch, self.n_anchors + 1, BEZIER_NUM_POINTS)
            xi = np.clip(np.round(curve[:, 0]).astype(int), 0, len(LAYER_T) - 1)
            yi = np.clip(np.round(curve[:, 1]).astype(int), 0, len(LAYER_F) - 1)
            self.bez_line.set_data(mdates.date2num(pd.to_datetime(LAYER_T[xi])), LAYER_F[yi])
            self.bez_ends.set_data(mdates.date2num(pd.to_datetime(
                LAYER_T[[v['x_start'], v['x_end']]])), LAYER_F[[v['y_start'], v['y_end']]])
            self.bez_anch.set_data(mdates.date2num(pd.to_datetime(
                LAYER_T[[int(a[0]) for a in anch]])), LAYER_F[[int(a[1]) for a in anch]])
        if self._bg is not None:
            self.fig.canvas.restore_region(self._bg)
            self._blit()
            self.fig.canvas.blit(self.ax.bbox)
        else:
            self.fig.canvas.draw_idle()
        self._update_status()

    def _line_for(self, label):
        if label not in self.done_lines:
            (self.done_lines[label],) = self.ax.plot([], [], '-', lw=2, alpha=0.9,
                                                     color=self.colour(label), animated=True,
                                                     label=label)
        return self.done_lines[label]

    def _redraw_label(self, label):
        xs, ys = [], []
        for rep in self.traces.get(label, []):
            xs += list(mdates.date2num(pd.to_datetime(rep['t']))) + [np.nan]
            ys += list(rep['f']) + [np.nan]
        self._line_for(label).set_data(xs, ys)

    # ---------------------------------------------------------------- Bezier helpers
    def _anchors(self, v):
        if self.n_anchors == 1:
            return [(v['cx1'], v['cy1'])]
        return [(v['cx1'], v['cy1']), (v['cx2'], v['cy2'])]

    def _bezier_rep(self, jitter=0):
        v = {k: s.value for k, s in self._sliders.items()}
        base = np.array([[v['x_start'], v['y_start']], [v['x_end'], v['y_end']],
                         *self._anchors(v)], float)
        if jitter > 0:
            base = base + self.rng.normal(0, jitter, base.shape)
        (sx, sy), (ex, ey) = base[0], base[1]
        xi, yi = bezier_indices(sx, sy, ex, ey, [list(p) for p in base[2:]], self.n_anchors + 1)
        return bezier_freq_lane(xi, yi)

    def _apply_method(self):
        show = '' if self.method == 'bezier' else 'none'
        self.bez_box.layout.display = show
        self.deg.layout.display = show
        self.jit.layout.display = show
        self.btn_undo.disabled = self.method != 'click'
        for i, row in enumerate(self._rows):
            row.layout.display = 'none' if (self.method != 'bezier' or
                                            (i >= 6 and self.n_anchors == 1)) else ''

    # ---------------------------------------------------------------- interaction
    def _on_band(self, change):
        self.band = change['new']
        self.current = {'t': [], 'f': []}
        self._sync_label()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()

    def _on_method(self, change):
        self.method = change['new']
        self.current = {'t': [], 'f': []}
        self.live.set_data([], [])
        for art in (self.bez_line, self.bez_ends, self.bez_anch):
            art.set_data([], [])
        self._apply_method()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()

    def _on_deg(self, change):
        self.n_anchors = change['new']
        self._apply_method()
        self._refresh()

    def _on_slider(self, change=None):
        self._refresh()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None:
            return
        mode = getattr(getattr(self.fig.canvas, 'toolbar', None), 'mode', '')
        if mode:                                  # zoom or pan is active, so do not add points
            return
        if self.method != 'click':
            # in Bezier mode the curve is placed with the sliders, so a stray click on the plot
            # must not record anything: End trace is the only way to commit it
            return
        if event.button == 3:
            self.end_trace()
            return
        self.current['t'].append(pd.Timestamp(mdates.num2date(event.xdata).replace(tzinfo=None)))
        self.current['f'].append(float(event.ydata))
        self._refresh()

    def undo(self):
        if self.method == 'click' and self.current['t']:
            self.current['t'].pop()
            self.current['f'].pop()
        self._refresh()

    def end_trace(self):
        lab = self.label
        if self.method == 'click':
            if len(self.current['f']) < 2:
                self._update_status('a trace needs at least two points', warn=True)
                return
            new_reps = [{'t': list(self.current['t']), 'f': list(self.current['f'])}]
            self.current = {'t': [], 'f': []}
        else:
            done = len(self.traces.get(lab, []))
            if done >= self.n_reps:
                self._update_status(f'{lab} already has its {self.n_reps} repeats - press '
                                    '"New lane" to move on', warn=True)
                return
            if self.jit.value:
                # repeat 0 is the curve you placed; the rest jitter the control points, which is
                # the Bezier equivalent of re-tracing a lane by hand
                new_reps = [self._bezier_rep(0 if (done + k) == 0 else BEZIER_JITTER)
                            for k in range(self.n_reps - done)]
            else:
                new_reps = [self._bezier_rep(0)]
        self.traces.setdefault(lab, []).extend(new_reps)
        self.history.extend([lab] * len(new_reps))
        # Any jittered repeat makes the lane's spread a jitter measurement, however many were
        # added. Keying this on len(new_reps) > 1 mislabels a lane that already had repeats and
        # is being topped up one at a time, and that label decides whether A.4 is allowed to
        # divide the spread by sqrt(N_pass).
        jittered = (self.method == 'bezier' and self.jit.value
                    and any((len(self.traces[lab]) - len(new_reps) + k) > 0
                            for k in range(len(new_reps))))
        if jittered:
            TRACE_KIND[lab] = 'bezier auto-repeats'
        else:
            TRACE_KIND.setdefault(lab, 'independent traces')
        self._redraw_label(lab)
        msg = f'recorded {lab}, {len(self.traces[lab])}/{self.n_reps} repeat(s)'
        self._sync_label()
        if self.label != lab:
            msg += f' - moving on to {self.label}'
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(msg)

    def new_lane(self):
        used = self._used(self.band)
        self.lane_no[self.band] = (max(used) + 1) if used else 1
        self.current = {'t': [], 'f': []}
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(f'started {self.label}')

    def delete_last(self):
        if not self.history:
            self._update_status('nothing recorded yet', warn=True)
            return
        lab = self.history.pop()
        if self.traces.get(lab):
            self.traces[lab].pop()
        if lab in self.traces and not self.traces[lab]:
            del self.traces[lab]
            # drop the kind with the lane. Leaving it behind means a lane later re-traced by hand
            # keeps the deleted lane's 'bezier auto-repeats' label, or worse, a lane later filled
            # with jittered repeats keeps 'independent traces' and gets its spread divided by
            # sqrt(N_pass) in A.4 as though it had been re-traced.
            TRACE_KIND.pop(lab, None)
        self._redraw_label(lab)
        self._sync_label(lab.split()[0])
        self._sync_label()
        self._title()
        self.fig.canvas.draw_idle()
        self._refresh()
        self._update_status(f'deleted one repeat of {lab}, '
                            f'{len(self.traces.get(lab, []))} left on it')

    def _update_status(self, msg='', warn=False):
        if self.traces:
            bits = []
            for lab in self.labels:
                reps = len(self.traces[lab])
                col = mpl.colors.to_hex(self.colour(lab))
                mark = '' if reps >= self.n_reps else ' <i>(incomplete)</i>'
                bits.append(f'<span style="color:{col}"><b>{lab}</b> {reps}/{self.n_reps}'
                            f'{mark}</span>')
            rec = ' &nbsp;|&nbsp; '.join(bits)
            tot = (f'{len(self.traces)} lane(s), '
                   f'{sum(len(v) for v in self.traces.values())} trace(s)')
        else:
            rec = '<i>nothing recorded yet</i>'
            tot = '0 lanes'
        extra = (f'&nbsp; <b>points in this trace:</b> {len(self.current["f"])}'
                 if self.method == 'click' else '&nbsp; <b>method:</b> Bezier')
        colour = '#a00' if warn else '#060'
        self.status.value = (f'<b>now tracing:</b> {self.label} {extra}<br>'
                             f'<b>recorded ({tot}):</b> {rec}<br>'
                             f'<span style="color:{colour}">{msg}</span>')

Run the cell below to open the tracer. Everything is traced in that one figure, in whichever
method you prefer &mdash; you can switch between clicking and B&eacute;zier at any point, even
between lanes. When you are done, run the cell after it to collect the traces.

In [ ]:
%matplotlib widget
tracer = LaneTracer()          # LaneTracer(reset=True) to discard everything and start over

In [ ]:
%matplotlib inline
# ---- collect the traces -------------------------------------------------------------------
passes = tracer.passes
if not passes:
    raise RuntimeError('nothing traced yet - trace the lanes in the figure above, press '
                       '"End trace" for each, then re-run this cell')

TRACED = [lab for lab in tracer.labels if any(p.get(lab, {}).get('f') for p in passes)]
LANE_COL = {lab: tracer.colour(lab) for lab in TRACED}
LANE_COL[JOINT] = 'black'
LANE_BAND = {lab: lab.split()[0] for lab in TRACED}
BANDS_TRACED = [b for b in BANDS if b in set(LANE_BAND.values())]

# say plainly what came out of the store, and complain if it does not look like a traced event
n_by_band = {b: sum(1 for l in TRACED if l.startswith(b + ' ')) for b in BANDS}
print(f'{len(passes)} pass(es) over {len(TRACED)} lane(s): '
      + ', '.join(f'{n_by_band[b]} x {BAND_NAME[b]}' for b in BANDS))
for b in BANDS:
    if n_by_band[b] == 0:
        print(f'  WARNING: no {BAND_NAME[b]} lane recorded - the {b} band diagnostics will be '
              'skipped. Switch the Band toggle and trace it, then re-run this cell.')
    elif n_by_band[b] == 1:
        print(f'  NOTE: only one {BAND_NAME[b]} lane recorded, so that band has no split, '
              'hence no X or M_A. Trace its other split branch if the spectrum shows one.')
incomplete = [l for l in TRACED if len(tracer.traces[l]) < tracer.n_reps]
if incomplete:
    print(f'  NOTE: fewer than {tracer.n_reps} repeats on: ' + ', '.join(incomplete))
AUTO_REPEAT = [l for l in TRACED if TRACE_KIND.get(l) == 'bezier auto-repeats']

# Are the repeats independent measurements? Only if every lane was re-traced by hand. This drives
# whether the aggregation is allowed to divide the spread by sqrt(N_pass) in A.4: averaging N
# jittered copies of one curve returns the curve you drew, so it buys no precision and the
# sqrt(N) is unearned.
# A lane whose repeats are byte-identical is not N measurements either. Pressing "End trace"
# repeatedly in Bezier mode without touching the sliders records the same curve every time: the
# spread is exactly zero, and calling that independent would divide an already-zero error by
# sqrt(N) and then propagate it as though the lane had been measured N times.
DUPLICATE_REPS = []
for _l in TRACED:
    _r = tracer.traces.get(_l, [])
    if len(_r) > 1:
        _n = min(len(x['f']) for x in _r)
        _st = np.vstack([np.asarray(x['f'], float)[:_n] for x in _r])
        if np.allclose(_st, _st[0], rtol=0, atol=1e-9):
            DUPLICATE_REPS.append(_l)
if DUPLICATE_REPS:
    print(f'\n  WARNING: {", ".join(DUPLICATE_REPS)} have IDENTICAL repeats - the same curve was '
          'recorded more\n        than once, so their spread is exactly zero. That is not a '
          'reproducibility measurement.\n        Move the curve between presses, or tick '
          'auto-repeats.')

REPEATS_INDEPENDENT = (len(AUTO_REPEAT) == 0 and len(DUPLICATE_REPS) == 0)
if AUTO_REPEAT:
    print(f'\n  NOTE: {", ".join(AUTO_REPEAT)} used Bezier AUTO-REPEATS. Those repeats are the '
          f'same curve with its\n        control points jittered by BEZIER_JITTER = '
          f'{BEZIER_JITTER:.1f} channels, so their spread measures that jitter,\n        NOT how '
          'reproducibly you can place a curve on the lane. The error bars for those lanes are\n'
          '        therefore fit-error-dominated and optimistic. For a genuine reproducibility '
          'error, untick\n        "auto-repeats" and place the curve independently '
          f'{tracer.n_reps} times.')
    print('        Because of this the aggregation in A.4 will NOT divide the spread by '
          'sqrt(N_pass).')

tracer.summary()

> **Table 1 &mdash; What is recorded.** `tracer.summary()`
>
> One row per traced lane: how many repeats it has, **how those repeats were produced**
> (`repeat_kind`), how many points each holds, and the time and frequency range it covers.
>
> *Read `repeat_kind` carefully.* `independent traces` means you placed the lane that many times by
> hand, so the spread between them is a genuine reproducibility error. `bezier auto-repeats` means
> repeat&nbsp;0 is the curve you placed and the rest are that same curve with its control points
> jittered by `BEZIER_JITTER` pixels &mdash; their spread measures the jitter, not your ability to
> place a curve, so the error bars for those lanes are optimistic.

In [ ]:
# The repeat spread is a few tenths of a MHz. Over a 60 MHz frequency axis that is thinner than
# the lines themselves, so on the full spectrum the repeats cannot separate no matter how they are
# styled. The bottom row therefore re-plots each lane over a short window at a frequency scale
# where the spread is wider than the linewidth: same curves, same data, readable magnification.
_nz = max(len(TRACED), 1)
fig = plt.figure(figsize=[15, 9])
gs = fig.add_gridspec(2, _nz, height_ratios=[2.6, 1], hspace=0.28, wspace=0.22)
ax = fig.add_subplot(gs[0, :])
pm = draw_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label=CBAR_LABEL)

# all repeats at one linewidth, plus the envelope they span: no repeat is privileged, and the
# shaded band is the spread itself rather than a claim about it
_spread, _zoom = {}, {}
for lab in TRACED:
    reps = [p[lab] for p in passes if p.get(lab) and p[lab]['f']]
    if not reps:
        continue
    for k, L in enumerate(reps):
        ax.plot(pd.to_datetime(L['t']), L['f'], color=LANE_COL[lab], lw=1.3, alpha=0.9,
                label=(lab if k == 0 else None))
    tref = mdates.date2num(pd.to_datetime(reps[0]['t']))
    stack = np.vstack([np.interp(tref, mdates.date2num(pd.to_datetime(L['t'])),
                                np.asarray(L['f'], float)) for L in reps])
    if len(reps) > 1:
        ax.fill_between(pd.to_datetime(reps[0]['t']), stack.min(0), stack.max(0),
                        color=LANE_COL[lab], alpha=0.35, lw=0)
        _spread[lab] = float(np.nanmean(np.nanstd(stack, axis=0)))
    _zoom[lab] = (reps, tref, stack)

_rms = float(np.nanmean(list(_spread.values()))) if _spread else np.nan
ax.set_title('Traced lanes, all repeats overplotted'
             + (f'  -  repeat spread {_rms:.3f} MHz rms (magnified below)'
                if np.isfinite(_rms) else ''))
ax.legend(fontsize=8, ncol=4, loc='lower left')

# --- one magnified window per lane, centred where the lane is ------------------------------
for j, lab in enumerate(TRACED):
    az = fig.add_subplot(gs[1, j])
    if lab not in _zoom:
        az.set_axis_off()
        continue
    reps, tref, stack = _zoom[lab]
    mid = len(tref) // 2
    half_s = max(30, 0.12 * (tref[-1] - tref[0]) * 86400 / 2)
    t_c = mdates.num2date(tref[mid]).replace(tzinfo=None)
    sel = np.abs((tref - tref[mid]) * 86400) <= half_s
    f_c = float(np.nanmean(stack[:, sel]))
    # frequency half-range: whichever is larger, 4x the spread or a 0.3 MHz floor, so the band
    # fills a useful part of the panel instead of collapsing to a line
    half_f = max(4 * _spread.get(lab, 0.1), 0.3)
    pmz = draw_layer(az, tstep=1)
    for L in reps:
        az.plot(pd.to_datetime(L['t']), L['f'], color=LANE_COL[lab], lw=1.3, alpha=0.95)
    az.fill_between(pd.to_datetime(reps[0]['t']), stack.min(0), stack.max(0),
                    color=LANE_COL[lab], alpha=0.3, lw=0)
    az.set_xlim(t_c - pd.Timedelta(seconds=half_s), t_c + pd.Timedelta(seconds=half_s))
    az.set_ylim(f_c + half_f, f_c - half_f) if INVERT_FREQ else az.set_ylim(f_c - half_f,
                                                                           f_c + half_f)
    az.set_title(f'{lab}: {len(reps)} repeats, '
                 + (f'{_spread[lab]:.3f} MHz rms' if lab in _spread else 'single trace'),
                 fontsize=9)
    # three ticks at most: a 60-90 s window at %H:%M:%S overruns the panel width otherwise, and
    # these labels are not to be rotated
    az.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=2, maxticks=3))
    az.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    az.tick_params(labelsize=7.5)
    az.set_xlabel('')
    az.set_ylabel('MHz' if j == 0 else '', fontsize=8)
    az.grid(alpha=0.25)
if not _spread:
    ax.text(0.5, 0.03, 'one repeat per lane - no spread to show',
            transform=ax.transAxes, ha='center', fontsize=9, color='0.3')
save_fig(fig, 'traced_lanes_preview')
plt.show()

# --- tracing quality: how far is each traced point from the local intensity ridge? ---
# worth checking in Bezier mode especially: a Bezier's interior control points are NOT on its
# curve, so a curve can look plausible while sitting systematically off the lane
qc = []
for lab in TRACED:
    L = passes[0][lab]
    ti = np.searchsorted(LAYER_T, pd.to_datetime(L['t']).to_numpy()).clip(0, len(LAYER_T) - 1)
    fi = np.abs(LAYER_F[:, None] - np.asarray(L['f'], float)[None, :]).argmin(axis=0)
    half = max(3, len(LAYER_F) // 60)
    off = []
    for it, jf in zip(ti, fi):
        lo, hi = max(0, jf - half), min(len(LAYER_F), jf + half + 1)
        col = LAYER_D[lo:hi, it]
        if np.isfinite(col).any():
            off.append(int(np.nanargmax(col)) + lo - jf)
    off = np.asarray(off, float)
    # how bright is the layer ON the trace? A lane drawn across empty spectrum has a small
    # offset-from-ridge by accident (the brightest noise pixel in the window is near zero
    # offset), so the offset check alone cannot tell it from a good trace. The intensity can.
    amp = np.array([LAYER_D[jf, it] for it, jf in zip(ti, fi)], float)
    qc.append({'lane': lab, 'median_offset_chan': np.nanmedian(off),
               'rms_offset_chan': np.sqrt(np.nanmean(off ** 2)),
               'offset_MHz': np.nanmedian(off) * np.nanmedian(np.diff(LAYER_F)),
               'median_on_trace': np.nanmedian(amp), 'min_on_trace': np.nanmin(amp)})
qc = pd.DataFrame(qc)

# the rms offset from the ridge is the honest 1-sigma frequency uncertainty of a traced point:
# it is how far the curve actually sits from the emission it is meant to follow
_chan = float(np.nanmedian(np.diff(LAYER_F)))
if LANE_SIGMA_MHZ == 'auto':
    LANE_SIGMA = {r['lane']: max(abs(r['rms_offset_chan']) * _chan, LANE_SIGMA_FLOOR)
                  for _, r in qc.iterrows()}
elif LANE_SIGMA_MHZ is None:
    LANE_SIGMA = {}
else:
    LANE_SIGMA = {lab: float(LANE_SIGMA_MHZ) for lab in TRACED}
qc['sigma_f_MHz'] = [LANE_SIGMA.get(l, np.nan) for l in qc['lane']]

_unit = 'dB above background' if LAYER_MODE == 'db_sub' else 'x background'
print(f'offset of each traced point from the local intensity maximum '
      f'(search window +/-{half} channels, channel width {_chan * 1e3:.0f} kHz).')
print('more than a channel or two of median offset means the trace is sitting off the lane.')
print(f'median_on_trace / min_on_trace are the layer value ON the trace, in {_unit}.')
print('sigma_f_MHz is the per-point frequency uncertainty the lane fits will use.')
_faint = qc[qc['median_on_trace'] < QC_MIN_SNR_DB]
if len(_faint):
    print()
    for _, r in _faint.iterrows():
        print(f'  *** WARNING: {r["lane"]} sits at only {r["median_on_trace"]:.1f} '
              f'{_unit} along its length (threshold {QC_MIN_SNR_DB}). Part of it is very likely '
              'drawn across empty spectrum - check it against the traced-lanes figure before '
              'using it. A low offset-from-ridge does NOT vouch for a trace over empty sky.')


def _ridge_rms(lab, half_ch, shift_ch=0):
    """rms offset [channels] between a lane and the brightest pixel within +/-half_ch of it.

    shift_ch displaces the lane in frequency first, which is how the control below puts the same
    curve over blank spectrum while keeping its shape, length and time sampling identical."""
    L = passes[0][lab]
    ti = np.searchsorted(LAYER_T, pd.to_datetime(L['t']).to_numpy()).clip(0, len(LAYER_T) - 1)
    fi = (np.abs(LAYER_F[:, None] - np.asarray(L['f'], float)[None, :]).argmin(axis=0)
          + shift_ch).clip(0, len(LAYER_F) - 1)
    o = []
    for it, jf in zip(ti, fi):
        lo_, hi_ = max(0, jf - half_ch), min(len(LAYER_F), jf + half_ch + 1)
        col = LAYER_D[lo_:hi_, it]
        if np.isfinite(col).any():
            o.append(int(np.nanargmax(col)) + lo_ - jf)
    return np.sqrt(np.nanmean(np.asarray(o, float) ** 2)) if o else np.nan


# Two questions about the number above, because sigma_f sets every statistical error bar in this
# notebook and it is easy to mistake it for something the data determined on its own.
#
#   (1) Does it depend on the search window? A brightest-pixel search has no intrinsic scale: if
#       the window is widened and the rms grows with it, the "ridge" is not pinning the trace
#       down and sigma_f is really half/sqrt(3) channels. A uniform distribution over the window
#       gives exactly that, so it is printed as the null.
#   (2) Is there a ridge at all? The same measurement is repeated with every lane displaced
#       QC_CONTROL_SHIFT channels in frequency, onto blank spectrum. If the on-lane rms is not
#       clearly below the displaced one, the traces are not following emission.
_halves = sorted({3, 5, half, 2 * half, 3 * half})
print(f'\nsigma_f depends on the search window. rms offset [channels] vs half-width, against the '
      f'{len(_halves) and ""}uniform-in-window null (half/sqrt3):')
print(f'  {"half":>5} {"null":>7} ' + ' '.join(f'{l:>11}' for l in TRACED) + '   on/off-lane')
_ratio_at_half = np.nan
for _h in _halves:
    _on = [_ridge_rms(l, _h) for l in TRACED]
    _off = [_ridge_rms(l, _h, shift_ch=QC_CONTROL_SHIFT) for l in TRACED]
    _rat = np.nanmean(_on) / np.nanmean(_off) if np.nanmean(_off) > 0 else np.nan
    if _h == half:
        _ratio_at_half = _rat
    print(f'  {_h:>5} {_h / np.sqrt(3):>7.2f} ' + ' '.join(f'{v:>11.2f}' for v in _on)
          + f'   {_rat:>6.2f}' + ('   <- used' if _h == half else ''))
print(f'  The rms grows with the window, so sigma_f is set mainly by the +/-{half}-channel search '
      f'and not by')
print(f'  a measured trace-to-ridge distance. Every statistical error bar below scales linearly '
      f'with it.')
if np.isfinite(_ratio_at_half) and _ratio_at_half < 0.95:
    print(f'  But the ridge is real: displacing the lanes {QC_CONTROL_SHIFT} channels onto blank '
          f'spectrum raises the rms by')
    print(f'  {100 * (1 / _ratio_at_half - 1):.0f}%, so the traced curves are following emission '
          f'rather than noise.')
else:
    print(f'  *** WARNING: displacing the lanes {QC_CONTROL_SHIFT} channels off the emission does '
          f'NOT raise the rms offset')
    print('      (ratio {:.2f}). The ridge finder is returning noise and these traces are not '
          'demonstrably'.format(_ratio_at_half))
    print('      on the lane. Re-check them against the traced-lanes figure before going further.')
qc.to_csv(os.path.join(OUTDIR, 'tracing_quality.csv'), index=False)
qc.round(3)

> **Figure 4 &mdash; Traced lanes.** `traced_lanes_preview.png`
>
> Every recorded lane drawn on the tracing layer; the bold line is the first repeat and the thin
> ones are the others, so the visible thickness of a bundle *is* the reproducibility spread.
>
> **Table 2 &mdash; Tracing quality.** (printed below the figure)
>
> For each lane: the median and rms offset, in frequency channels, between the traced points and
> the local intensity maximum within a $\pm$ few-channel search window; the layer value **on** the
> trace; and `sigma_f_MHz`, the per-point frequency uncertainty the lane fits will use.
>
> *Both columns are needed.* A median offset above a channel or two means the curve is sitting
> beside the lane rather than on it. But the offset test alone cannot detect a lane drawn across
> **empty spectrum** &mdash; over blank sky the brightest pixel inside the search window is noise,
> which lands near zero offset by chance. `median_on_trace` is what catches that, and a lane below
> `QC_MIN_SNR_DB` is flagged.

## A.4 &middot; Shock diagnostics from the traced lanes

**Every lane is analysed on its own.** Nothing is merged. `MAKE_JOINT = True` additionally builds
an `F+H joint` track that averages the two bands' *upstream* lanes where they overlap and takes
whichever exists elsewhere; it buys a longer baseline for the second derivative but mixes two
independent measurements, so it is off by default.

**Which lane is which.** The lanes of each band are ordered by frequency **over the interval in
which they overlap in time**: the lower is the **upstream** (unshocked) branch of that band's
split, the next the **downstream** (compressed) branch, so $X=(f_U/f_L)^2\ge1$. Comparing over the
overlap rather than over each lane's own span matters &mdash; a long lane running high to low can
have a higher own-span mean than a short lane sitting above it, and ordering on that inverts the
split, sends $X$ below 1, and makes $M_A$, $v_A$ and $B$ all NaN.

### The estimators, in order

1. **Lane fit.** Each traced lane is fitted with a degree-`LANE_DEG` polynomial in $\log_{10}f$
   against time. Log space because a type II lane spans close to an octave, over which a
   polynomial in $f$ is far too stiff. Drift rates come off that fit analytically:
   $\dot f = f\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$, and the relative drift
   $\dot f/f=\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$ is a property of the shock alone, free of
   the band's harmonic number.
2. **Band split.** $X=(f_U/f_L)^2$ and $M_A=\sqrt{X(X+5)/[2(4-X)]}$, evaluated where both branches
   of a band exist. No density model enters, so each band gives an independent measurement.
3. **Height.** $n_e=(f/s/\mathrm{PLASMA\_CONST})^2$ with $s=1$ for F and $s=2$ for H, then
   $n_e(r)$ is inverted numerically for $r$. Out-of-range densities return NaN rather than an
   extrapolation.
4. **Speed and acceleration.** $r(t)$ is fitted with a degree-`KIN_DEG` polynomial and
   differentiated analytically. `KIN_DEG = 2`, so $a$ is constant per lane; §A.7 gives the
   time-dependent version. The fit uses `KIN_N_DENSE` samples along the **lane's own** span
   rather than the handful of shared-grid points that happen to fall inside it &mdash; the lane
   fit is analytic, so it can be sampled as finely as the polynomial needs.

   **No lane is refused a quadratic because it is short.** Every lane is fitted at `KIN_DEG`
   whenever it has the points for it, and whether the resulting curvature is a *measurement* is
   then decided by comparing $|a|$ against its own uncertainty (`KIN_A_SIGMA`). A fixed cut on
   traced duration would make a four-second difference in lane length decide whether an
   acceleration exists, and would hide the one thing worth knowing: a degenerate quadratic
   announces itself through a large error bar, so it is better fitted and reported as
   insignificant than silently replaced by a straight line. `KIN_MIN_BASELINE_S` survives only as
   the span below which the notebook says the baseline is short.
5. **Alfv&eacute;n speed and field.** $v_A=v_{\rm sh}/M_A$ and $B=v_A\sqrt{\mu_0\rho}$ with
   $\rho=\mu m_p n_e$. Both inherit $M_A$ from that lane's band, so a lane whose band has only one
   traced branch has no $v_A$ and no $B$.

### Where the error bars come from, and what they do **not** cover

Two sources are propagated together by Monte Carlo. The **repeat error** is the scatter between
your repeats of a lane. The **fit error** is drawn from the covariance of the lane's polynomial
coefficients: for each pass, `N_MC` coefficient vectors are drawn from that covariance and pushed
through the entire chain above. The quoted `_sd` is the standard deviation over all
draws&times;passes.

**`_se` is only `_sd`$/\sqrt{N_{\rm pass}}$ when the repeats are independent re-tracings.** Dividing
by $\sqrt{N}$ claims the precision that comes from repeating a measurement, and B&eacute;zier
auto-repeats are not a repeated measurement &mdash; they are one curve displaced `BEZIER_JITTER`
pixels $N_{\rm REPS}$ times, so averaging them recovers the curve you drew and says nothing about
how well it sits on the ridge. &sect;A.3 sets `REPEATS_INDEPENDENT` from `TRACE_KIND`, and when it
is `False` the aggregation leaves `_se = _sd`. If you want an error bar that means *how
reproducibly can this lane be traced*, untick **auto-repeats** and place the curve from scratch
`N_REPS` times &mdash; nothing else in the chain can manufacture that number for you.

Two things are done to stop that error being fictitiously small. A B&eacute;zier is sampled at
`BEZIER_NUM_POINTS` along a curve with only about eight free parameters, so those samples are
nowhere near independent; the fit **thins them to one point per `FIT_MIN_DT_S` seconds**. And
`np.polyfit`'s default covariance scales with the residual scatter, which for a smooth curve
measures how smooth the curve is rather than how well it sits on the lane; each point is instead
given an explicit frequency uncertainty (`LANE_SIGMA_MHZ = 'auto'` takes the rms offset from the
ridge measured by the §A.3 QC) and the covariance is returned **unscaled**.

Even so the height errors stay small, and that is real rather than a bug: in an exponential corona
$r$ depends only weakly on $f$. For Newkirk at $r\simeq1.8\,R_\odot$,
$\mathrm{d}\ln r/\mathrm{d}\ln f\approx-0.35$, so a 1% frequency error is a 0.35% height error,
about $0.006\,R_\odot$. **The uncertainty that matters is the density model**, and that is the
model&times;fold range in §A.8. Quote the reference-model value with that range beside it.

In [ ]:
t0 = TYPEII_WINDOW[0]        # shared time origin for every lane


def _thin(ts, min_dt=FIT_MIN_DT_S):
    """Keep one point per min_dt seconds. Points sampled densely along a Bezier are not
    independent measurements - the curve has ~8 free parameters - and feeding all of them to
    polyfit shrinks the covariance by a factor it has not earned.

    Walks the points in time order and returns indices into the array as given, so it is correct
    whether or not the input arrives sorted; comparing against the previous element of an
    unsorted array would keep points a fraction of a second apart."""
    ts = np.asarray(ts, float)
    keep, last = [], -np.inf
    for i in np.argsort(ts, kind='stable'):
        if ts[i] - last >= min_dt:
            keep.append(int(i))
            last = ts[i]
    return np.asarray(sorted(keep), int)


def _lane_fit(lane_data, deg=LANE_DEG, sigma_f=None):
    """Polynomial fit of a lane, in log10 f when FIT_IN_LOGF.

    sigma_f is the assumed 1-sigma frequency uncertainty of a traced point in MHz. When given,
    the fit is weighted by it and the covariance is returned UNSCALED, so it reflects that stated
    uncertainty instead of the residual scatter of an artificially smooth curve."""
    if not lane_data or len(lane_data['f']) < 2:
        return None
    t = pd.to_datetime(lane_data['t'])
    f = np.asarray(lane_data['f'], float)
    ts = (t - t0).total_seconds().to_numpy()
    o = np.argsort(ts)
    ts, f = ts[o], f[o]
    tmin, tmax = ts.min(), ts.max()
    k = _thin(ts)
    if len(k) >= deg + 2:
        ts, f = ts[k], f[k]
    y = np.log10(f) if FIT_IN_LOGF else f
    sy = None
    if sigma_f:
        # d(log10 f) = df / (f ln10)
        sy = (sigma_f / (f * np.log(10))) if FIT_IN_LOGF else np.full_like(f, float(sigma_f))
    d = min(deg, len(ts) - 1)
    cov = None
    try:
        if sy is not None and np.all(np.isfinite(sy)) and np.all(sy > 0) and len(ts) > d + 1:
            p, cov = np.polyfit(ts, y, d, w=1 / sy, cov='unscaled')
        elif len(ts) >= d + 2:
            p, cov = np.polyfit(ts, y, d, cov=True)
        else:
            p = np.polyfit(ts, y, d)
    except (ValueError, np.linalg.LinAlgError):
        p = np.polyfit(ts, y, d)
    return {'p': p, 'cov': cov, 'tmin': tmin, 'tmax': tmax, 'n_fit': len(ts),
            'sigma_f': sigma_f, 'fmin': float(np.nanmin(f)), 'fmax': float(np.nanmax(f))}


def _coeffs(fit, rng=None, sample=False):
    if fit is None:
        return None
    if sample and rng is not None and fit['cov'] is not None:
        try:
            return rng.multivariate_normal(fit['p'], fit['cov'])
        except np.linalg.LinAlgError:
            return fit['p']
    return fit['p']


def _eval(fit, coeffs, tg):
    """Frequency [MHz] of a lane fit on the grid, blanked outside the traced span.

    Blanked in FREQUENCY as well as in time. A polynomial evaluated at the very edge of its time
    span can leave the frequency range that was actually traced - and a Monte-Carlo draw, or a
    grid point covered by only one of several passes, reaches further still. Those samples are
    extrapolation dressed as measurement: on this event the harmonic ran 0.4 MHz below its lowest
    traced point, past the bottom of the analysis band, and turned the height track back on
    itself. A small tolerance is allowed so a legitimate endpoint is not clipped by rounding."""
    if fit is None or coeffs is None:
        return np.full_like(tg, np.nan)
    y = np.polyval(coeffs, tg)
    f = 10 ** y if FIT_IN_LOGF else y
    f[(tg < fit['tmin']) | (tg > fit['tmax'])] = np.nan
    if 'fmin' in fit and FIT_CLIP_TO_TRACED_F:
        tol = FIT_F_TOL_MHZ
        f[(f < fit['fmin'] - tol) | (f > fit['fmax'] + tol)] = np.nan
    return f


def lane_deriv(fit, ts):
    """Frequency, drift rate [MHz/s] and relative drift (1/f)(df/dt) [1/s] of a lane fit at ts.
    With a log fit, d(log10 f)/dt converts as df/dt = f ln(10) d(log10 f)/dt, and the relative
    drift is just ln(10) d(log10 f)/dt - a property of the shock, free of the band's harmonic."""
    dy = np.polyval(np.polyder(fit['p']), ts)
    if FIT_IN_LOGF:
        f = 10 ** np.polyval(fit['p'], ts)
        return f, f * np.log(10) * dy, np.log(10) * dy
    f = np.polyval(fit['p'], ts)
    return f, dy, dy / f


def pass_fits(pas):
    return {lab: _lane_fit(pas.get(lab), sigma_f=LANE_SIGMA.get(lab)) for lab in TRACED}


def lane_fit_quality(lane_data, fit):
    """Does the polynomial actually describe the traced curve?

    Two failure modes that nothing else in the chain notices, because every downstream quantity
    is computed from the FIT and never looks back at the points it came from:

      resid_MHz   rms distance between the fitted curve and the traced points. A Bezier that
                  bends more than a quadratic in log f can follow leaves a residual many times
                  LANE_SIGMA, and the drift, speed and acceleration are then describing the
                  polynomial rather than the burst.
      turnover_s  time at which the fitted df/dt changes sign inside the traced span, if it does.
                  A type II lane drifts one way. A fit that turns over has run out of curvature
                  budget at the end of the lane, and that is what sends the speed to zero and
                  the acceleration to implausible values."""
    if fit is None or not lane_data or len(lane_data['f']) < 3:
        return {'resid_MHz': np.nan, 'resid_over_sigma': np.nan, 'turnover_s': np.nan}
    t = pd.to_datetime(lane_data['t'])
    f = np.asarray(lane_data['f'], float)
    ts = (t - t0).total_seconds().to_numpy()
    o = np.argsort(ts)
    ts, f = ts[o], f[o]
    resid = float(np.sqrt(np.nanmean((f - _eval(fit, fit['p'], ts)) ** 2)))
    sig = fit.get('sigma_f') or LANE_SIGMA_FLOOR
    gg = np.linspace(fit['tmin'], fit['tmax'], 200)
    dv = lane_deriv(fit, gg)[1]
    sign_flip = np.flatnonzero(np.diff(np.sign(dv[np.isfinite(dv)])) != 0)
    return {'resid_MHz': resid, 'resid_over_sigma': resid / sig,
            'turnover_s': float(gg[sign_flip[0]]) if len(sign_flip) else np.nan}


def order_lanes(labs):
    """Order a band's lanes by frequency over the interval where they OVERLAP in time.

    Comparing the mean frequency over each lane's own span is wrong as soon as the lanes cover
    different stretches of a drifting burst: a long lane that starts high and ends low can have
    a higher own-span mean than a short lane sitting above it, which inverts the split. The
    inversion is silent but fatal - it sends X = (f_U/f_L)^2 below 1, and M_A, v_A and B then
    all come out NaN."""
    fits = {l: _lane_fit(passes[0][l], sigma_f=LANE_SIGMA.get(l)) for l in labs}
    lo = max(f['tmin'] for f in fits.values())
    hi = min(f['tmax'] for f in fits.values())
    if hi > lo:
        ts = np.linspace(lo, hi, 40)
        note = f'compared over their {hi - lo:.0f} s of overlap'
    else:
        ts = np.linspace(min(f['tmin'] for f in fits.values()),
                         max(f['tmax'] for f in fits.values()), 40)
        note = ('WARNING: these lanes never overlap in time, so they are ordered on EXTRAPOLATED '
                'fits. Re-trace so the split branches share some time.')
    key = {l: float(np.nanmean(lane_deriv(fits[l], ts)[0])) for l in labs}
    return sorted(labs, key=lambda l: key[l]), key, note


def build_grid(passes, n=N_GRID):
    """One time grid spanning the whole burst - both bands belong to the same shock."""
    secs = []
    for pas in passes:
        for lab in TRACED:
            if pas.get(lab) and pas[lab]['f']:
                secs += list((pd.to_datetime(pas[lab]['t']) - t0).total_seconds())
    if not secs:
        raise RuntimeError('no traced points to build a time grid from')
    return np.linspace(np.nanmin(secs), np.nanmax(secs), n)


def _invert_grid(model, r_bounds=R_BOUNDS, nres=6000):
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    return rr, model(rr)


def sg_smooth(y, window=9, poly=2):
    """Savitzky-Golay trend through a track. Interior gaps are bridged, but nothing is
    extrapolated past the traced span. np.interp clamps to the end values, which would draw a
    flat horizontal trend across the part of the grid where the lane does not exist."""
    y = np.asarray(y, float)
    m = np.isfinite(y)
    if m.sum() < 5:
        return y
    idx = np.flatnonzero(m)
    inside = np.arange(idx[0], idx[-1] + 1)
    yy = np.interp(inside, idx, y[m])
    w = min(window, len(inside))
    if w % 2 == 0:
        w -= 1
    out = np.full_like(y, np.nan)
    out[inside] = yy if w < poly + 2 else savgol_filter(yy, w, poly)
    return out


def kin_degree(npts, deg=None):
    """Degree of the r(t) fit a lane can support, set only by how many points it has.

    A degree-d polynomial needs d+2 points before its curvature means anything at all, and that
    is the only hard limit. There is deliberately no cut on the traced baseline: a lane traced
    for 596 s is not categorically different from one traced for 604 s, and gating on a fixed
    duration makes a four-second difference decide whether an acceleration exists. A quadratic
    over a short baseline is allowed to be fitted and then to report the large uncertainty it has
    earned - see accel_is_measured()."""
    deg = KIN_DEG if deg is None else deg
    return max(1, min(deg, int(npts) - 2))


def accel_is_measured(a, a_err, n_sigma=KIN_A_SIGMA, bias=0):
    """Is an acceleration distinguishable from zero? It has to clear two different hurdles.

    Its own error bar is the first. The second is `bias`: the acceleration this chain returns for
    a shock that is genuinely not accelerating at all (see accel_bias_floor). That floor is a
    property of the method, not of the data, so no amount of Monte Carlo will reveal it - the
    error bars are perfectly happy to be tight around a wrong number."""
    if not (np.isfinite(a) and np.isfinite(a_err)):
        return False
    if a_err <= 0 or abs(a) < n_sigma * a_err:
        return False
    return not (np.isfinite(bias) and abs(a) < abs(bias))


def accel_bias_floor(fit, band, invert, v_ref=None):
    """Acceleration this chain reports for a shock moving at EXACTLY constant speed.

    Take the lane's own traced span, put a constant-velocity shock through the same height range,
    convert it to frequencies with the same harmonic number, then run it back through the same
    lane fit, the same inversion and the same r(t) fit. The answer should be zero. Whatever comes
    out instead is bias contributed by the method: the lane polynomial cannot exactly represent
    the log f(t) of a shock climbing through a structured corona, and that misfit re-emerges as
    curvature in r(t). Any measured |a| below this floor is describing the fit, not the shock."""
    if fit is None:
        return np.nan
    rr, ne = invert
    tt = np.linspace(fit['tmin'], fit['tmax'], KIN_N_DENSE)
    # Use the speed this lane actually shows, not a fixed 600 km/s. A synthetic shock moving at
    # the wrong speed sweeps a different height range over the same time and returns a bias that
    # is not commensurate with the acceleration it is meant to be compared against - on the slower
    # tracks here that was a factor of two in the swept range.
    if v_ref is None:
        _f0v = _eval(fit, fit['p'], tt)
        _r0v = np.interp(freq_to_density(_f0v * 1e6, harmonic=HARM[band]), ne[::-1], rr[::-1])
        _okv = np.isfinite(_r0v)
        v_ref = (abs(np.nanmax(_r0v[_okv]) - np.nanmin(_r0v[_okv])) * R_SUN_M / 1e3
                 / max(tt.max() - tt.min(), 1)) if _okv.sum() > 2 else 600
        v_ref = float(np.clip(v_ref, 50, 3000))
    f_lane = _eval(fit, fit['p'], tt)
    ne_lane = freq_to_density(f_lane * 1e6, harmonic=HARM[band])
    r_lane = np.interp(ne_lane, ne[::-1], rr[::-1])
    if not np.isfinite(r_lane).any():
        return np.nan
    # a straight line through the same heights, over the same time, at constant speed
    r_lin = np.nanmin(r_lane) + (v_ref * 1e3 / R_SUN_M) * (tt - tt.min())
    ok = (r_lin >= np.nanmin(rr)) & (r_lin <= np.nanmax(rr))
    if ok.sum() < KIN_MIN_PTS + 2:
        return np.nan
    f_syn = PLASMA_CONST * np.sqrt(np.interp(r_lin[ok], rr, ne)) * HARM[band] / 1e6
    syn = {'t': [t0 + pd.Timedelta(seconds=float(x)) for x in tt[ok]], 'f': list(f_syn)}
    fs = _lane_fit(syn, sigma_f=fit.get('sigma_f'))
    if fs is None:
        return np.nan
    td = np.linspace(fs['tmin'], fs['tmax'], KIN_N_DENSE)
    tgs = np.linspace(fs['tmin'], fs['tmax'], N_GRID)
    to_r = lambda x: np.interp(freq_to_density(_eval(fs, fs['p'], x) * 1e6, harmonic=HARM[band]),
                               ne[::-1], rr[::-1])
    return float(np.nanmean(kinematics(to_r(tgs), tgs, t_fit=td, r_fit=to_r(td))[1]))


def kinematics(r, tg, deg=None, span=None, baseline=None, t_fit=None, r_fit=None):
    """Speed [km/s] and acceleration [m/s^2] from a height track.

    The gate asks whether the density model resolves the lane, so it is judged against the
    lane's OWN traced span (`span`), not against the shared grid. Judging it against the grid
    discards every short lane - one covering 4 of the burst's 32 minutes is a perfectly good
    measurement, it just covers less time."""
    v = np.full_like(tg, np.nan)
    a = np.full_like(tg, np.nan)
    good = np.isfinite(r)
    span = good if span is None else np.asarray(span, bool)
    # Fit over the lane's own dense samples when they are supplied. r(t) comes from an analytic
    # lane fit, so it can be sampled as finely as the polynomial needs; taking the fit points off
    # the shared grid instead leaves a short lane with a handful of them and makes the curvature
    # far worse determined than the data warrant.
    if t_fit is not None and r_fit is not None:
        tf, rf = np.asarray(t_fit, float), np.asarray(r_fit, float)
        okf = np.isfinite(rf)
        tf, rf = tf[okf], rf[okf]
    else:
        tf, rf = tg[good], r[good]
    deg = kin_degree(len(tf), deg)
    n_span = max(int(span.sum()), 1)
    # Count the points the fit will ACTUALLY use. Counting the shared grid's points instead
    # discards any lane shorter than about three grid steps - it has fewer than KIN_MIN_PTS of
    # them - even though the dense per-lane samples are there and perfectly well behaved. That
    # loses the speed as well as the acceleration, for a lane that was traced without complaint.
    n_used = len(tf)
    frac = (good.sum() / n_span) if (t_fit is None or r_fit is None) else 1
    # measure the height range on the same points the fit uses, for the same reason
    r_range = (np.nanmax(rf) - np.nanmin(rf)) if n_used > 3 else 0
    if (n_used >= KIN_MIN_PTS and frac >= KIN_MIN_FRAC
            and r_range > 0.01 and n_used > deg + 1 and good.any()):
        dg = deg
        pr = np.polyfit(tf, rf, dg)
        v = np.polyval(np.polyder(pr, 1), tg) * R_SUN_M / 1e3
        # A straight line has no second derivative to report. Returning the 0.0 that polyder
        # hands back would put a hard "a = 0.0 +/- 0.0" in the tables, which reads as a measured
        # null result rather than the absence of a measurement. NaN says the right thing.
        a = (np.polyval(np.polyder(pr, 2), tg) * R_SUN_M if dg >= 2
             else np.full_like(tg, np.nan))
        # blank the derivatives outside the span this track was traced over: each band covers a
        # different part of the burst, and a cubic extrapolated past its data runs away fast
        v[~good] = np.nan
        a[~good] = np.nan
    return np.where((v > 0) & (v < 3000), v, np.nan), a

In [ ]:
def realise_lanes(fits, tg, invert, rng=None, sample=False, mu=MU):
    """One realisation of height, speed, acceleration, Alfven speed and B for every lane, plus the
    combined F+H track. Each lane uses its band's harmonic number; B additionally needs that band's
    density jump X = (f_U/f_L)^2, so it only exists where the band has both split branches."""
    rr, ne = invert
    ne_min, ne_max = np.nanmin(ne), np.nanmax(ne)
    # keep the drawn coefficients: the dense per-lane track used for the r(t) fit has to come from
    # the SAME realisation as the gridded one, or the Monte Carlo mixes draws
    cf = {lab: _coeffs(fits.get(lab), rng, sample) for lab in TRACED}
    fval = {lab: _eval(fits.get(lab), cf[lab], tg) for lab in TRACED}

    def _to_r(f_mhz, band):
        ne_t = freq_to_density(f_mhz * 1e6, harmonic=HARM[band])
        r = np.interp(ne_t, ne[::-1], rr[::-1])
        r[(ne_t > ne_max) | (ne_t < ne_min)] = np.nan
        return r, ne_t

    out = {}
    for b in BANDS_TRACED:
        lo_lab, up_lab = SPLIT_PAIR[b]
        if up_lab is not None:
            with np.errstate(all='ignore'):
                X = (fval[up_lab] / fval[lo_lab]) ** 2
        else:
            X = np.full_like(tg, np.nan)
        MA = alfven_mach_from_X(X)
        for lab in LANE_ORDER[b]:
            r, ne_t = _to_r(fval[lab], b)
            _lf = fits.get(lab)
            t_d = r_d = None
            if _lf is not None:
                t_d = np.linspace(_lf['tmin'], _lf['tmax'], KIN_N_DENSE)
                r_d = _to_r(_eval(_lf, cf[lab], t_d), b)[0]
            v, a = kinematics(r, tg, span=np.isfinite(fval[lab]),
                              baseline=(float(_lf['tmax'] - _lf['tmin']) if _lf else None),
                              t_fit=t_d, r_fit=r_d)
            vA = v / MA
            B = (vA * 1e3) * np.sqrt(MU0 * mu * M_P * (ne_t * 1e6)) * 1e4      # Gauss
            out[lab] = dict(r=r, v=v, a=a, vA=vA, B=B, X=X, MA=MA, ne=ne_t, f=fval[lab])

    # --- combined fundamental + harmonic track -------------------------------------------
    # the two bands are the same shock, so the upstream branches' heights and densities are
    # averaged where both exist and taken singly elsewhere: one trajectory across the whole burst
    ups = [out[SPLIT_PAIR[b][0]] for b in BANDS_TRACED if SPLIT_PAIR[b][0] in out]
    if MAKE_JOINT and len(ups) > 1:
        stack = lambda key: np.nanmean(np.vstack([u[key] for u in ups]), axis=0)
        r_j, ne_j, MA_j = stack('r'), stack('ne'), stack('MA')
        v_j, a_j = kinematics(r_j, tg)
        vA_j = v_j / MA_j
        B_j = (vA_j * 1e3) * np.sqrt(MU0 * mu * M_P * (ne_j * 1e6)) * 1e4
        out[JOINT] = dict(r=r_j, v=v_j, a=a_j, vA=vA_j, B=B_j, X=stack('X'), MA=MA_j,
                          ne=ne_j, f=np.full_like(tg, np.nan))
    return out


def common_mask(d, keys=('r', 'v', 'vA', 'B', 'ne', 'X', 'MA')):
    """Grid samples where EVERY listed quantity of a track exists.

    r, v and n_e are defined over a lane's whole traced span, but X, M_A, v_A and B need both
    branches of that band's split and so exist only where the two branches overlap in time.
    Averaging each over its own finite samples puts the quantities of one table row on different,
    non-overlapping intervals, and the row then fails its own identities: v_A comes out unequal to
    v_sh/M_A, and B unequal to v_A sqrt(mu0 rho) evaluated at the n_e printed beside it. Anything
    reported as one row of one table has to be averaged over one window."""
    ms = [np.isfinite(d[k + '_mean']) for k in keys if k + '_mean' in d]
    return np.logical_and.reduce(ms) if ms else None


def grid_scalar(d, key, mask=None):
    """Grid-average of a track's quantity and its combined standard error.

    Pass `mask` (usually from common_mask) whenever the result sits in a row beside other
    quantities; leave it None only for a number quoted on its own."""
    m, se = d[key + '_mean'], d[key + '_se']
    good = np.isfinite(m) if mask is None else (np.isfinite(m) & mask)
    if good.sum() == 0:
        return np.nan, np.nan
    return np.nanmean(m[good]), np.sqrt(np.nanmean(se[good] ** 2))


def lane_windows(d, lab, tgrid, t_ref, upstream=None):
    """Every scalar a lane supports, on both of its windows, computed in one place.

    `upstream` decides whether v_A and B are returned at all. It defaults to reading LANE_ROLE,
    which A.4 fills in before anything calls this - but it is a parameter rather than only a
    global lookup, so a caller can be explicit and a test can set it without having to reproduce
    the whole naming state. Passing the wrong value silently changes which quantities appear,
    which is exactly the kind of coupling worth making visible in the signature.

    Three sections quote these numbers, and three private copies of this arithmetic is exactly how
    v_A came to be printed as v_sh/M_A in one cell and 23% away from it in another. They all call
    this instead.

    `span` is everything the lane traced: the window for r, v_sh, a and n_e, and the one that
    describes the burst. `split` is the sub-interval where both branches of that band exist, the
    only place X and M_A are defined and therefore the only place v_A = v_sh/M_A and
    B = v_A sqrt(mu0 rho) mean anything; r, v_sh and n_e are repeated there so a reader can close
    the identities from the printed numbers alone. Neither window is the right one for both groups,
    so both are returned and every caller has to say which it is showing.

    v_A and B come back NaN for a downstream branch. v_A from Rankine-Hugoniot is by construction
    the UPSTREAM Alfven speed; pairing it with the downstream density returns B_1 sqrt(X), which is
    neither the upstream field nor the downstream one."""
    out = {'lane': lab,
           'upstream': (LANE_ROLE.get(lab, '').startswith('upstream') if upstream is None
                        else bool(upstream))}
    fmt = lambda msk: (
        f'{(t_ref + pd.Timedelta(seconds=float(tgrid[msk].min()))).strftime("%H:%M:%S")}-'
        f'{(t_ref + pd.Timedelta(seconds=float(tgrid[msk].max()))).strftime("%H:%M:%S")} UT')
    span = np.isfinite(d['r_mean']) & np.isfinite(d['v_mean'])
    out['span_mask'] = span
    out['n_span'] = int(span.sum())
    out['span_window'] = fmt(span) if span.any() else ''
    for k in ('r', 'v', 'a', 'ne'):
        out[k + '_span'], out[k + '_span_e'] = grid_scalar(d, k, mask=span if span.any() else None)
    cm = common_mask(d)
    out['split_mask'] = cm
    out['n_split'] = int(cm.sum()) if cm is not None else 0
    out['has_split'] = out['n_split'] > 0
    out['split_window'] = fmt(cm) if out['has_split'] else ''
    for k in ('r', 'v', 'ne', 'X', 'MA', 'vA', 'B'):
        if not out['has_split'] or (k in ('vA', 'B') and not out['upstream']):
            out[k + '_split'], out[k + '_split_e'] = np.nan, np.nan
        else:
            out[k + '_split'], out[k + '_split_e'] = grid_scalar(d, k, mask=cm)
    return out


AGG_KEYS = ('r', 'v', 'a', 'vA', 'B', 'X', 'MA', 'ne', 'f')

def aggregate_lanes(passes, tg, model, n_mc=N_MC, seed=0):
    """Monte-Carlo aggregation per track, combining the fitting and repeat errors."""
    rng = np.random.default_rng(seed)
    invert = _invert_grid(model)
    fitsets = [pass_fits(pas) for pas in passes]
    stacks = {k: {kk: [] for kk in AGG_KEYS} for k in ALL_TRACKS}
    for fits in fitsets:
        for _ in range(n_mc):
            res = realise_lanes(fits, tg, invert, rng=rng, sample=True)
            for key, d in res.items():
                for kk in AGG_KEYS:
                    stacks[key][kk].append(d[kk])
    out = {}
    # Dividing the spread by sqrt(N_pass) claims the precision gain that comes from repeating an
    # independent measurement. Jittered Bezier repeats are not that: they are ONE curve displaced
    # N_REPS times, so averaging them recovers the curve you drew and nothing about how well it
    # sits on the ridge. Only divide when the repeats were genuinely re-traced.
    npass = max(len(passes), 1)
    root_n = np.sqrt(npass) if REPEATS_INDEPENDENT else 1
    for key in ALL_TRACKS:
        if not stacks[key]['r']:
            continue
        out[key] = {}
        for kk in AGG_KEYS:
            M = np.vstack(stacks[key][kk])
            out[key][kk + '_mean'] = np.nanmean(M, axis=0)
            out[key][kk + '_sd'] = np.nanstd(M, axis=0)
            out[key][kk + '_se'] = out[key][kk + '_sd'] / root_n
            # how many realisations actually reached this grid point. The passes have slightly
            # different traced spans, so the first and last points of a track are averaged over a
            # subset of them; that makes the mean there noisy enough to send r backwards by more
            # than its own error bar. Downstream code uses this to drop partly covered samples.
            out[key][kk + '_n'] = np.sum(np.isfinite(M), axis=0)
    return out


def scalar_summary(passes):
    """Model-independent scalars per band: drift rate, relative drift, density jump, Alfven Mach
    number and relative bandwidth, each as mean +/- standard error across the passes. The absolute
    drift of the harmonic is twice the fundamental's, but the RELATIVE drift (1/f)(df/dt) is the
    same quantity for both bands and is the one to compare."""
    def mse(arr):
        arr = np.asarray(arr, float)
        if arr.size == 0:
            return (np.nan, np.nan)
        se = np.nanstd(arr, ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else 0
        return np.nanmean(arr), se

    def combine(per_pass_mean, along_lane_sd):
        """Uncertainty on a quantity that is averaged along the overlap.

        The scatter between passes is NOT the whole error, and with jittered Bezier repeats it is
        not even the larger part of it: it only says how far BEZIER_JITTER moved the curve. X is
        quoted as a single number for the whole burst, so how much it actually varies ALONG the
        overlap belongs in its error bar too - and for a real band split that variation dominates.
        Leaving it out is what made two perfectly compatible X values look 10 sigma apart."""
        m, se = mse(per_pass_mean)
        sd = np.nanmean(along_lane_sd) if len(along_lane_sd) else np.nan
        if not np.isfinite(sd):
            return (m, se)
        return (m, float(np.hypot(se if np.isfinite(se) else 0, sd)))

    rows = {}
    for b in BANDS_TRACED:
        lo_lab, up_lab = SPLIT_PAIR[b]
        Xv, MAv, drift, relbw, reldrift = [], [], [], [], []
        Xsd, Xlo, Xhi, rdsd, MAsd = [], [], [], [], []
        Xbad, Xnear4 = [], []
        for pas in passes:
            fl = _lane_fit(pas.get(lo_lab), sigma_f=LANE_SIGMA.get(lo_lab))
            fu = (_lane_fit(pas.get(up_lab), sigma_f=LANE_SIGMA.get(up_lab))
                  if up_lab else None)
            if fl and fu:
                lo, hi = max(fl['tmin'], fu['tmin']), min(fl['tmax'], fu['tmax'])
                if hi > lo:
                    gg = np.linspace(lo, hi, 20)
                    fLv, fUv = lane_deriv(fl, gg)[0], lane_deriv(fu, gg)[0]
                    Xg = (fUv / fLv) ** 2
                    Xv.append(np.nanmean(Xg))
                    Xsd.append(np.nanstd(Xg))
                    Xlo.append(np.nanmin(Xg))
                    Xhi.append(np.nanmax(Xg))
                    # How much of the overlap is unusable? alfven_mach_from_X returns NaN outside
                    # 1 <= X < 4, and every mean below is a nanmean, so a split whose branches
                    # cross part-way silently reports M_A from the half that still works while
                    # the mean X stays close to 1 and looks unremarkable.
                    Xbad.append(float(np.mean(~((Xg >= 1) & (Xg < 4)))))
                    Xnear4.append(float(np.mean(Xg > 3.5)))
                    MAg = alfven_mach_from_X(Xg)
                    MAv.append(np.nanmean(MAg))
                    MAsd.append(np.nanstd(MAg))
                    relbw.append(np.nanmean((fUv - fLv) / fLv))
            if fl:
                gg = np.linspace(fl['tmin'], fl['tmax'], 20)
                _, dv, rd = lane_deriv(fl, gg)
                drift.append(np.nanmean(dv))                                  # MHz/s
                reldrift.append(np.nanmean(rd))                               # 1/s
                rdsd.append(np.nanstd(rd))
        rows[b] = {'X': combine(Xv, Xsd), 'M_A': combine(MAv, MAsd), 'drift_MHz_s': mse(drift),
                   'rel_drift_s': combine(reldrift, rdsd), 'rel_bandwidth': mse(relbw),
                   'X_range': (np.nanmean(Xlo) if Xlo else np.nan,
                               np.nanmean(Xhi) if Xhi else np.nan),
                   'X_pass_se': mse(Xv)[1],
                   'X_frac_invalid': float(np.mean(Xbad)) if Xbad else np.nan,
                   'X_frac_near4': float(np.mean(Xnear4)) if Xnear4 else np.nan}
    return rows

In [ ]:
# --- which lane of each band is the upstream branch, decided over their overlap -------------
LANE_ORDER, LANE_FREQ, LANE_ROLE = {}, {}, {}
for b in BANDS_TRACED:
    labs = [l for l in TRACED if LANE_BAND[l] == b]
    LANE_ORDER[b], key, note = order_lanes(labs)
    LANE_FREQ.update(key)
    for i, lab in enumerate(LANE_ORDER[b]):
        LANE_ROLE[lab] = ('upstream (lower)' if i == 0 else
                          'downstream (upper)' if i == 1 else 'extra')
    print(f'{BAND_NAME[b]} (s={HARM[b]}), {note}:')
    for lab in LANE_ORDER[b]:
        print(f'    {lab:12s} {LANE_FREQ[lab]:6.1f} MHz  ->  {LANE_ROLE[lab]}')
    if len(labs) < 2:
        print('    only one lane on this band, so it has no band split, hence no X or M_A')
SPLIT_PAIR = {b: (v[0], v[1] if len(v) > 1 else None) for b, v in LANE_ORDER.items()}
ALL_TRACKS = list(TRACED) + ([JOINT] if MAKE_JOINT else [])
print(f'\nanalysing {len(TRACED)} lane(s) separately'
      + ('; plus the combined F+H track (MAKE_JOINT = True)' if MAKE_JOINT
         else '. Set MAKE_JOINT = True to add the combined F+H track as well.'))

tg = build_grid(passes)
ALref = aggregate_lanes(passes, tg, MODEL_GRID[REF_MODEL_NAME])
scalars = scalar_summary(passes)
TRACKS = [k for k in ALL_TRACKS if k in ALref]

# --- does the polynomial actually describe each traced lane? --------------------------------
_f0 = pass_fits(passes[0])
LANE_FIT_QC = pd.DataFrame([{'lane': lab, 'sigma_f_MHz': LANE_SIGMA.get(lab, np.nan),
                             **lane_fit_quality(passes[0].get(lab), _f0.get(lab))}
                            for lab in TRACED])
LANE_FIT_QC['verdict'] = np.where(
    LANE_FIT_QC['resid_over_sigma'] > LANE_FIT_MAX_RESID, 'FLAG: fit misses the trace',
    np.where(LANE_FIT_QC['turnover_s'].notna(), 'FLAG: fit turns over', 'ok'))
LANE_FIT_QC.to_csv(os.path.join(OUTDIR, 'lane_fit_quality.csv'), index=False)
_fitvar = 'log10 f' if FIT_IN_LOGF else 'f'
print(f'\nlane fit quality (does a degree-{LANE_DEG} polynomial in {_fitvar} '
      'follow the traced curve?)')
for _, r in LANE_FIT_QC.iterrows():
    print(f'  {r["lane"]:12s} rms residual {r["resid_MHz"]:6.3f} MHz '
          f'= {r["resid_over_sigma"]:5.1f} x sigma_f   {r["verdict"]}')
for _, r in LANE_FIT_QC[LANE_FIT_QC.verdict.str.startswith('FLAG')].iterrows():
    if np.isfinite(r['turnover_s']):
        print(f'  *** {r["lane"]}: the fitted drift changes sign at '
              f'{(t0 + pd.Timedelta(seconds=r["turnover_s"])).strftime("%H:%M:%S")} UT, inside '
              'the traced span. A type II lane drifts one way, so the speed and acceleration '
              'near the end of this lane are describing the polynomial, not the shock.')
    else:
        print(f'  *** {r["lane"]}: the fit sits {r["resid_MHz"]:.2f} MHz rms off the traced '
              f'points ({r["resid_over_sigma"]:.0f} x the assumed frequency error). Raise '
              'LANE_DEG or re-trace this lane; every quantity derived from it is suspect.')

print(f'\nReference density model: {REF_MODEL_NAME}\n')
print('model-independent scalars, measured separately from each band')
for b in BANDS_TRACED:
    s = scalars[b]
    print(f'  {BAND_NAME[b]} (s={HARM[b]}):  '
          f'drift = {s["drift_MHz_s"][0]:+.4f} +/- {s["drift_MHz_s"][1]:.4f} MHz/s,'
          f'  rel. drift = {s["rel_drift_s"][0]:+.5f} +/- {s["rel_drift_s"][1]:.5f} 1/s,'
          f'  X = {s["X"][0]:.3f} +/- {s["X"][1]:.3f},'
          f'  M_A = {s["M_A"][0]:.3f} +/- {s["M_A"][1]:.3f},'
          f'  BDW = {s["rel_bandwidth"][0]:.3f} +/- {s["rel_bandwidth"][1]:.3f}')
    if np.isfinite(s['X_range'][0]):
        print(f'      X runs {s["X_range"][0]:.3f} to {s["X_range"][1]:.3f} along the overlap; '
              f'the pass-to-pass scatter alone is only +/-{s["X_pass_se"]:.4f}. The quoted error '
              'is the two combined,')
        print('      because X is being reported as one number for a burst over which it '
              'demonstrably varies.')
    _bad, _n4 = s.get('X_frac_invalid', np.nan), s.get('X_frac_near4', np.nan)
    if np.isfinite(_bad) and _bad > 0:
        print(f'      *** WARNING: X falls outside 1 <= X < 4 over {100 * _bad:.0f}% of the '
              'overlap, so M_A, v_A and B are')
        print('      NaN there. Every mean below is a nanmean, so those samples are dropped '
              'silently and the')
        print('      quoted values describe only the part of the overlap that still works. '
              'Check the branch ordering.')
    if np.isfinite(_n4) and _n4 > 0:
        print(f'      *** WARNING: X > 3.5 over {100 * _n4:.0f}% of the overlap. M_A diverges as '
              'X -> 4 (X = 3.99 gives')
        print('      M_A = 42), so v_A = v_sh/M_A collapses and B with it. Values from that '
              'region are not usable.')

for b in BANDS_TRACED:
    x = scalars[b]['X'][0]
    if np.isfinite(x) and x < 1:
        print(f'  ERROR: X < 1 for the {BAND_NAME[b]} band. The split branches are the wrong way '
              'round, so M_A, v_A and B will all be NaN. Check the ordering printed above '
              'against the traced-lanes figure.')

KIN_DEG_USED, KIN_BASELINE = {}, {}
for lab in TRACED:
    d = ALref.get(lab)
    if d is None:
        continue
    sp = np.isfinite(d['f_mean'])
    # Measure the baseline from the lane's OWN traced span, not from the grid points that fall
    # inside it. The shared grid has N_GRID points across the whole burst (~33 s apart here), so
    # a lane is always short by up to two grid steps - enough to push a genuinely 624 s lane
    # under KIN_MIN_BASELINE_S = 600 s and silently drop its acceleration.
    _ff = _f0.get(lab)
    KIN_BASELINE[lab] = (float(_ff['tmax'] - _ff['tmin']) if _ff is not None
                         else (float(np.nanmax(tg[sp]) - np.nanmin(tg[sp])) if sp.any() else 0))
    KIN_DEG_USED[lab] = kin_degree(KIN_N_DENSE if _ff is not None else int(sp.sum()))

# Every lane is fitted at KIN_DEG; whether its curvature is a measurement is decided by the error
# bar, not by the traced duration.
_INV_REF = _invert_grid(MODEL_GRID[REF_MODEL_NAME])
A_BIAS = {lab: accel_bias_floor(_f0.get(lab), LANE_BAND[lab], _INV_REF) for lab in TRACED}
A_MEASURED = {}
for lab in TRACED:
    if lab in ALref:
        _a, _ea = grid_scalar(ALref[lab], 'a')
        A_MEASURED[lab] = accel_is_measured(_a, _ea, bias=A_BIAS.get(lab, 0))

print(f'\nmodel-dependent quantities on {REF_MODEL_NAME}')
for lab in TRACED:
    if lab in A_MEASURED and not A_MEASURED[lab]:
        _a, _ea = grid_scalar(ALref[lab], 'a')
        print(f'  NOTE: {lab} gives a = {_a:+.1f} +/- {_ea:.1f} m/s^2 over a {KIN_BASELINE[lab]:.0f} s '
              f'baseline - under {KIN_A_SIGMA} sigma, so its curvature is NOT SIGNIFICANT.')
        print('        The value and its error are still reported; the lane is not silently '
              'dropped, and a short')
        print('        baseline is not by itself a reason to refuse the fit.')
print('  Each lane gets two lines, because the quantities below do not all live on the same')
print('  interval and averaging them as though they did makes a row contradict itself.')
print('    full span         r and v_sh over everything the lane traced. A lane covering the')
print('                      earlier part of a decelerating burst returns a higher mean speed,')
print('                      so speeds are only comparable between lanes with the same coverage.')
print('    band-split window the sub-interval where BOTH branches of that band exist. X and M_A')
print('                      are defined only here, so v_A and B are too; r, v_sh and n_e are')
print('                      repeated so the row closes v_A = v_sh/M_A on its own numbers.')
for key in TRACKS:
    d = ALref[key]
    if np.isfinite(d['r_mean']).sum() <= 2:
        print(f'  {key:12s}:  no height solution for this model')
        continue
    w = lane_windows(d, key, tg, t0)
    print(f'  {key:12s} full span         :  r = {w["r_span"]:.3f} +/- {w["r_span_e"]:.3f} Rsun,'
          f'  v_sh = {w["v_span"]:.0f} +/- {w["v_span_e"]:.0f} km/s'
          f'   [{w["span_window"]}, {w["n_span"]} grid pts]')
    if not w['has_split']:
        print(f'  {"":12s} band-split window :  the two branches of this band never overlap in '
              f'time, so X, M_A, v_A and B are undefined for it')
        continue
    tail = (f'  v_A = {w["vA_split"]:.0f} +/- {w["vA_split_e"]:.0f} km/s,'
            f'  B = {w["B_split"]:.3f} +/- {w["B_split_e"]:.3f} G'
            if w['upstream'] else
            '  v_A, B: not defined for a downstream branch (v_A from Rankine-Hugoniot is the '
            'UPSTREAM Alfven speed)')
    print(f'  {"":12s} band-split window :  r = {w["r_split"]:.3f} +/- {w["r_split_e"]:.3f} Rsun,'
          f'  v_sh = {w["v_split"]:.0f} +/- {w["v_split_e"]:.0f} km/s,{tail}')
    chk = (f',  check v_sh/M_A = {w["v_split"] / w["MA_split"]:.0f} km/s'
           if w['upstream'] and np.isfinite(w['MA_split']) and w['MA_split'] > 0 else '')
    print(f'  {"":12s}                      [{w["split_window"]}, {w["n_split"]} of '
          f'{w["n_span"]} grid pts]  n_e = {w["ne_split"]:.3e} cm^-3, '
          f'M_A = {w["MA_split"]:.3f}{chk}')

> **Output 1 &mdash; Lane roles and the model-independent scalars.** (printed above)
>
> First block: which lane of each band is the **upstream** (lower-frequency) branch of its split and
> which is the **downstream** branch, together with the frequencies they were compared at and the
> length of the interval over which the comparison was made.
>
> Second block: the drift rate, the relative drift $f^{-1}\dot f$, the density jump
> $X=(f_U/f_L)^2$, the Alfv&eacute;n Mach number $M_A$ and the relative bandwidth &mdash; per band,
> and **free of any density model**.
>
> Third block: height, shock speed, Alfv&eacute;n speed and magnetic field per lane on the
> reference model, with a note naming any lane whose traced baseline is too short for a quadratic
> $r(t)$ and which was therefore fitted with a straight line.

## A.5 &middot; Checking the fundamental / harmonic identification

The two bands are assumed to be $s=1$ and $s=2$ of one shock. Four checks on that, all made
against the data rather than assumed.

**1. Frequency ratio over the overlap.** Where both bands are traced, the ratio of their upstream
branches must be 2. Comparing the harmonic's upstream branch against the fundamental's *downstream*
branch instead gives $2/(1+\mathrm{BDW})$, so the branches have to be paired correctly for this
test to mean anything &mdash; which is why &sect;A.4 sorts them in frequency.

**2. Relative drift rate.** The absolute drift $\mathrm{d}f/\mathrm{d}t$ of the harmonic is twice
the fundamental's, purely because its frequency is twice as high. The **relative** drift
$f^{-1}\mathrm{d}f/\mathrm{d}t$ removes that factor and is a property of the shock alone, so the two
bands must return the same value &mdash; evaluated over the same interval, since it changes through
the burst.

This one is measured on the **traced samples inside the overlap**, not on the lane fits, and the
distinction decides the answer. Each lane fit is a single cubic over that lane's whole length, and
a polynomial's derivative is at its least reliable near the ends of its own range. The overlap is
the worst case available: it is the last third of the fundamental's fit and the first eighth of the
harmonic's, so asking the two cubics for a drift there compares one curve's tail against another's
nose. On this event that choice alone opened a 15% gap and reported it at 5.6$\sigma$; the same
harmonic cubic returns $-7.9\times10^{-4}$ over its own full span against $-1.25\times10^{-3}$ on
the overlap, which is the size of the swing involved. Fitting $\log f$ linearly against time on the
points that actually lie in the window &mdash; over 191&nbsp;s the lane is straight enough in
$\log f$ for a line to be the right model &mdash; removes the extrapolation, and the two bands then
agree. Both numbers are printed, and the cubic-derived one is labelled a diagnostic.

The same applies to test 1: the ratio ramp is built from those two cubics, so it inherits the same
edge behaviour, and the ramp measured on the traced points is quoted beside it.

**3. Height agreement.** Converting each band with its own $s$ must place both on the same
height-time trajectory. This is not an independent test of the density model &mdash; any model maps
$f_H/2$ and $f_F$ to the same density &mdash; but it does verify the harmonic assignment and shows
how well the two segments join where they overlap.

**4. Circular polarisation.** The Stokes V/I frame is sampled in a
$\pm$`POL_DT_S`&nbsp;s&nbsp;&times;&nbsp;$\pm$`POL_DF_MHZ`&nbsp;MHz box along every traced lane.
The number to compare between bands is the **signed** mean. Averaging $|V/I|$ instead measures the
noise, not the polarisation: for a signal at or below the noise, $\langle|x|\rangle\to0.8\sigma$
whatever the true mean is, so two bands with opposite signed polarisation return the same
$|V/I|$. Both are tabulated, but only the signed mean is used for the comparison.
Fundamental plasma emission escapes in the o-mode and is expected to be the more strongly polarised
of the two; harmonic emission comes from a coalescence that largely cancels the sense of
polarisation. That is the expectation, and the cell below reports how far this event is from being
able to test it: the band-to-band difference is compared against the point-to-point scatter along a
lane, and unless it clears that scatter by a factor of two the contrast is reported as not
significant and the identification is not allowed to lean on it. The scatter is used undivided,
because the samples along a B&eacute;zier lane are not independent and neither are the overlapping
V/I boxes of adjacent samples; and it is still an underestimate, since it contains no
instrumental-leakage term. `polarisation_caveats` tests for that separately, by checking whether
two lanes agree far more closely than either is individually determined &mdash; the signature of a
common offset rather than of lane-specific polarisation. This test is the weakest of the four.

In [ ]:
POL_T = POL.index.to_numpy()
POL_F = np.asarray(POL.columns, float)
POL_V = POL.to_numpy()


def sample_polarisation(t_list, f_list, dt_s=POL_DT_S, df_mhz=POL_DF_MHZ):
    """Median Stokes V/I in a (+/- dt_s, +/- df_mhz) box around each traced (t, f) point, taken
    from the decimated V/I layer."""
    tv = pd.to_datetime(pd.Series(t_list)).to_numpy()
    fv = np.asarray(f_list, float)
    half = np.timedelta64(int(dt_s * 1e3), 'ms')
    out = np.full(len(fv), np.nan)
    for k, (tt, ff) in enumerate(zip(tv, fv)):
        # side='right' on the upper edge closes the interval. With the default the box is
        # [t-dt, t+dt), so it holds one fewer sample on the late side than the early side and its
        # centre of mass sits half a sample early - a systematic offset along a drifting lane,
        # not a rounding detail.
        i0 = np.searchsorted(POL_T, tt - half, side='left')
        i1 = max(np.searchsorted(POL_T, tt + half, side='right'), i0 + 1)
        j = np.abs(POL_F - ff) <= df_mhz
        if j.any():
            blk = POL_V[i0:i1][:, j]
            if blk.size:
                out[k] = np.nanmedian(blk)
    return out


def polarisation_caveats(tab, tol=2e-4):
    """Flag lanes whose signed mean V/I agree to more digits than either is determined to.

    Two independent lanes landing on the same value to four significant figures is the signature
    of a common additive offset - instrumental leakage - rather than of lane-specific
    polarisation. Worth catching, because at the ~1% level measured on this event the leakage
    floor is not obviously smaller than the signal."""
    notes = []
    v = tab.set_index('lane')['V_over_I_mean']
    for a in v.index:
        for b in v.index:
            if a < b and abs(v[a] - v[b]) < tol:
                notes.append(f'{a} and {b} agree to {abs(v[a] - v[b]):.1e} in signed mean V/I, '
                             'much closer than either is individually determined - consistent '
                             'with a common instrumental offset, not lane-specific polarisation')
    return notes


pol_rows = []
POL_TRACK = {}
for lab in TRACED:
    ref = next(p[lab] for p in passes if p.get(lab, {}).get('f'))
    # resample the lane fit on a dense grid so the polarisation is not read at the click points only
    fit = _lane_fit(ref)
    ts = np.linspace(fit['tmin'], fit['tmax'], 120)
    tt = [t0 + pd.Timedelta(seconds=float(s)) for s in ts]
    ff = lane_deriv(fit, ts)[0]
    p = sample_polarisation(tt, ff)
    POL_TRACK[lab] = dict(t=pd.to_datetime(tt), f=ff, p=p)
    n = int(np.isfinite(p).sum())
    pol_rows.append({'lane': lab, 'band': LANE_BAND[lab], 'role': LANE_ROLE[lab],
                     'V_over_I_mean': np.nanmean(p), 'V_over_I_sd': np.nanstd(p),
                     'V_over_I_sem': np.nanstd(p) / np.sqrt(max(n, 1)),
                     'abs_V_over_I_mean': np.nanmean(np.abs(p)),
                     'abs_V_over_I_max': np.nanmax(np.abs(p)), 'n_samples': n})

pol_table = pd.DataFrame(pol_rows)
pol_table.to_csv(os.path.join(OUTDIR, 'typeii_polarisation.csv'), index=False)

# The SIGNED mean is the physical quantity. Taking |V/I| first and then averaging measures the
# noise, not the polarisation: for a signal buried in noise, mean|x| -> 0.8 sigma whatever the
# true mean is, so two bands with opposite signed polarisation report the same |V/I|.
pF = pol_table[pol_table['band'] == 'F']['V_over_I_mean'].mean()
pH = pol_table[pol_table['band'] == 'H']['V_over_I_mean'].mean()
noise = pol_table['V_over_I_sd'].median()
print(f'signed mean V/I:  fundamental {pF:+.4f},  harmonic {pH:+.4f}   '
      f'(point-to-point scatter along a lane {noise:.4f})')
# The bar for calling the contrast real. |pF - pH| is measured against the point-to-point scatter
# along a lane, undivided: dividing by sqrt(N) would assume independent samples, and neither the
# points along a Bezier curve nor the overlapping V/I boxes of neighbouring points are independent.
# Even undivided it is an UNDERSTATEMENT, because it carries no instrumental-leakage term - which is
# what polarisation_caveats() tests for, and why a caveat firing withdraws the claim outright.
_dpol = abs(pF - pH)
_nsig = _dpol / noise if noise > 0 else np.nan
_pol_caveats = polarisation_caveats(pol_table)
print(f'  the two bands differ by {_dpol:.4f}, which is {_nsig:.1f}x that scatter')
if not np.isfinite(_nsig) or _nsig < 2 or _pol_caveats:
    print('  that is NOT a significant contrast. Fundamental plasma emission escapes in the o-mode '
          'and is expected to be the')
    print('  more strongly polarised of the two, but these numbers neither confirm nor contradict '
          'that expectation, and')
    print('  the band identification below does not rest on them.')
elif abs(pF) > abs(pH):
    print('  the fundamental is the more strongly polarised, which is the sense expected for '
          'o-mode fundamental emission')
else:
    print('  the harmonic is the more strongly polarised - unusual, worth a second look at the '
          'tracing and at the RFI stripes in panel (a)')
print(f'mean |V/I| is also tabulated but do NOT use it to compare the bands: for |V/I| ~ the '
      f'noise it just returns the noise level ({0.8 * noise:.4f} here).')
for _n in _pol_caveats:
    print(f'  *** WARNING: {_n}')
print('V/I here is the mean of an ALREADY-DIVIDED ratio with no instrumental-leakage floor '
      'removed. Where Stokes I is near')
print('the background that ratio is noise-dominated and averaging it pulls the result toward the '
      'noise mean, diluting the')
print('band-to-band contrast. At the ~1% level measured here, treat these as an upper bound on a '
      'polarisation contrast,')
print('not as a detection.')
pol_table.round(4)

> **Table 3 &mdash; Circular polarisation along each lane.** `typeii_polarisation.csv`
>
> The Stokes V/I frame sampled in a $\pm$`POL_DT_S`&nbsp;s&nbsp;$\times$&nbsp;$\pm$`POL_DF_MHZ`&nbsp;MHz
> box centred on every point of the fitted lane: the signed mean, its scatter and standard error,
> the mean and maximum of $|V/I|$, and the number of samples.
>
> *Use the signed mean, not $|V/I|$.* For a signal at or below the noise
> $\langle|x|\rangle\rightarrow0.8\sigma$ regardless of the true mean, so $|V/I|$ returns the noise
> level for every band and destroys exactly the contrast you are looking for. Fundamental emission
> escapes in the o-mode and is expected to be the more strongly polarised of the two.

In [ ]:
# ---- tests 1-3: ratio, relative drift and height agreement over the overlap ----
_fits = pass_fits(passes[0])
_rr, _ne = _invert_grid(MODEL_GRID[REF_MODEL_NAME])
fh_rows = []


def _height(fit, band, ts):
    f = lane_deriv(fit, ts)[0]
    ne_t = freq_to_density(f * 1e6, harmonic=HARM[band])
    r = np.interp(ne_t, _ne[::-1], _rr[::-1])
    r[(ne_t > np.nanmax(_ne)) | (ne_t < np.nanmin(_ne))] = np.nan
    return f, r


def rel_drift_on_points(lab, lo, hi):
    """Relative drift measured straight off the traced samples that fall inside [lo, hi].

    A lane fit is one cubic spanning that lane's whole traced length, and its DERIVATIVE near the
    ends of that span is the least constrained quantity it produces. The F/H overlap is the worst
    possible place to ask two of them for a derivative: on this event it is the last 32% of the
    fundamental's fit and the first 13% of the harmonic's, so comparing the cubics there compares
    one curve's tail against another's nose. That alone opened a 15% gap between two bands of the
    same shock and reported it at 5.6 sigma. Evaluating the harmonic's cubic over its own full
    span instead gives -0.00079 1/s against -0.00125 on the overlap, which is the size of the
    swing involved.

    Fitting log f linearly against time on the points that actually lie in the window removes the
    extrapolation entirely. Over a couple of hundred seconds the lane is straight enough in log f
    that a line is the right model, and the answer then depends only on samples inside the window.

    Returns the mean slope over passes, its standard error, the number of points used, and
    separately the pass-to-pass spread - which for jittered Bezier repeats measures the jitter and
    not the reproducibility, so it is reported but not used as the error."""
    slopes, ses, npts = [], [], 0
    for pas in passes:
        p = pas.get(lab)
        if not p:
            continue
        tt = (pd.to_datetime(p['t']) - t0).total_seconds().to_numpy()
        ff = np.asarray(p['f'], float)
        m = (tt >= lo) & (tt <= hi) & np.isfinite(ff) & (ff > 0)
        if m.sum() < 3:
            continue
        x, y = tt[m], np.log(ff[m])
        sig = LANE_SIGMA.get(lab)
        w = (ff[m] / sig) if sig else None               # d(ln f) = df / f
        try:
            c, cv = np.polyfit(x, y, 1, w=w, cov='unscaled') if w is not None \
                else np.polyfit(x, y, 1, cov=True)
            ses.append(float(np.sqrt(cv[0, 0])))
        except (ValueError, np.linalg.LinAlgError):
            c = np.polyfit(x, y, 1)
            ses.append(np.nan)
        slopes.append(float(c[0]))
        npts = int(m.sum())
    if not slopes:
        return np.nan, np.nan, 0, np.nan
    se = float(np.sqrt(np.nanmean(np.asarray(ses) ** 2))) if np.any(np.isfinite(ses)) else np.nan
    spread = float(np.std(slopes, ddof=1)) if len(slopes) > 1 else 0.0
    return float(np.mean(slopes)), se, npts, spread


fF = _fits.get(SPLIT_PAIR['F'][0]) if 'F' in SPLIT_PAIR else None
fH = _fits.get(SPLIT_PAIR['H'][0]) if 'H' in SPLIT_PAIR else None
OVERLAP = None
if fF is None or fH is None:
    print('both bands need at least one traced lane for the F/H consistency tests - skipped')
else:
    lo, hi = max(fF['tmin'], fH['tmin']), min(fF['tmax'], fH['tmax'])
    OVERLAP = (lo, hi) if hi > lo else None
    if OVERLAP is None:
        print(f'the two bands do not overlap in time ({lo - hi:.0f} s apart); the ratio below is '
              'EXTRAPOLATED and is much weaker evidence. Re-trace so the bands share some time '
              'if the spectrum allows it.')
        lo, hi = min(fF['tmin'], fH['tmin']), max(fF['tmax'], fH['tmax'])
    ts = np.linspace(lo, hi, 60)
    f_F, r_F = _height(fF, 'F', ts)
    f_H, r_H = _height(fH, 'H', ts)
    ratio = f_H / f_F
    dres = r_H - r_F
    # the relative drift must be compared ON THE OVERLAP: it changes through the burst, and the
    # band-averaged values in `scalars` are averages over two different stretches of it
    rdF_fit = np.nanmean(lane_deriv(fF, ts)[2])
    rdH_fit = np.nanmean(lane_deriv(fH, ts)[2])
    rdF, rdF_e, rdF_n, rdF_sp = rel_drift_on_points(SPLIT_PAIR['F'][0], lo, hi)
    rdH, rdH_e, rdH_n, rdH_sp = rel_drift_on_points(SPLIT_PAIR['H'][0], lo, hi)
    if not (np.isfinite(rdF) and np.isfinite(rdH)):      # too few points in the window
        rdF, rdF_e = rdF_fit, np.nan
        rdH, rdH_e = rdH_fit, np.nan

    print(f'{SPLIT_PAIR["F"][0]} vs {SPLIT_PAIR["H"][0]}, '
          f'{(t0 + pd.Timedelta(seconds=lo)).strftime("%H:%M:%S")}'
          f'-{(t0 + pd.Timedelta(seconds=hi)).strftime("%H:%M:%S")} UT  ({hi - lo:.0f} s)\n')
    # If the ratio misses 2, say by how much the two bands would have to be offset IN TIME to
    # meet it. "1.94 instead of 2.00" is a number nobody can act on; "the two bands agree if the
    # harmonic runs 21 s ahead" is a statement about the source that can be checked against the
    # spectrum, and it separates a genuine harmonic-number error (which no shift can repair)
    # from two lanes tracking one shock at slightly different places on the front.
    _shifts = np.arange(-180, 181, 1)
    _obj = [abs(np.nanmean(_height(fH, 'H', ts)[0]
                           / lane_deriv(fF, ts + d)[0]) - 2) for d in _shifts]
    _best = float(_shifts[int(np.nanargmin(_obj))])
    _resid = float(np.nanmin(_obj))
    _n_ok = max(int(np.isfinite(ratio).sum()), 1)
    _sem = float(np.nanstd(ratio) / np.sqrt(_n_ok))
    print(f'1. frequency ratio   f_H / f_F = {np.nanmean(ratio):.3f} +/- {_sem:.3f} (s.e.m.)'
          '        (2.000 expected)')
    print(f'   the ratio RAMPS from {ratio[0]:.3f} to {ratio[-1]:.3f} across the overlap instead '
          'of scattering about a constant,')
    print('   so its spread is not an uncertainty. That ramp is exactly test 2: '
          'd ln(f_H/f_F)/dt = rel drift H - rel drift F.')
    print('   Tests 1 and 2 are one statement seen twice - do not count them as two independent '
          'checks.')
    # Both f_F and f_H here are the two cubics, so the ramp inherits their edge behaviour. Quote
    # what the ramp would be from the drifts measured on the traced points, for comparison.
    if np.isfinite(rdF) and np.isfinite(rdH):
        _ramp = float(np.nanmean(ratio)) * np.exp((rdH - rdF) * np.array([-.5, .5]) * (hi - lo))
        print(f'   measured on the traced points instead of the fitted cubics, the same ramp is '
              f'{_ramp[0]:.3f} to {_ramp[1]:.3f}')
    if abs(np.nanmean(ratio) - 2) > 2 * _sem:
        # convert the shift into a distance using the speed measured on this very track
        _vF = float(np.nanmean(np.gradient(r_F, ts))) * R_SUN_M / 1e3          # km/s
        _dr = abs(_best) * _vF * 1e3 / R_SUN_M
        print(f'   the ratio reaches 2.000 (to {_resid:.4f}) if the fundamental is evaluated '
              f'{_best:+.0f} s later than the harmonic.')
        print(f'   At the {_vF:.0f} km/s measured on this track that is {_dr:.3f} Rsun of travel: '
              'small enough for two source')
        print('   points on one shock front, too large to be rounding. A wrong harmonic number '
              'would NOT be repaired')
        print('   by any shift, so a small best-fit offset supports the F/H identification '
              'rather than undermining it.')
    _dsig = np.hypot(rdF_e, rdH_e)
    print(f'2. relative drift    over the overlap, measured on the TRACED POINTS in that window '
          f'({rdF_n} F, {rdH_n} H samples per pass)')
    print(f'                     F: {rdF:+.6f} +/- {rdF_e:.6f} 1/s,   '
          f'H: {rdH:+.6f} +/- {rdH_e:.6f} 1/s')
    print(f'                     differ by {100 * abs(rdH - rdF) / abs(rdF):.1f}% = '
          f'{abs(rdH - rdF) / _dsig:.1f} sigma; f^-1 df/dt is free of the harmonic number, so '
          f'these must agree')
    print(f'                     the fitted cubics give {rdF_fit:+.6f} and {rdH_fit:+.6f} '
          f'({100 * abs(rdH_fit - rdF_fit) / abs(rdF_fit):.1f}% apart) - NOT the number to quote.')
    print(f'                     The overlap is the last '
          f'{100 * (fF["tmax"] - lo) / (fF["tmax"] - fF["tmin"]):.0f}% of the fundamental\'s fit '
          f'and the first {100 * (hi - fH["tmin"]) / (fH["tmax"] - fH["tmin"]):.0f}% of the '
          f'harmonic\'s,')
    print('                     so those two derivatives are the least constrained values either '
          'cubic produces.')
    print(f'3. height residual   r_H - r_F = {np.nanmean(dres):+.4f} +/- {np.nanstd(dres):.4f} Rsun'
          f'   ({100 * np.nanmean(np.abs(dres)) / np.nanmean(r_F):.2f}% of the height)')
    print(f'4. band splits       X_F = {scalars["F"]["X"][0]:.3f} +/- {scalars["F"]["X"][1]:.3f},   '
          f'X_H = {scalars["H"]["X"][0]:.3f} +/- {scalars["H"]["X"][1]:.3f}   '
          '(same shock -> same density jump)')

    # Each row is self-contained. `kind` says whether it is a raw MEASUREMENT or a
    # CONSISTENCY TEST; only tests have an `expected` value and an n_sigma, and a blank expected
    # means the row is a measurement with nothing to compare against, not a missing number.
    def _row(kind, test, value, error, expected=np.nan, note=''):
        n = (abs(value - expected) / error) if (np.isfinite(expected) and np.isfinite(error)
                                                and error > 0) else np.nan
        return {'kind': kind, 'test': test, 'value': value, 'error': error,
                'expected': expected, 'n_sigma': n,
                'verdict': ('' if not np.isfinite(n) else
                            ('FLAG' if n > CONSIST_SIGMA else 'ok')), 'note': note}

    _eX = np.hypot(scalars['F']['X'][1], scalars['H']['X'][1])
    fh_rows = [
        # The uncertainty is the STANDARD ERROR of the mean, not the scatter along the overlap.
        # The ratio does not scatter about a constant, it RAMPS, because the two bands have
        # different relative drifts - and feeding that ramp in as an error made this test pass
        # automatically whenever the drift test failed. The ramp is reported as a range instead.
        _row('consistency test', 'f_H / f_F over the overlap', float(np.nanmean(ratio)),
             float(np.nanstd(ratio) / np.sqrt(max(int(np.isfinite(ratio).sum()), 1))), 2,
             'must be 2 for a fundamental and its harmonic. NOT independent of the relative-drift '
             'row: the ratio ramps at exactly (rel drift H - rel drift F), so read them together'),
        _row('measurement', 'f_H / f_F at the start of the overlap', float(ratio[0]), np.nan,
             np.nan, 'the ratio ramps rather than scattering; start and end bracket it'),
        _row('measurement', 'f_H / f_F at the end of the overlap', float(ratio[-1]), np.nan,
             np.nan, 'the ratio ramps rather than scattering; start and end bracket it'),
        # Measured on the traced samples inside the overlap, not on the two cubics. Comparing the
        # cubics puts the fundamental's last 32% against the harmonic's first 13%, where a cubic's
        # derivative is at its least constrained, and that alone produced a 15% gap at 5.6 sigma.
        # The error is now a real one from the fits to those points, in place of a nominal 2%.
        _row('consistency test', 'relative drift, H minus F [1/s]', rdH - rdF,
             float(np.hypot(rdF_e, rdH_e)), 0,
             'f^-1 df/dt is a property of the shock, so both bands must give the same value. '
             'Measured on the traced points inside the overlap; the fitted cubics give '
             f'{rdH_fit - rdF_fit:+.6f} there, which is edge behaviour of the fits, not the burst'),
        _row('measurement', 'relative drift, H minus F, from the fitted cubics [1/s]',
             rdH_fit - rdF_fit, np.nan, np.nan,
             'diagnostic only - shows how far two cubics disagree when each is evaluated near the '
             'end of its own span; the row above is the measurement'),
        _row('consistency test', 'height residual r_H - r_F [Rsun]', float(np.nanmean(dres)),
             float(np.nanstd(dres)), 0,
             'converting each band with its own harmonic number must put them at the same height'),
        _row('consistency test', 'density jump, X_H minus X_F',
             scalars['H']['X'][0] - scalars['F']['X'][0], _eX, 0,
             'X is model-independent and both bands measure the same shock'),
        _row('measurement', 'relative drift F, overlap [1/s]', rdF, rdF_e, np.nan,
             f'log-linear fit to the {rdF_n} traced samples per pass inside the overlap'),
        _row('measurement', 'relative drift H, overlap [1/s]', rdH, rdH_e, np.nan,
             f'log-linear fit to the {rdH_n} traced samples per pass inside the overlap'),
        _row('measurement', 'density jump X (F)', *scalars['F']['X'], np.nan,
             'from the F band split alone'),
        _row('measurement', 'density jump X (H)', *scalars['H']['X'], np.nan,
             'from the H band split alone'),
        _row('measurement', 'signed mean V/I (F)', pF, np.nan, np.nan,
             'scatter along the lane is in typeii_polarisation.csv'),
        _row('measurement', 'signed mean V/I (H)', pH, np.nan, np.nan,
             'scatter along the lane is in typeii_polarisation.csv'),
    ]

fh_test = pd.DataFrame(fh_rows)
if len(fh_test):
    fh_test.to_csv(os.path.join(OUTDIR, 'fundamental_harmonic_tests.csv'), index=False)
    print('\nkind = "consistency test" rows have an expected value and an n_sigma; '
          '"measurement" rows do not.')
    print('a blank expected/n_sigma means there is nothing to compare that number against, '
          'not a missing result.')
fh_test.round(4)

> **Table 4 &mdash; Fundamental / harmonic consistency.** `fundamental_harmonic_tests.csv`
>
> `kind` separates the two sorts of row. A **consistency test** compares two measurements of the
> same physical quantity and therefore has an `expected` value, an `n_sigma` and a verdict:
> $f_H/f_F$ must be 2, the two bands' relative drifts must be equal, the height residual
> $r_H-r_F$ must be zero, and $X_H-X_F$ must be zero. A **measurement** row is a raw number with
> nothing to compare it against, so its `expected` and `n_sigma` are deliberately blank &mdash;
> that is not a missing result.
>
> *This is the check that the whole F/H interpretation rests on.* A `FLAG` here means a lane is
> mis-traced or a branch is misassigned, and it must be resolved before any of these numbers are
> used.

In [ ]:
# ---- consistency audit: quantities that MUST agree if the physics is right ----------------
# Everything below compares two measurements of the same thing. A disagreement beyond
# CONSIST_SIGMA is not a rounding issue, it means a lane is mis-traced or a branch is misassigned,
# and it has to be resolved before any of these numbers are used.
def _pair(a, ea, b, eb):
    d = a - b
    ed = np.hypot(ea if np.isfinite(ea) else 0, eb if np.isfinite(eb) else 0)
    return d, ed, (abs(d) / ed if ed > 0 else np.inf)


audit = []
WIDENING_EXPLAINS = {}       # band -> (separation rate km/s, n_sigma) when it accounts for a flag


def _check(name, a, ea, b, eb, what, tol=CONSIST_SIGMA):
    d, ed, n = _pair(a, ea, b, eb)
    audit.append({'check': name, 'value_1': a, 'value_2': b, 'difference': d,
                  'combined_error': ed, 'n_sigma': n, 'verdict': 'FLAG' if n > tol else 'ok',
                  'why_it_matters': what})


# 1. the two bands are the same shock, so the density jump and the relative drift must match
if len(BANDS_TRACED) > 1 and all(np.isfinite(scalars[b]['X'][0]) for b in BANDS_TRACED):
    _check('density jump X: F vs H', *scalars['F']['X'], *scalars['H']['X'],
           'X is model-independent and both bands measure the same shock')
if OVERLAP is not None:
    # rdF and rdH are now measured on the traced points inside the overlap and carry real errors,
    # so the nominal 2% that used to stand in for an uncertainty here is gone.
    _check('relative drift over the overlap: F vs H', rdF, rdF_e, rdH, rdH_e,
           'f^-1 df/dt is a property of the shock, free of the harmonic number')

# 2. within a band the two split branches straddle one shock front, so their kinematics and B
#    must be close; a large disagreement means one branch is not where you think it is.
#
#    Both branches are averaged over the IDENTICAL window here. On their own spans they cover
#    different stretches of a curving track, and a speed difference would then be mostly a
#    coverage difference.
#
#    A branch-to-branch speed difference that survives that is not automatically an error: the two
#    branches separating in height IS the band split widening, which is the same fact as X rising
#    along the overlap. So the difference is also tested against d(r_lower - r_upper)/dt. When
#    those agree, the speed and acceleration flags and the "X varies along the overlap" note are
#    one measurement reported three times, and should be discussed as one.
for b in BANDS_TRACED:
    lo_, up_ = SPLIT_PAIR[b]
    if up_ is None or lo_ not in ALref or up_ not in ALref:
        continue
    _dlo, _dup = ALref[lo_], ALref[up_]
    _cmb = common_mask(_dlo)
    _cmu = common_mask(_dup)
    if _cmb is None or _cmu is None:
        continue
    _cmb = _cmb & _cmu
    if _cmb.sum() < 3:
        continue
    _sep = _dlo['r_mean'][_cmb] - _dup['r_mean'][_cmb]
    _dsep = float(np.polyfit(tg[_cmb], _sep, 1)[0]) * R_SUN_M / 1e3          # km/s
    # B is deliberately absent: it is only defined for the upstream branch (see A.10), so a
    # branch-to-branch comparison of it is not a consistency test.
    for key, lab in [('v', 'shock speed'), ('a', 'acceleration')]:
        if key == 'a' and not (A_MEASURED.get(lo_, False) and A_MEASURED.get(up_, False)):
            continue          # at least one branch's curvature does not clear its own error bar,
                              # so comparing the two would be comparing noise
        m1, e1 = grid_scalar(_dlo, key, mask=_cmb)
        m2, e2 = grid_scalar(_dup, key, mask=_cmb)
        if np.isfinite(m1) and np.isfinite(m2):
            _check(f'{lab}: {lo_} vs {up_}', m1, e1, m2, e2,
                   'the two split branches of one band bracket the same shock front '
                   '(both averaged over the same window)')
        if key == 'v' and np.isfinite(m1) and np.isfinite(m2):
            _check(f'band split widening explains the {b} speed difference',
                   m1 - m2, np.hypot(e1, e2), _dsep, np.nan,
                   f'the {b} branches separate at {_dsep:+.0f} km/s, i.e. the split is widening '
                   f'(X runs {scalars[b]["X_range"][0]:.3f} to {scalars[b]["X_range"][1]:.3f} '
                   f'along the overlap). If this row passes, the speed and acceleration rows '
                   f'above are that same widening and not independent inconsistencies')
            if audit[-1]['verdict'] == 'ok':
                WIDENING_EXPLAINS[b] = (_dsep, float(audit[-1]['n_sigma']))

audit = pd.DataFrame(audit)
if len(audit):
    audit.to_csv(os.path.join(OUTDIR, 'consistency_audit.csv'), index=False)
    for _, r in audit[audit.verdict == 'FLAG'].iterrows():
        # a flag on a split pair whose separation rate accounts for the difference is not a fourth
        # independent problem; it is the widening, already reported, seen through another quantity
        _b = next((k for k in WIDENING_EXPLAINS
                   if k in BANDS_TRACED and SPLIT_PAIR.get(k, (None,))[0] in str(r['check'])),
                  None)
        print(f'  *** FLAG: {r["check"]} differ by {r["n_sigma"]:.1f} sigma '
              f'({r["value_1"]:.3g} vs {r["value_2"]:.3g}). {r["why_it_matters"]}.')
        if _b is not None:
            _d, _n = WIDENING_EXPLAINS[_b]
            print(f'            Accounted for: the {_b} branches are separating at {_d:+.0f} km/s '
                  f'(the split widening), which matches this difference to {_n:.1f} sigma.')
            print(f'            Report it as the band split widening through the event, not as '
                  f'{_b} lane 1 and {_b} lane 2 disagreeing about one shock.')
    if not (audit.verdict == 'FLAG').any():
        print('  all consistency checks pass')

# 3. is the acceleration meaningful? Two separate questions, asked separately: does it clear its
#    own error bar, and is its size physically plausible. Every lane is fitted at KIN_DEG, so
#    every lane gets an answer rather than being excluded in advance by its traced duration.
print()
print(f'acceleration: significance (|a| > {KIN_A_SIGMA} sigma), method bias floor, plausibility')
print('the bias floor is what this chain returns for a shock at EXACTLY constant speed over the')
print('same span - a property of the method that no error bar can reveal')
for lab in TRACED:
    d = ALref.get(lab)
    if d is None:
        continue
    a_, ea_ = grid_scalar(d, 'a')
    if not np.isfinite(a_):
        print(f'  {lab:10s} no acceleration: the density model does not resolve this lane')
        continue
    base = KIN_BASELINE.get(lab, np.nan)
    sig = abs(a_) / ea_ if (np.isfinite(ea_) and ea_ > 0) else np.nan
    bias = A_BIAS.get(lab, np.nan)
    tag = []
    if not accel_is_measured(a_, ea_, bias=bias):
        if np.isfinite(ea_) and ea_ > 0 and abs(a_) < KIN_A_SIGMA * ea_:
            tag.append(f'NOT SIGNIFICANT at {sig:.1f} sigma')
        else:
            tag.append(f'BELOW THE METHOD BIAS FLOOR of {abs(bias):.0f} m/s^2')
    if abs(a_) > A_PLAUSIBLE_MS2:
        tag.append(f'|a| = {abs(a_):.0f} m/s^2 is {abs(a_) / 50:.0f}x a typical coronal value')
    print(f'  {lab:10s} a = {a_:+8.1f} +/- {ea_:5.1f} m/s^2 over {base:5.0f} s '
          f'({sig:4.1f} sigma, bias floor {bias:+6.1f})'
          + ('   <-- ' + '; '.join(tag) if tag else '   (ok)'))
print('a is the second derivative of a height track whose curvature is set largely by the assumed')
print('density profile, so it carries a systematic far larger than the error bars above: see the')
print('model x fold range in A.8 before quoting it.')

> **Table 5 &mdash; Consistency audit.** `consistency_audit.csv`
>
> Every pair of quantities that must agree if the physics is right, with the difference, the
> combined error, the significance and a verdict. Two families: **between bands** (the density jump
> and the relative drift, both model-independent), and **within a band** (the two split branches
> straddle one shock front, so their speed, acceleration and $B$ must be close).
>
> Below it, an **acceleration reality check**: $|a|$ against a plausibility threshold, the traced
> baseline it came from, and an explicit `NOT MEASURED` for any lane whose $r(t)$ was fitted with a
> straight line. Statistical significance is not the issue for $a$ &mdash; its curvature is set
> largely by the assumed density profile, so it carries a systematic far larger than its error bar.

In [ ]:
fig = plt.figure(figsize=[15, 11])

# (a) the V/I spectrogram with every traced lane on top
ax = fig.add_subplot(311)
step = max(1, POL.shape[0] // PREVIEW_MAX_TCOLS)
pv = np.nanpercentile(np.abs(POL.to_numpy()), 99)
pm = ax.pcolormesh(POL.index[::step], np.asarray(POL.columns, float), POL.to_numpy().T[:, ::step],
                   vmin=-pv, vmax=pv, cmap='seismic', rasterized=True)
fig.colorbar(pm, ax=ax, pad=0.01, label='Stokes V/I')
for lab in TRACED:
    ax.plot(POL_TRACK[lab]['t'], POL_TRACK[lab]['f'], color='k', lw=1.8)
    ax.plot(POL_TRACK[lab]['t'], POL_TRACK[lab]['f'], color=LANE_COL[lab], lw=1.1, label=lab)
if INVERT_FREQ:
    ax.invert_yaxis()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
ax.set_ylabel('Frequency (MHz)')
ax.set_title('(a) Circular polarisation with the traced lanes overlaid')
ax.legend(fontsize=8, ncol=4, loc='lower left')

# (b) V/I along each lane, F against H
ax = fig.add_subplot(312)
for lab in TRACED:
    d = POL_TRACK[lab]
    ax.plot(d['t'], d['p'], 'o', ms=3, color=LANE_COL[lab], alpha=0.45)
    ax.plot(d['t'], sg_smooth(d['p'], window=11), '-', lw=2, color=LANE_COL[lab],
            label=f'{lab}: {np.nanmean(d["p"]):+.3f} $\\pm$ {np.nanstd(d["p"]):.3f}')
ax.axhline(0, color='0.6', lw=0.8)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
ax.set_ylabel('Stokes V/I along the lane')
ax.set_title('(b) Degree of circular polarisation: signed mean $V/I$ = '
             f'{pF:+.3f} (F) vs {pH:+.3f} (H)')
ax.legend(fontsize=8, ncol=4)
ax.grid(alpha=0.3)

# (c) the two height tracks, which must be one trajectory
ax = fig.add_subplot(313)
for b, col in zip(BANDS_TRACED, ['navy', 'firebrick']):
    fit = _fits.get(SPLIT_PAIR[b][0])
    if fit is None:
        continue
    ts = np.linspace(fit['tmin'], fit['tmax'], 80)
    _, r = _height(fit, b, ts)
    ax.plot(ts, r, '-', lw=2.4, color=col, label=f'{BAND_NAME[b]} ($s$ = {HARM[b]})')
d = ALref.get(JOINT)
if d is not None:
    ax.errorbar(tg, d['r_mean'], yerr=d['r_se'], fmt='o', ms=3.5, color='0.35', capsize=2,
                alpha=0.75, label=JOINT, zorder=0)
if OVERLAP is not None:
    ax.axvspan(*OVERLAP, color='0.85', alpha=0.6, zorder=-1)
    ax.text(np.mean(OVERLAP), ax.get_ylim()[0], 'overlap', ha='center', va='bottom', fontsize=9)
ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
ax.set_ylabel(r'$r\,/\,R_\odot$')
ax.set_title(f'(c) Both bands on one height-time trajectory ({REF_MODEL_NAME})')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

fig.tight_layout()
save_fig(fig, 'fundamental_harmonic_checks')
plt.show()

> **Figure 5 &mdash; Fundamental / harmonic checks.** `fundamental_harmonic_checks.png`
>
> *(a)* The Stokes V/I dynamic spectrum with every traced lane overlaid, so you can see at a glance
> whether a lane crosses an RFI stripe. *(b)* The degree of circular polarisation along each lane,
> points and a Savitzky-Golay trend, with the signed mean and scatter per lane in the legend.
> *(c)* Both bands converted to height with their own harmonic number, on one axis, with the
> combined track and the interval in which the two overlap shaded.
>
> *What to look for:* in (c) the two segments must lie on a single continuous trajectory. They
> will, for any density model, provided the harmonic assignment is right &mdash; that is what makes
> this a test of the identification rather than of the model.

## A.6 &middot; Kinematics and coronal magnetic field

Height, shock speed, acceleration and $B$ for every traced lane and for the combined `F+H joint`
track, on the reference model, with the Monte-Carlo $\pm$SEM error bars and a Savitzky-Golay trend
through each. The legend quotes the grid-averaged value. The joint track is the one to quote: it
uses the fundamental early in the burst and the harmonic late, and so covers the full interval over
which the shock was observed.

Two things to read carefully here. **The error bars are statistical only** &mdash; the tracing
repeats plus the polynomial covariance. They are small because the height is a weak function of
frequency in an exponential corona; the uncertainty that actually matters is the density model,
and that is the model&times;fold range in &sect;A.8. Quote both. **Acceleration is constant per
track by construction**, because `KIN_DEG = 2` fits $r(t)$ with a quadratic. That is deliberate: a
cubic bends the height track into an S and then reports a spurious $\sim10^3$ m s$^{-2}$ near the
ends. If you want a time-dependent acceleration, take it from the Byrne fit in &sect;A.7, which is
non-parametric, or raise `KIN_DEG` knowing what it does.

Each track is drawn only over the interval it was traced; nothing is extrapolated.

In [ ]:
def _has(key, k):
    return key in ALref and np.isfinite(ALref[key][k + '_mean']).sum() > 2


def track_panel(ax, key_list, k, ylabel, unit, fmt='{:.3g}'):
    for key in key_list:
        if not _has(key, k):
            continue
        c = LANE_COL[key]
        m, err = ALref[key][k + '_mean'], ALref[key][k + '_se']
        joint = key == JOINT
        ax.errorbar(tg, m, yerr=err, fmt='o', ms=3, color=c, ecolor=c, elinewidth=0.7,
                    capsize=1.5, alpha=(0.6 if joint else 0.4))
        mval, eval_ = np.nanmean(m), np.sqrt(np.nanmean(err ** 2))
        lbl = f'{key}: {fmt.format(mval)} $\\pm$ {fmt.format(eval_)} {unit}'
        # An acceleration that does not clear its own error bar is still plotted and still
        # labelled with its value: marking it "n.s." says what the reader needs without
        # pretending the lane produced nothing.
        if k == 'a' and not accel_is_measured(mval, eval_, bias=A_BIAS.get(key, 0)):
            lbl += '  (n.s.)'
        ax.plot(tg, sg_smooth(m), '-', color=c, lw=(2.6 if joint else 1.6),
                zorder=(5 if joint else 2), label=lbl)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)


fig = plt.figure(figsize=[14, 9])

ax = fig.add_subplot(221)
track_panel(ax, TRACKS, 'r', r'$r\,/\,R_\odot$', r'$R_\odot$', '{:.3f}')
ax.set_title(f'(a) Height-time  ({REF_MODEL_NAME})')

ax = fig.add_subplot(222)
track_panel(ax, TRACKS, 'v', r'$v_{\rm sh}$ [km s$^{-1}$]', 'km/s', '{:.0f}')
ax.set_title('(b) Shock speed')

ax = fig.add_subplot(223)
track_panel(ax, TRACKS, 'a', r'$a$ [m s$^{-2}$]', r'm s$^{-2}$', '{:.1f}')
ax.axhline(0, color='0.6', lw=0.8)
ax.set_title('(c) Acceleration')

ax = fig.add_subplot(224)
# UPSTREAM branches only, matching the table in A.10. v_A = v_sh/M_A is the upstream Alfven speed,
# so pairing it with a downstream branch's density gives B_1 sqrt(X) - neither the upstream field
# nor the downstream one. Plotting all four lanes here invited exactly the branch-to-branch
# comparison that is not meaningful.
_B_TRACKS = [k for k in TRACKS if LANE_ROLE.get(k, '').startswith('upstream') or k == JOINT]
track_panel(ax, _B_TRACKS, 'B', r'$B$ [G]', 'G', '{:.3f}')
ax.set_title('(d) Coronal magnetic field  (upstream branches only)')
if not any(_has(k, 'B') for k in TRACKS):
    # B needs M_A, which needs 1 <= X < 4. Say so in the panel instead of leaving it blank.
    bad = [f'{BAND_NAME[b]}: X = {scalars[b]["X"][0]:.3f}' for b in BANDS_TRACED
           if not np.isfinite(scalars[b]['M_A'][0])]
    ax.text(0.5, 0.5, 'no B: it needs $M_A$, which is only defined for $1 \\leq X < 4$\n'
            + ('\n'.join(bad) if bad else 'no band split traced'),
            transform=ax.transAxes, ha='center', va='center', fontsize=10, color='firebrick')

# Spell out what the bars are, rather than calling them "statistical". They propagate exactly one
# thing - the assumed frequency uncertainty of the traced lane - through the fit. They do not
# contain the density-model choice (a factor ~2, A.8), and with jittered repeats they do not
# contain a tracing-reproducibility term either.
_errsrc = ('bars propagate the traced-lane frequency error (LANE_SIGMA ~ '
           f'{np.nanmean([LANE_SIGMA.get(l, np.nan) for l in TRACED]):.2f} MHz) through the fit'
           + ('' if REPEATS_INDEPENDENT else ', with NO independent re-tracing term')
           + '; the density-model systematic (~2x, A.8) is NOT included')
fig.suptitle(f'Type II kinematics and coronal magnetic field, NenuFAR {EVENT_DATE}\n' + _errsrc,
             y=1.03, fontsize=11)
fig.tight_layout()
save_fig(fig, 'typeii_kinematics_Bfield')
plt.show()

> **Figure 6 &mdash; Kinematics and coronal magnetic field.** `typeii_kinematics_Bfield.png`
>
> Per lane, on the reference density model: *(a)* height, *(b)* shock speed, *(c)* acceleration,
> *(d)* coronal magnetic field. Points carry the Monte-Carlo error, the line is a Savitzky-Golay
> trend, and the legend quotes each lane's grid-averaged value. Each lane is drawn only over the
> interval it was actually traced; nothing is extrapolated.
>
> *Two things to keep in mind.* The error bars are **statistical only** &mdash; the density-model
> systematic is Figure&nbsp;8, and it is far larger. And the acceleration is **constant per lane by
> construction**, because $r(t)$ is fitted with a quadratic (`KIN_DEG = 2`); lanes shorter than
> `KIN_MIN_BASELINE_S` get a straight line and no acceleration at all.

## A.7 &middot; Height-time fitting: polynomial vs Gallagher vs Byrne

&sect;A.6 obtained $v$ and $a$ by differentiating a polynomial fit of $r(t)$. Because the
acceleration is the second derivative of a fitted curve, it is the quantity most exposed to the
choice of fitting function, so the **combined F+H height-time track** is refit three ways and
compared:

1. **Polynomial** &mdash; degree `POLY_DEG`, weighted by the height errors.
2. **Gallagher, Lawrence & Dennis (2003)** &mdash; a two-phase acceleration profile
$$a(t)=\left[\frac{1}{a_r e^{t/\tau_r}}+\frac{1}{a_d e^{-t/\tau_d}}\right]^{-1},$$
   with $v$ and $h$ obtained by integrating $a(t)$ numerically (the exact double integral, not an
   analytic approximation to it).
3. **Byrne et&nbsp;al. (2013)** &mdash; Savitzky-Golay smoothing of $h(t)$ with a bootstrap for the
   error band. This is the actual Byrne method; the arctangent height-time form sometimes
   attributed to that paper is a different thing and is not used here.

**A caveat on Gallagher that matters.** With $a_r,a_d>0$ &mdash; which the fit requires, or the
reciprocal sum diverges &mdash; the profile above is **strictly positive**. It was built to
describe the impulsive *acceleration* phase of an eruption, and it **cannot represent a
decelerating shock at all**. Fitted to a decelerating track it pins $a_r,a_d$ at their lower bound,
returns $a\simeq0$, and draws a straight line through curved data without complaint. The cell
below detects this &mdash; both by comparing $\chi^2_{\rm red}$ against the best method and by
checking whether Gallagher's $a(t)$ can go negative when the polynomial says it should &mdash; and
marks the method inapplicable. Most type II shocks decelerate once past the impulsive phase, so
**expect this warning to fire and do not quote Gallagher numbers when it does.**

Each fit is bootstrapped `N_BOOT` times by resampling the heights within their errors, giving the
shaded bands. Using the joint track matters here: the two bands separately each cover only part of
the burst, and a second derivative taken over a short segment is barely constrained.

**Nothing is finite differenced.** Each fit returns $h$, $v$ and $a$ as callables with analytic
derivatives: the polynomial is differentiated exactly, Gallagher's $a(t)$ *is* the model so $v$ and
$h$ are its exact integrals, and Byrne's derivatives come from the Savitzky-Golay filter itself
(`deriv=1`, `deriv=2`), which is the method as published. Differencing an interpolant instead
returns the slope of whichever segment the step lands in &mdash; a staircase in $v$ and delta
functions in $a$ &mdash; and differentiating a spline through SG-smoothed points twice makes the
acceleration ring.

**The bands: a parametric bootstrap.** Each fit is repeated `N_BOOT` times. On each repetition
every traced height is displaced by a draw from a Gaussian of its own 1$\sigma$ error,
$h_i\rightarrow h_i+\mathcal{N}(0,\sigma_i)$, and the curve is refitted from scratch. The band is
the standard deviation of those `N_BOOT` curves at each instant, so it propagates the height
uncertainty into $h$, $v$ and $a$ without assuming the fit is linear in its parameters. Note this
resamples from the *assumed noise model*, which is a parametric bootstrap &mdash; it is not the
non-parametric kind that resamples the data points with replacement, and it therefore inherits
whatever $\sigma_i$ you fed it. `band_scale` in the comparison table is the factor the band was multiplied by: it is
$\sqrt{\chi^2_{\rm red}}$ when `INFLATE_BY_CHI2` is on and the fit is worse than the errors
imply, and **1 otherwise, which is the default**. Inflating is the standard remedy for
underestimated errors, but for comparing models it hides the very thing you are looking for, so it
is off and `band_scale` should read 1 everywhere. A polynomial band is often narrower than its own line width: two free parameters
average the height scatter down hard, whereas Byrne is a local smoother and follows each
realisation, so its band is much the widest. That difference is information, not a plotting fault.

In [ ]:
RS_KM = R_SUN_M / 1e3

# which track the height-time fits and the model sweep use
if REF_LANE is not None:
    REF_TRACK = REF_LANE
elif MAKE_JOINT and JOINT in ALref and np.isfinite(ALref[JOINT]['r_mean']).sum() > 3:
    REF_TRACK = JOINT
else:
    _cand = [SPLIT_PAIR[b][0] for b in BANDS_TRACED] + list(TRACED)
    REF_TRACK = next((k for k in _cand
                      if k in ALref and np.isfinite(ALref[k]['r_mean']).sum() > 3), None)
if REF_TRACK is None:
    raise RuntimeError('no usable height-time track on ' + REF_MODEL_NAME)
print(f'height-time fits and the model sweep use: {REF_TRACK}  ({REF_MODEL_NAME})')


# Each fitter returns h, v and a as callables with ANALYTIC derivatives. Nothing is finite
# differenced: differencing an interpolant returns the slope of whichever segment the step lands
# in, which produces staircase speeds and spikes of order 1e6 m/s^2 in the acceleration.
def fit_polynomial(t, h, sig):
    """Weighted polynomial of degree POLY_DEG; v and a are its exact derivatives."""
    p = np.polyfit(t, h, POLY_DEG, w=(1 / sig if sig is not None else None))
    dp, ddp = np.polyder(p, 1), np.polyder(p, 2)
    return {'h': lambda tt: np.polyval(p, tt),
            'v': lambda tt: np.polyval(dp, tt),               # km/s
            'a': lambda tt: np.polyval(ddp, tt) * 1000}       # m/s^2


def gallagher_accel(t, ar, ad, tr, td):
    """Gallagher, Lawrence & Dennis (2003) reciprocal-sum acceleration profile [km/s^2]."""
    return 1 / (1 / (ar * np.exp(t / tr)) + 1 / (ad * np.exp(-t / td)))


def fit_gallagher(t, h, sig):
    """Gallagher et al. (2003): a(t) is the model, so v and h are its exact integrals and both
    derivatives come straight back out of the fitted parameters."""
    tgrid = np.linspace(t.min(), t.max(), 400)
    def model(tt, ar, ad, tr, td, h0, v0):
        acc = gallagher_accel(tgrid, ar, ad, tr, td)
        vv = v0 + cumulative_trapezoid(acc, tgrid, initial=0)
        hh = h0 + cumulative_trapezoid(vv, tgrid, initial=0)
        return np.interp(tt, tgrid, hh)
    v0g = (h[-1] - h[0]) / (t[-1] - t[0])
    p0 = [1e-3, 1e-3, 150, 150, h[0], v0g]
    lo = [1e-6, 1e-6, 20, 20, h[0] - 5e4, -2000]
    hi = [1e2, 1e2, 5e3, 5e3, h[0] + 5e4, 3000]
    popt, _ = curve_fit(model, t, h, p0=p0, sigma=sig, absolute_sigma=False,
                        bounds=(lo, hi), maxfev=200000)
    acc = gallagher_accel(tgrid, *popt[:4])
    vv = popt[5] + cumulative_trapezoid(acc, tgrid, initial=0)
    hh = popt[4] + cumulative_trapezoid(vv, tgrid, initial=0)
    return {'h': CubicSpline(tgrid, hh), 'v': CubicSpline(tgrid, vv),
            'a': lambda tt, _g=tgrid, _a=acc: np.interp(tt, _g, _a) * 1000}


def fit_byrne(t, h, sig):
    """Byrne et al. (2013): Savitzky-Golay. The SG filter returns the derivatives directly
    (deriv=1, 2), which is the method as published and is smooth. Differentiating a spline
    through the smoothed points instead makes the second derivative ring."""
    tu = np.linspace(t.min(), t.max(), max(len(t), 21))
    hu = np.interp(tu, t, h)
    dt = tu[1] - tu[0]
    n = len(tu)
    win = min(11, n if n % 2 == 1 else n - 1)
    if win <= POLY_DEG + 1:
        win = POLY_DEG + 3 if (POLY_DEG + 3) % 2 == 1 else POLY_DEG + 2
    hs = savgol_filter(hu, win, POLY_DEG)
    vs = savgol_filter(hu, win, POLY_DEG, deriv=1, delta=dt)
    ac = savgol_filter(hu, win, POLY_DEG, deriv=2, delta=dt)
    return {'h': CubicSpline(tu, hs), 'v': CubicSpline(tu, vs),
            'a': lambda tt, _c=CubicSpline(tu, ac): _c(tt) * 1000}


def hva(fit, t):
    """Height [km], speed [km/s] and acceleration [m/s^2] of a fitted track at t."""
    return fit['h'](t), fit['v'](t), fit['a'](t)


FIT_METHODS = {'Polynomial': fit_polynomial, 'Gallagher (2003)': fit_gallagher,
               'Byrne (2013)': fit_byrne}
FIT_COLOR = {'Polynomial': 'tab:blue', 'Gallagher (2003)': 'tab:green', 'Byrne (2013)': 'tab:red'}

_d = ALref[REF_TRACK]
_good = np.isfinite(_d['r_mean'])
t_fit = tg[_good].astype(float)
r_fit = _d['r_mean'][_good].astype(float)
se_r = _d['r_se'][_good].astype(float)
_pos = se_r[np.isfinite(se_r) & (se_r > 0)]
se_r = np.where(np.isfinite(se_r) & (se_r > 0), se_r, np.nanmedian(_pos) if _pos.size else 1e-3)
h_fit = r_fit * RS_KM
sig_h = se_r * RS_KM
t_dense = np.linspace(t_fit.min(), t_fit.max(), 400)

FIT_OUT = {}
rng = np.random.default_rng(1)
for name, fitter in FIT_METHODS.items():
    try:
        f0 = fitter(t_fit, h_fit, sig_h)
    except (RuntimeError, ValueError, TypeError) as ex:
        print(f'{name}: fit failed ({ex})')
        continue
    h0, v0, a0 = hva(f0, t_dense)
    boot = {'h': [], 'v': [], 'a': []}
    for _ in tqdm(range(N_BOOT), desc=name, leave=False):
        try:
            fb = fitter(t_fit, h_fit + rng.normal(0, sig_h), sig_h)
            hh, vv, aa = hva(fb, t_dense)
            boot['h'].append(hh)
            boot['v'].append(vv)
            boot['a'].append(aa)
        except (RuntimeError, ValueError):
            continue
    resid = f0['h'](t_fit) - h_fit
    # Free parameters differ by method: the polynomial has POLY_DEG+1, Gallagher has 6
    # (a_r, a_d, tau_r, tau_d, h0, v0), and Savitzky-Golay is a local smoother whose effective
    # count is about (points / window) x (POLY_DEG + 1). Using POLY_DEG+1 for all three, as this
    # did, understates Gallagher's dof and overstates Byrne's.
    _npar = {'Polynomial': POLY_DEG + 1, 'Gallagher (2003)': 6,
             'Byrne (2013)': max(int(np.ceil(len(t_fit) / 11)) * (POLY_DEG + 1), POLY_DEG + 1)}
    dof = max(len(t_fit) - _npar.get(name, POLY_DEG + 1), 1)
    chi2_red = float(np.nansum((resid / sig_h) ** 2) / dof)
    # chi2_red >> 1 means the fit misses the points by more than the quoted errors allow, so the
    # bootstrap band built from those errors is too narrow by roughly sqrt(chi2_red)
    scale = np.sqrt(max(chi2_red, 1)) if INFLATE_BY_CHI2 else 1
    FIT_OUT[name] = dict(
        h=h0 / RS_KM, v=v0, a=a0,
        h_sd=(np.nanstd(boot['h'], axis=0) / RS_KM if boot['h'] else np.zeros_like(t_dense)) * scale,
        v_sd=(np.nanstd(boot['v'], axis=0) if boot['v'] else np.zeros_like(t_dense)) * scale,
        a_sd=(np.nanstd(boot['a'], axis=0) if boot['a'] else np.zeros_like(t_dense)) * scale,
        chi2_red=chi2_red, scale=scale,
        rse_Rsun=np.sqrt(np.nansum(resid ** 2) / dof) / RS_KM)
    # Even with the right dof this is not a goodness-of-fit statistic here: the points being
    # fitted are an analytic lane polynomial pushed through the density inversion, not independent
    # measurements, so the residuals are small by construction and chi2_red comes out far below 1.
    # rse_Rsun is the number to compare between methods.
    print(f'{name:18s} chi2_red = {chi2_red:8.3f} (not a goodness of fit - see note), '
          f'rms residual = {np.sqrt(np.nansum(resid ** 2) / dof) / RS_KM:.4f} Rsun')
print(f'{len(FIT_OUT)} of {len(FIT_METHODS)} methods converged')

# The spread BETWEEN methods is the number worth quoting from this section, so it is computed
# rather than read off the figure. Also state how these compare with A.6, because the same lane
# gets an acceleration in both places and they are not identical.
_vm = [o['v'].mean() for o in FIT_OUT.values() if np.isfinite(o['v']).any()]
_am = [o['a'].mean() for o in FIT_OUT.values() if np.isfinite(o['a']).any()]
if len(_vm) > 1:
    print(f'\nspread between methods: mean speed {100 * (max(_vm) / min(_vm) - 1):.1f}% '
          f'({min(_vm):.0f}-{max(_vm):.0f} km/s), '
          f'mean acceleration {100 * (max(_am) / min(_am) - 1):.1f}% '
          f'({min(_am):.0f}-{max(_am):.0f} m/s^2)')
_a6, _a6e = grid_scalar(ALref[REF_TRACK], 'a')
_v6, _v6e = grid_scalar(ALref[REF_TRACK], 'v')
print(f'against A.6 for the same lane: v_sh {_v6:.0f} +/- {_v6e:.0f} km/s, '
      f'a {_a6:+.0f} +/- {_a6e:.0f} m/s^2')
print('  The speeds agree; the accelerations need not, and the difference is not an error in')
print('  either. A.6 fits the lane over KIN_N_DENSE samples of its OWN traced span and averages')
print('  the result over the Monte-Carlo draws; this section fits the shared-grid points that')
print('  fall inside the lane and bootstraps them. r(t) is not exactly a quadratic, so a fitted')
print('  curvature depends on where the samples sit - which is precisely why the spread between')
print('  the three methods above is quoted as the uncertainty on a, not any single method\'s')
print('  error bar. Quote A.6 for per-lane numbers; quote this section for method sensitivity.')

# --- can each method actually describe this track? ------------------------------------------
# Converging is not the same as being applicable. A model whose functional form cannot represent
# the data will still return parameters; it just fits badly, and quoting its kinematics would be
# wrong. Both checks below are about the shape of the model, not the quality of the data.
APPLICABLE = {name: True for name in FIT_OUT}
if FIT_OUT:
    _best = min(d['chi2_red'] for d in FIT_OUT.values())
    for name, d in FIT_OUT.items():
        if d['chi2_red'] > max(10 * _best, 10):
            APPLICABLE[name] = False
            print(f'\n  *** WARNING: {name} fits this track far worse than the best method '
                  f'(chi2_red {d["chi2_red"]:.1f} vs {_best:.2f}, rms residual '
                  f'{d["rse_Rsun"]:.4f} Rsun). Do not quote its kinematics.')
    _g, _p = FIT_OUT.get('Gallagher (2003)'), FIT_OUT.get('Polynomial')
    if _g is not None and np.nanmin(_g['a']) >= 0 and _p is not None and np.nanmin(_p['a']) < 0:
        APPLICABLE['Gallagher (2003)'] = False
        print('\n  *** WARNING: the Gallagher et al. (2003) reciprocal-sum profile is '
              'POSITIVE-DEFINITE by construction -')
        print('      a(t) = [ (a_r e^(t/tau_r))^-1 + (a_d e^(-t/tau_d))^-1 ]^-1 with a_r, a_d > 0 '
              'is strictly > 0.')
        print('      It describes the impulsive ACCELERATION phase of an eruption and cannot '
              'represent a decelerating')
        print('      shock. The polynomial wants deceleration on this track, so the Gallagher '
              'curve is inapplicable here.')

In [ ]:
fig = plt.figure(figsize=[16, 4.8])
panels = [('h', r'$r\,/\,R_\odot$', '(a) Height'),
          ('v', r'$v_{\rm sh}$ [km s$^{-1}$]', '(b) Speed'),
          ('a', r'$a$ [m s$^{-2}$]', '(c) Acceleration')]
for i, (key, ylab, ttl) in enumerate(panels, start=1):
    ax = fig.add_subplot(1, 3, i)
    if key == 'h':
        # quote what the traced points actually are, not the word "SEM"
        ax.errorbar(t_fit, r_fit, yerr=se_r, fmt='o', ms=4, color='0.4', capsize=2, alpha=0.8,
                    zorder=1, label=(rf'traced: {len(t_fit)} pts, '
                                     rf'{r_fit.min():.3f}$-${r_fit.max():.3f} $R_\odot$, '
                                     rf'$\sigma$ = {np.mean(se_r):.4f} $R_\odot$'))
    for name, d in FIT_OUT.items():
        c = FIT_COLOR[name]
        ok = APPLICABLE.get(name, True)
        y, sd = d[key], d[key + '_sd']
        if key == 'h':
            lbl = (rf'{name}: rms {d["rse_Rsun"]:.4f} $R_\odot$, '
                   rf'$\chi^2_\nu$ = {d["chi2_red"]:.2f}')
        elif key == 'v':
            lbl = (f'{name}: {np.nanmin(y):.0f}$-${np.nanmax(y):.0f}, '
                   f'mean {np.nanmean(y):.0f} km s$^{{-1}}$')
        else:
            lbl = (f'{name}: {np.nanmin(y):+.0f} to {np.nanmax(y):+.0f}, '
                   f'mean {np.nanmean(y):+.0f} m s$^{{-2}}$')
        if not ok:
            lbl += '  [INAPPLICABLE]'
        ax.plot(t_dense, y, ls=('-' if ok else '--'), color=c, lw=1.8, label=lbl)
        ax.fill_between(t_dense, y - sd, y + sd, color=c, alpha=0.18)
    if key == 'a':
        ax.axhline(0, color='0.6', lw=0.8)
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.set_ylabel(ylab)
    ax.set_title(ttl)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7.5)
_scaled = 'bands widened by sqrt(chi2) ' if INFLATE_BY_CHI2 else 'bands are the raw bootstrap 1$\sigma$ '
fig.suptitle(f'Height-time fit comparison, {REF_TRACK} track ({REF_MODEL_NAME})\n'
             + _scaled + f'over {N_BOOT} refits; dashed = the model cannot describe this track',
             y=1.06, fontsize=12)
fig.tight_layout()
save_fig(fig, 'typeii_kinematics_fit_comparison')
plt.show()

In [ ]:
# goodness of fit and the grid-averaged kinematics each method implies. chi2_red is the primary
# metric (about 1 means the fit is consistent with the height errors); rse is the residual
# standard error in Rsun, i.e. the typical deviation of the fit from the traced heights
fit_compare = pd.DataFrame([{'method': name, 'applicable': APPLICABLE.get(name, True),
                             'chi2_red': d['chi2_red'],
                             'band_scale': d['scale'], 'rse_Rsun': d['rse_Rsun'],
                             'v_mean_kms': np.nanmean(d['v']), 'v_min_kms': np.nanmin(d['v']),
                             'v_max_kms': np.nanmax(d['v']), 'a_mean_ms2': np.nanmean(d['a'])}
                            for name, d in FIT_OUT.items()])
fit_compare.to_csv(os.path.join(OUTDIR, 'height_time_fit_comparison.csv'), index=False)
fit_compare.round(3)

> **Figure 7 &mdash; Height-time fit comparison.** `typeii_kinematics_fit_comparison.png`
>
> The same height-time track fitted three ways &mdash; a weighted polynomial, the Gallagher
> et&nbsp;al. (2003) two-phase acceleration profile, and Byrne et&nbsp;al. (2013) Savitzky-Golay
> smoothing &mdash; with *(a)* height, *(b)* speed and *(c)* acceleration. Every legend entry
> carries the numbers: the traced points give their count, height range and $\sigma$; each method
> gives its rms residual and $\chi^2_\nu$ in (a), its speed range and mean in (b), its acceleration
> range and mean in (c). Shaded bands are the raw parametric-bootstrap $1\sigma$.
>
> *A dashed line marked `[INAPPLICABLE]`* means the model's functional form cannot describe this
> track. The Gallagher profile in particular is positive-definite by construction and cannot
> represent a decelerating shock; since most type IIs decelerate past the impulsive phase, expect
> to see it flagged.
>
> **Table 6 &mdash; Fit comparison.** `height_time_fit_comparison.csv` &mdash; per method:
> applicability, $\chi^2_\nu$, the band scaling (1 unless `INFLATE_BY_CHI2` is on), the rms
> residual, and the speed and acceleration each method implies.

## A.8 &middot; Sensitivity to the density model &times; fold

**What "sweep" means here.** The entire chain from §A.4 &mdash; frequency to density to height to
$v_{\rm sh}$, $v_A$ and $B$ &mdash; is recomputed from the same traced lanes for every one of the
5&nbsp;models&nbsp;&times;&nbsp;4&nbsp;folds = 20 density profiles. `model_grid_sweep.csv` holds one
row per profile. Nothing about the tracing changes; only the assumed $n_e(r)$ does. The spread down
each column is therefore exactly the systematic error the density model contributes.

The band-split $X$ and $M_A$ are **model independent** and are quoted once per band. The height,
shock speed, Alfv&eacute;n speed and $B$ all depend on the assumed $n_e(r)$, so the spread across
the 5&nbsp;models&nbsp;&times;&nbsp;4&nbsp;folds grid *is* the systematic uncertainty and should be
quoted alongside the statistical error bars. Combinations that place the source below the solar
surface for this event return NaN and drop out of the grid rather than being extrapolated.

Note that the fundamental/harmonic pair gives no extra leverage here: since $f_H/2$ and $f_F$ map
to the same plasma density, both bands land on the same height in any given model. What the pair
does buy is **time coverage** &mdash; the joint track spans the whole burst, so the sweep is run on
it.

In [ ]:
sweep_rows = []
for name, model in tqdm(MODEL_GRID.items(), desc='model x fold'):
    agg = aggregate_lanes(passes, tg, model)
    if REF_TRACK not in agg:
        continue
    row = {'model': name}
    for key, lab in [('r', 'r_Rsun'), ('v', 'v_kms'), ('a', 'a_ms2'), ('vA', 'vA_kms'),
                     ('B', 'B_G'), ('ne', 'ne_cm3')]:
        # No common mask here: this table reports the RANGE of each quantity across the model
        # grid, not a set of numbers that have to satisfy an identity with each other. r and v
        # should therefore span the whole traced lane, while v_A and B are confined to the
        # band-split window by their own definition.
        row[lab], row[lab + '_se'] = grid_scalar(agg[REF_TRACK], key)
    # The height on the band-split window, carried alongside, because r_Rsun above is NOT the
    # height at which B_G was measured and the two must not be plotted as a pair. B exists only
    # where both branches of the band do, and on this event that sub-interval sits 0.18 Rsun
    # higher than the lane average. Putting B at the lane-average height moved it along a
    # (r-1)^-1.5 reference curve far enough to change the quoted offset from Dulk & McLean from
    # 2.1x to 1.5x - an error in the one comparison that figure exists to make.
    _cms = common_mask(agg[REF_TRACK])
    row['r_at_B_Rsun'], row['r_at_B_Rsun_se'] = grid_scalar(agg[REF_TRACK], 'r', mask=_cms)
    sweep_rows.append(row)

sweep = pd.DataFrame(sweep_rows)
sweep['base'] = [m.rsplit(' x', 1)[0] for m in sweep['model']]
sweep['fold'] = [int(m.rsplit(' x', 1)[1]) for m in sweep['model']]
sweep.to_csv(os.path.join(OUTDIR, 'model_grid_sweep.csv'), index=False)
print(f'the full chain was recomputed on the {REF_TRACK} track for all {len(MODEL_GRID)} '
      'density profiles (5 models x 4 folds); only n_e(r) changed, the tracing did not')
print('the spread down each column IS the systematic error; compare it with the statistical '
      'errors in A.6 before quoting anything')
sweep[['model', 'r_Rsun', 'v_kms', 'a_ms2', 'vA_kms', 'B_G']].round(3)

> **Table 7 &mdash; Density-model sweep.** `model_grid_sweep.csv`
>
> The entire chain &mdash; frequency to density to height to $v_{\rm sh}$, $a$, $v_A$ and $B$
> &mdash; recomputed from the *same traced lanes* for each of the 5&nbsp;models&nbsp;$\times$&nbsp;4
> folds. Only the assumed $n_e(r)$ changes between rows; nothing about the tracing does.
>
> *The spread down each column is the systematic error the density model contributes*, and for this
> class of measurement it dominates the statistical error by one to two orders of magnitude. Quote
> the reference-model value with this range beside it.

In [ ]:
bases = list(BASE_MODELS.keys())
xpos = np.arange(len(bases))
fold_off = {f: (f - 2.5) * 0.16 for f in FOLDS}
fold_col = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:green', 4: 'tab:red'}

# The window is part of the axis label. r and v are lane averages over everything traced; v_A and
# B exist only where both branches of the band do, which here is a 214 s sub-interval of a 596 s
# lane. A reader comparing panel (a) with panel (d) is otherwise comparing two different heights.
panels = [('r_Rsun', 'height $r$ [$R_\\odot$]\n(full traced span)'),
          ('v_kms', 'shock speed $v_{\\rm sh}$ [km s$^{-1}$]\n(full traced span)'),
          ('vA_kms', 'Alfv$\\acute{\\rm e}$n speed $v_A$ [km s$^{-1}$]\n(band-split window)'),
          ('B_G', '$B$ [G]\n(band-split window)')]

fig = plt.figure(figsize=[14, 9])
for i, (col, ylab) in enumerate(panels, start=1):
    ax = fig.add_subplot(2, 2, i)
    for f in FOLDS:
        sub = sweep[sweep['fold'] == f].set_index('base').reindex(bases)
        ax.errorbar(xpos + fold_off[f], sub[col], yerr=sub[col + '_se'], fmt='o',
                    color=fold_col[f], ms=6, capsize=3, mec='k', mew=0.4,
                    label=(f'fold {f}' if i == 1 else None))
    ax.set_xticks(xpos)
    ax.set_xticklabels(bases, fontsize=8)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3, axis='y')
    if i == 1:
        ax.legend(title='fold', fontsize=8, ncol=2)

fig.suptitle(f'Shock characteristics vs density model x fold ({REF_TRACK} track)',
             y=1.01, fontsize=13)
fig.tight_layout()
save_fig(fig, 'characteristics_vs_model_fold')
plt.show()

> **Figure 8 &mdash; Characteristics against density model and fold.**
> `characteristics_vs_model_fold.png`
>
> Height, shock speed, Alfv&eacute;n speed and magnetic field for every model&times;fold
> combination, colour-coded by fold. Error bars are the statistical (Monte-Carlo) errors of each
> combination.
>
> *Read the vertical scatter, not the individual points.* The points are not competing
> measurements; they are the same measurement under twenty different assumptions about the corona,
> and their spread is the honest uncertainty on any height-dependent quantity.

## A.9 &middot; Coronal magnetic field against empirical $B(r)$ laws

The band-split estimate $B=v_A\sqrt{\mu_0\rho}$ gives one point per model&times;fold, placed at the
height that model assigns. Comparing the cloud against published radial profiles shows whether the
type II is asking for a field strength that a normal corona can supply at that height.

Read the three reference curves for what each one is, because they do not carry equal weight and
they do not agree with each other:

- **Dulk &amp; McLean (1978)**, $B=0.5\,(r-1)^{-1.5}$&nbsp;G, valid 1.02&ndash;10&nbsp;$R_\odot$, so
  this event sits inside its range. An empirical envelope assembled from many instruments.
- **Mann et al. (2023)**, their Eq.&nbsp;8, $B_r = 6r^{-3}+1.18r^{-2}$&nbsp;G &mdash; a dipole plus
  radial-component model, and the same paper that supplies one of the density models used here.
- **Gopalswamy &amp; Yashiro (2011)**, $B = 0.409\,r^{-1.30}$&nbsp;G, from the standoff distance
  between a CME-driven shock and its flux rope in white-light coronagraph images &mdash; an
  independent technique, but calibrated over 6&ndash;23&nbsp;$R_\odot$ and extrapolated several
  solar radii below its range here. An indication only.

At the heights this burst occupies the first two are themselves about a factor of two apart, so
"consistent with the published profiles" is not a statement worth making: the figure and &sect;A.10
report which one the measurement lands on, and by how much.

In [ ]:
def B_dulk_mclean(r):
    """Dulk & McLean (1978); valid 1.02 <= r <= 10 Rsun."""
    r = np.asarray(r, float)
    return 0.5 * (r - 1) ** -1.5

def B_gopalswamy_yashiro(r):
    """Gopalswamy & Yashiro (2011), ApJL 736, L17: B = 0.409 r^-1.30 [G].

    Their headline profile, and an INDEPENDENT method from the one used here - not band splitting.
    They measured the standoff distance between a CME-driven shock and the flux-rope leading edge
    in white-light coronagraph images, together with the rope's radius of curvature, over
    6-23 Rsun, and recovered B falling from 48 to 8 mG across that range. Because the technique
    shares none of this notebook's assumptions, agreement or disagreement with it carries more
    weight than with a profile fitted to the same kind of data - but only inside 6-23 Rsun.

    They also quote B = 0.377 r^-1.25 from the same measurements reduced with the Saito et al.
    (1977) density model instead of Leblanc et al. (1998); the two differ by the density model,
    not by any assumption about gamma. 0.409 r^-1.30 is the one they compare with other
    techniques, so it is the one plotted.

    At 1-3 Rsun this is an extrapolation of several solar radii below its calibration range, so a
    large offset here is expected and is not evidence against a measurement."""
    r = np.asarray(r, float)
    return 0.409 * r ** -1.30

def B_mann2023(r):
    """Mann et al. (2023), A&A 679, A64, Eq. 8, radial field [G]."""
    r = np.asarray(r, float)
    return 6 * r ** -3 + 1.18 * r ** -2


rr = np.linspace(1.05, 3, 300)
fig = plt.figure(figsize=[9, 6.5])
ax = fig.add_subplot(111)
ax.plot(rr, B_dulk_mclean(rr), 'k-',
        label=r'Dulk & McLean (1978), $0.5\,(r-1)^{-1.5}$')
ax.plot(rr, B_gopalswamy_yashiro(rr), 'k--',
        label=r'Gopalswamy & Yashiro (2011), $0.409\,r^{-1.30}$ (standoff distance,'
              '\n' r'    calibrated 6$-$23 $R_\odot$, extrapolated here)')
ax.plot(rr, B_mann2023(rr), 'k:',
        label=r'Mann et al. (2023) Eq. 8, $6r^{-3}+1.18r^{-2}$')

mk = {'Newkirk': 'o', 'Saito': 's', 'Leblanc': '^', 'Baumbach-Allen': 'D', 'Mann 2023': 'v'}
# x is r_at_B_Rsun, the height on the band-split window, NOT the lane-average r_Rsun of the A.8
# table. B only exists on that window, and the reference curves are steep functions of r, so the
# pair has to share one interval or the comparison is against the wrong part of the curve.
for _, row in sweep.iterrows():
    if np.isfinite(row['r_at_B_Rsun']) and np.isfinite(row['B_G']):
        ax.errorbar(row['r_at_B_Rsun'], row['B_G'], yerr=row['B_G_se'],
                    xerr=row['r_at_B_Rsun_se'],
                    fmt=mk[row['base']], color=fold_col[row['fold']], ms=8, capsize=2,
                    mec='k', mew=0.5, alpha=0.9)
if not np.isfinite(sweep['B_G']).any():
    ax.text(0.5, 0.5, 'no band-split estimate to plot: $B$ is NaN for every model.\n'
            '$B = v_A\\sqrt{\\mu_0\\rho}$ needs $M_A$, which is only defined for '
            '$1 \\leq X < 4$.\nCheck the upstream/downstream ordering printed in A.4.',
            transform=ax.transAxes, ha='center', va='center', fontsize=10, color='firebrick')
handles = [plt.Line2D([], [], marker=mk[b], color='0.4', ls='', mec='k', label=b) for b in bases]
handles += [plt.Line2D([], [], marker='o', color=fold_col[f], ls='', label=f'fold {f}') for f in FOLDS]
# Both legends stack in the top-right corner. The lower-left placement sat on top of the
# Gopalswamy & Yashiro curve, which is the one the reader most needs to see is extrapolated.
leg1 = ax.legend(loc='upper right', bbox_to_anchor=(1, 1), borderaxespad=0.4, fontsize=9)
ax.add_artist(leg1)
_h1 = leg1.get_window_extent().transformed(ax.transAxes.inverted()).height
ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1, 1 - _h1 - 0.03),
          borderaxespad=0.4, fontsize=8, ncol=2, title='band-split estimate')
ax.set_yscale('log')
ax.set_xlabel(r'$r\,/\,R_\odot$  (on the band-split window, where $B$ is defined)')
ax.set_ylabel(r'$B$ [G]')
ax.set_title('Coronal magnetic field: band-split estimate vs empirical laws')
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
save_fig(fig, 'Bfield_comparison')
plt.show()

> **Figure 9 &mdash; Coronal magnetic field against empirical laws.** `Bfield_comparison.png`
>
> The band-split estimate $B=v_A\sqrt{\mu_0\rho}$, one point per model&times;fold at the height that
> model assigns (marker = model, colour = fold), against three published radial profiles:
> Dulk &amp; McLean (1978), $0.5\,(r-1)^{-1.5}$, whose 1.02&ndash;10&nbsp;$R_\odot$ range covers this
> event; Mann et al. (2023) Eq.&nbsp;8, $6r^{-3}+1.18r^{-2}$; and Gopalswamy &amp; Yashiro (2011),
> $0.409\,r^{-1.30}$, derived from the standoff distance between a CME-driven shock and its flux
> rope in white-light coronagraph images &mdash; an independent technique, but calibrated over
> 6&ndash;23&nbsp;$R_\odot$ and extrapolated several solar radii below its range here, so indicative
> only.
>
> *What it tests:* whether the type II is asking for a field strength a normal corona can supply at
> that height. Points lying far above every law would suggest the density jump, and hence $M_A$,
> has been overestimated. Note that the first two curves are themselves about a factor of two apart
> over 1&ndash;3&nbsp;$R_\odot$, so this figure can show which profile the measurement follows but
> cannot be passed by "agreeing with the published laws" in general.

## A.10 &middot; Characteristics table, LaTeX export and summary text

In [ ]:
rows = []
for b in BANDS_TRACED:
    s = scalars[b]
    tag = f'{BAND_NAME[b].lower()}, s={HARM[b]}'
    for key, lab, unit in [('drift_MHz_s', 'drift rate', 'MHz/s'),
                           ('rel_drift_s', 'relative drift (1/f)(df/dt)', '1/s'),
                           ('X', 'density jump X', '-'),
                           ('M_A', 'Alfven Mach number M_A', '-'),
                           ('rel_bandwidth', 'relative band split (f_U-f_L)/f_L', '-')]:
        rows.append({'track': b, 'quantity': f'{lab} ({tag})', 'value': s[key][0],
                     'error': s[key][1], 'unit': unit, 'note': 'model-independent'})

for key in TRACKS:
    d = ALref[key]
    if np.isfinite(d['r_mean']).sum() <= 2:
        continue
    tag = key
    fm = d['f_mean'][np.isfinite(d['f_mean'])]
    if fm.size:
        rows.append({'track': tag, 'quantity': f'frequency range ({tag})', 'value': np.nanmin(fm),
                     'error': np.nanmax(fm), 'unit': 'MHz (min, max)', 'note': 'observed'})
    # Two different averaging windows are unavoidable here, so both are stated rather than one
    # being quietly imposed. r, v, a and n_e exist over the lane's WHOLE traced span and that is
    # the measurement worth quoting. X, M_A, v_A and B need both branches of the band split, so
    # they exist only where the branches overlap in time. Averaging the first group over the whole
    # span and the second over the overlap - and then printing them in one row without saying so -
    # is what makes a row fail its own identity: v_A comes out unequal to v_sh/M_A and B unequal to
    # v_A sqrt(mu0 rho) at the n_e beside it. Forcing everything onto the overlap instead would fix
    # the identity but throw away most of the traced lane, so the speed and height would no longer
    # describe the burst. Both windows are therefore reported, each labelled, plus the speed on the
    # split window so the identity can be checked directly.
    _cm = common_mask(d)
    _fmt_win = lambda msk: (
        f'{(t0 + pd.Timedelta(seconds=float(tg[msk].min()))).strftime("%H:%M:%S")}-'
        f'{(t0 + pd.Timedelta(seconds=float(tg[msk].max()))).strftime("%H:%M:%S")} UT')
    _win_split = _fmt_win(_cm) if (_cm is not None and _cm.any()) else ''
    for k, lab, unit in [('r', 'height r', 'Rsun'), ('v', 'shock speed', 'km/s'),
                         ('a', 'acceleration', 'm/s^2'),
                         ('ne', 'upstream density n_e', 'cm^-3')]:
        m, se = grid_scalar(d, k)
        _msk = np.isfinite(d[k + '_mean'])
        rows.append({'track': tag, 'quantity': f'{lab} ({tag})', 'value': m, 'error': se,
                     'unit': unit, 'note': f'{REF_MODEL_NAME}, full traced span '
                                           f'{_fmt_win(_msk) if _msk.any() else ""}'})
    # Height and density on the split window as well as the full span. Only these are comparable
    # BETWEEN the two branches of a band: on their own spans the branches cover different stretches
    # of a drifting burst, so the full-span n_e of the upper branch comes out BELOW the lower one
    # and the heights come out the wrong way round, contradicting X > 1. On the common window the
    # ordering is the physical one.
    for k, lab, unit in [('r', 'height r on the band-split window', 'Rsun'),
                         ('v', 'shock speed on the band-split window', 'km/s'),
                         ('ne', 'n_e on the band-split window', 'cm^-3')]:
        m, se = grid_scalar(d, k, mask=_cm)
        rows.append({'track': tag, 'quantity': f'{lab} ({tag})', 'value': m, 'error': se,
                     'unit': unit,
                     'note': f'{REF_MODEL_NAME}, band-split window {_win_split}; only these are '
                             'comparable between the two branches of a band'})
    # v_A and B belong to the UPSTREAM branch only. v_A = v_sh/M_A from Rankine-Hugoniot is by
    # construction the upstream Alfven speed, so pairing it with the downstream density gives
    # B_1 sqrt(X) - neither the upstream field B_1 nor the downstream field X B_1. Emitting it for
    # the downstream lane produced a number that is the field of nothing, and an audit row that
    # compared the two branches' "B" and passed only because two errors partly cancelled.
    if LANE_ROLE.get(tag, '').startswith('upstream'):
        for k, lab, unit in [('vA', 'Alfven speed', 'km/s'), ('B', 'magnetic field B', 'G')]:
            m, se = grid_scalar(d, k, mask=_cm)
            rows.append({'track': tag, 'quantity': f'{lab} ({tag})', 'value': m, 'error': se,
                         'unit': unit,
                         'note': f'{REF_MODEL_NAME}, band-split window {_win_split}, UPSTREAM '
                                 'branch; satisfies v_A = v_sh/M_A and B = v_A sqrt(mu0 rho) '
                                 'with the two rows above'})
    else:
        rows.append({'track': tag, 'quantity': f'magnetic field B ({tag})', 'value': np.nan,
                     'error': np.nan, 'unit': 'G',
                     'note': 'not defined for a downstream branch: v_A from Rankine-Hugoniot is '
                             'the UPSTREAM Alfven speed, so pairing it with this branch\'s '
                             'density gives B_1 sqrt(X), which is neither B_1 nor B_2'})
    if REPORT_EXCITER_ENERGY:
        v_mean, _ = grid_scalar(d, 'v')
        rows.append({'track': tag, 'quantity': f'bulk exciter energy ({tag})',
                     'value': electron_energy_from_speed(v_mean) * 1e3, 'error': np.nan,
                     'unit': 'eV', 'note': f'{REF_MODEL_NAME}, E = (gamma-1) m_e c^2 at v_sh; '
                                           'NOT the energy of the emitting electrons'})

# the model x fold spread, i.e. the systematic error on every height-dependent quantity
for col, lab, unit in [('r_Rsun', 'height r', 'Rsun'), ('v_kms', 'shock speed', 'km/s'),
                       ('a_ms2', 'acceleration', 'm/s^2'),
                       ('vA_kms', 'Alfven speed', 'km/s'), ('B_G', 'magnetic field B', 'G')]:
    vals = sweep[col].to_numpy(float)
    vals = vals[np.isfinite(vals)]
    if vals.size:
        rows.append({'track': REF_TRACK,
                     'quantity': f'{lab} (model x fold range)', 'value': np.nanmin(vals),
                     'error': np.nanmax(vals), 'unit': f'{unit} (min, max)',
                     'note': 'systematic across the 20-model grid'})

char_table = pd.DataFrame(rows)
char_table.to_csv(os.path.join(OUTDIR, 'typeii_characteristics.csv'), index=False)


def df_to_latex(df, caption, label, float_fmt='{:.4g}'):
    """Minimal LaTeX table writer. Written by hand rather than via DataFrame.to_latex so it needs
    no jinja2, escapes the characters that appear in these tables, and emits a float-ready
    table environment."""
    def esc(x):
        if isinstance(x, float):
            return '' if not np.isfinite(x) else float_fmt.format(x)
        s = str(x)
        for a, b in [('\\', r'\textbackslash '), ('_', r'\_'), ('%', r'\%'), ('&', r'\&'),
                     ('#', r'\#'), ('^', r'\^{}'), ('~', r'\~{}')]:
            s = s.replace(a, b)
        return s
    cols = list(df.columns)
    out = ['\\begin{table}[htbp]', '\\centering',
           f'\\caption{{{caption}}}', f'\\label{{{label}}}',
           '\\begin{tabular}{' + 'l' * len(cols) + '}', '\\hline',
           ' & '.join(esc(c) for c in cols) + ' \\\\', '\\hline']
    for _, r in df.iterrows():
        out.append(' & '.join(esc(v) for v in r.tolist()) + ' \\\\')
    out += ['\\hline', '\\end{tabular}', '\\end{table}', '']
    return '\n'.join(out)


with open(os.path.join(OUTDIR, 'typeii_characteristics.tex'), 'w') as fh:
    fh.write(df_to_latex(char_table,
                         f'Characteristics of the type II radio burst observed by NenuFAR on '
                         f'{EVENT_DATE}, from the band-split fundamental and harmonic lanes. '
                         f'Model-independent quantities come from the band splits alone; the rest '
                         f'assume the {REF_MODEL_NAME} density model, with the range across the '
                         f'full model grid given separately.',
                         'tab:typeii_nenufar'))

picks = {'traces': tracer.traces, 'passes': passes, 'polarisation': pol_table,
         'fh_tests': fh_test, 'lane_order': LANE_ORDER, 'lane_role': LANE_ROLE,
         'config': {'N_REPS': N_REPS, 'TYPEII_WINDOW': TYPEII_WINDOW,
                    'TYPEII_FLIM': TYPEII_FLIM, 'HARM': HARM, 'POLY_DEG': POLY_DEG,
                    'FIT_IN_LOGF': FIT_IN_LOGF, 'REF_MODEL_NAME': REF_MODEL_NAME,
                    'REF_TRACK': REF_TRACK}}
with open(os.path.join(OUTDIR, 'typeii_picks.pkl'), 'wb') as fh:
    pickle.dump(picks, fh)

# anything in OUTDIR older than this run is left over from a previous version of the analysis.
# Mixing the two is exactly the kind of mistake that is impossible to spot later.
stale = sorted(f for f in os.listdir(OUTDIR)
               if not f.startswith('.')
               and os.path.getmtime(os.path.join(OUTDIR, f)) < RUN_START)
print('written to', OUTDIR)
if stale:
    print(f'\n  *** WARNING: {len(stale)} file(s) in {OUTDIR} were NOT written by this run and '
          'are left over from an earlier one:')
    for f in stale:
        print(f'        {f}')
    print('      delete them, or move this run to a fresh OUTDIR, before using anything from '
          'that folder.')
char_table.round(4)

> **Table 8 &mdash; Burst characteristics.** `typeii_characteristics.csv`, `.tex`
>
> The summary table. Model-independent quantities first (drift rate, relative drift, density
> jump, Alfv&eacute;n Mach number, relative bandwidth, per band), then the model-dependent ones per
> lane on the reference model (frequency range, height, speed, acceleration, Alfv&eacute;n speed,
> upstream density, magnetic field), and finally the **model&times;fold range** of each
> height-dependent quantity as its systematic error. The `.tex` file is the same table as a
> `table` environment ready to `\input`.
>
> A warning is printed here listing any file in the output folder that this run did **not** write
> &mdash; leftovers from an earlier version of the analysis are otherwise impossible to spot later.

In [ ]:
# ---- summary text, filled with the numbers actually measured above ----
def _dp(e, min_dp=1, max_dp=5):
    """Decimal places that show the error to two significant figures."""
    if e is None or not np.isfinite(e) or e <= 0:
        return min_dp
    return int(np.clip(-np.floor(np.log10(abs(e))) + 1, min_dp, max_dp))


def _fmt(v, e, min_dp=1, sep=' +/- '):
    """Value +/- error with the precision set by the error, not by a fixed format."""
    if not np.isfinite(v):
        return 'n/a'
    nd = _dp(e, min_dp)
    if e is None or not np.isfinite(e) or e <= 0:
        return f'{v:.{nd}f}'
    return f'{v:.{nd}f}{sep}{e:.{nd}f}'


def _sci(v, nd=1):
    """Scientific notation as LaTeX inline maths."""
    if not np.isfinite(v) or v == 0:
        return 'n/a'
    ex = int(np.floor(np.log10(abs(v))))
    return f'${v / 10 ** ex:.{nd}f} \\times 10^{{{ex}}}$'


def _tex(v, e, min_dp=1):
    """The same as _fmt, wrapped as inline maths."""
    if not np.isfinite(v):
        return 'n/a'
    nd = _dp(e, min_dp)
    if e is None or not np.isfinite(e) or e <= 0:
        return f'${v:.{nd}f}$'
    return f'${v:.{nd}f} \\pm {e:.{nd}f}$'


def _B_context(r_, B_):
    """Place the measured B against the published radial profiles by evaluating them, not by
    asserting agreement.

    Writing "in line with Dulk & McLean and Mann et al." costs nothing to type and is checkable in
    seconds by anyone reading it. At the height measured here the two profiles are themselves a
    factor of two apart, so one number cannot be in line with both, and which one it lands on is a
    result. This sentence therefore reports the ratios and lets them say what they say."""
    if not (np.isfinite(r_) and np.isfinite(B_)) or B_ <= 0 or r_ <= 1:
        return ''
    refs = [('Dulk \\& McLean (1978)', 'dulk1978', float(np.atleast_1d(B_dulk_mclean(r_))[0])),
            ('Mann et al. (2023)', 'mann2023', float(np.atleast_1d(B_mann2023(r_))[0]))]
    refs = [x for x in refs if np.isfinite(x[2]) and x[2] > 0]
    if not refs:
        return ''
    listed = ' and '.join(f'{v:.2f}~G \\citep{{{k}}}' for _, k, v in refs)
    ranked = sorted(((abs(np.log(B_ / v)), n, v) for n, _, v in refs))
    clauses = []
    for lg, n, v in ranked:
        if lg < 0.18:                                    # within ~20%, i.e. the model spread itself
            clauses.append(f'agrees with {n} to {100 * abs(B_ / v - 1):.0f}\\%')
        else:
            clauses.append(f'sits a factor of {max(B_ / v, v / B_):.1f} '
                           f'{"above" if B_ > v else "below"} {n}')
    out = [f'At that height the published radial profiles give {listed}, so the measurement '
           + ' but '.join(clauses) + '.']
    if len(refs) > 1:
        sp = max(x[2] for x in refs) / min(x[2] for x in refs)
        out.append(f'Those two profiles are themselves a factor of {sp:.1f} apart here, so '
                   f'consistency with either one on its own is a weak constraint.')
    return ' '.join(out)


d = ALref[REF_TRACK]
# The paragraph quotes two groups of numbers and has to keep them apart. r, v_sh and a describe
# the burst and belong on the whole traced lane. v_A and B exist only where both branches of the
# band are present, so they come with that window's own r and n_e - which for this lane is its
# last 214 s, where an accelerating shock is 143 km/s faster than its lane average. X and M_A come
# from scalars[], the same object the table above prints, so the paragraph cannot quote a density
# jump that disagrees with its own table.
_w0 = lane_windows(d, REF_TRACK, tg, t0)
r0, r0e = _w0['r_span'], _w0['r_span_e']
v0, v0e = _w0['v_span'], _w0['v_span_e']
a0, a0e = _w0['a_span'], _w0['a_span_e']
rS, rSe = _w0['r_split'], _w0['r_split_e']
vA0, vA0e = _w0['vA_split'], _w0['vA_split_e']
B0, B0e = _w0['B_split'], _w0['B_split_e']
ne0 = _w0['ne_split']
_bref = LANE_BAND[REF_TRACK]
X0, X0e = scalars[_bref]['X']
MA0, MA0e = scalars[_bref]['M_A']
ta = (t0 + pd.Timedelta(seconds=float(np.nanmin(tg)))).strftime('%H:%M')
tb = (t0 + pd.Timedelta(seconds=float(np.nanmax(tg)))).strftime('%H:%M')

lines = [f'Type II radio burst, {EVENT_DATE}, NenuFAR', '=' * 78, '',
         f'observed {ta}-{tb} UT; one shock seen in both harmonics',
         f'every lane is analysed separately'
         + (f'; {JOINT} is the combined track' if MAKE_JOINT else ''), '']
for b in BANDS_TRACED:
    s_ = scalars[b]
    lines += [f'{BAND_NAME[b]} band (s = {HARM[b]})', '-' * 78,
              f'  drift rate                   {_fmt(*s_["drift_MHz_s"], min_dp=4)} MHz/s',
              f'  relative drift (1/f)(df/dt)  {_fmt(*s_["rel_drift_s"], min_dp=5)} 1/s',
              f'  relative band split          {_fmt(*s_["rel_bandwidth"], min_dp=3)}',
              f'  density jump X               {_fmt(*s_["X"], min_dp=2)}   (model-independent)',
              f'  Alfven Mach number M_A       {_fmt(*s_["M_A"], min_dp=2)}   (model-independent)']
    for lab in LANE_ORDER[b]:
        d = ALref.get(lab)
        if d is None:
            continue
        ff = _fits[lab]
        wa = (t0 + pd.Timedelta(seconds=float(ff['tmin']))).strftime('%H:%M:%S')
        wb_ = (t0 + pd.Timedelta(seconds=float(ff['tmax']))).strftime('%H:%M:%S')
        # The frequency extent has to come from the TRACED POINTS, not from d['f_mean'], which is
        # the fit sampled on the shared N_GRID grid. That grid spans the whole burst, so its
        # spacing (~30 s here) is coarse next to a steeply drifting lane: the first grid point
        # inside the lane already sits several MHz below where the trace actually starts. It cost
        # H lane 1 6.7 MHz off the top of its reported range and H lane 2 9.1 MHz.
        _pts = passes[0].get(lab)
        fm = (np.asarray(_pts['f'], float) if _pts and _pts['f']
              else d['f_mean'][np.isfinite(d['f_mean'])])
        pol = pol_table[pol_table['lane'] == lab]
        # Same helper as A.4 and the A.9 audit. Every row states the interval it was averaged over,
        # because r and v_sh on the full traced span and v_A and B on the band-split window are
        # different measurements: for F lane 1 the split window is the last 214 s of a 596 s lane
        # that is accelerating, and the mean speed there is 771 km/s against 628 over the whole
        # lane. Printing one of those under a heading naming the other interval is how a 23%
        # discrepancy gets into a manuscript.
        w = lane_windows(d, lab, tg, t0)
        lines += ['', f'  {lab}  [{LANE_ROLE[lab]}]',
                  f'    traced                     {wa}-{wb_} UT over '
                  f'{np.nanmin(fm):.1f}-{np.nanmax(fm):.1f} MHz',
                  f'    -- over the full traced span ({w["n_span"]} grid points) --',
                  f'    height r                   {_fmt(w["r_span"], w["r_span_e"], 2)} Rsun',
                  f'    shock speed v_sh           {_fmt(w["v_span"], w["v_span_e"], 0)} km/s',
                  f'    acceleration a             {_fmt(w["a_span"], w["a_span_e"], 1)} m/s^2',
                  f'    signed mean V/I            '
                  f'{pol["V_over_I_mean"].iloc[0]:+.4f} +/- {pol["V_over_I_sd"].iloc[0]:.4f} (sd)'
                  if len(pol) else '']
        if not w['has_split']:
            lines += ['    -- band-split window: the two branches of this band never overlap, '
                      'so X, M_A, v_A and B are undefined --']
            continue
        lines += [f'    -- on the band-split window {w["split_window"]} '
                  f'({w["n_split"]} of {w["n_span"]} grid points) --',
                  f'    height r                   {_fmt(w["r_split"], w["r_split_e"], 2)} Rsun',
                  f'    shock speed v_sh           {_fmt(w["v_split"], w["v_split_e"], 0)} km/s',
                  f'    electron density n_e       {w["ne_split"]:.3e} cm^-3']
        if w['upstream']:
            lines += [f'    Alfven speed v_A           '
                      f'{_fmt(w["vA_split"], w["vA_split_e"], 0)} km/s',
                      f'    magnetic field B           {_fmt(w["B_split"], w["B_split_e"], 2)} G']
        else:
            lines += ['    Alfven speed v_A           not defined for a downstream branch',
                      '    magnetic field B           not defined for a downstream branch: v_A '
                      'from Rankine-Hugoniot is the',
                      '                               UPSTREAM Alfven speed, so pairing it with '
                      'this branch\'s density']
            lines += ['                               gives B_1 sqrt(X), neither B_1 nor B_2']
    lines += ['']

lines += [f'{REF_TRACK} on the model x fold grid ({REF_MODEL_NAME} is the reference)', '-' * 78,
          f'  height r      {np.nanmin(sweep["r_Rsun"]):.2f}-{np.nanmax(sweep["r_Rsun"]):.2f} Rsun',
          f'  shock speed   {np.nanmin(sweep["v_kms"]):.0f}-{np.nanmax(sweep["v_kms"]):.0f} km/s',
          f'  Alfven speed  {np.nanmin(sweep["vA_kms"]):.0f}-{np.nanmax(sweep["vA_kms"]):.0f} km/s',
          f'  magnetic field {np.nanmin(sweep["B_G"]):.3f}-{np.nanmax(sweep["B_G"]):.3f} G', '']

if len(fh_test):
    lines += ['Fundamental / harmonic consistency', '-' * 66]
    for _, row in fh_test.iterrows():
        nd = 5 if abs(row['value']) < 0.01 else 3        # the drifts are ~1e-3 per second
        exp = '' if not np.isfinite(row['expected']) else f'   (expected {row["expected"]:.{nd}f})'
        lines.append(f'  {row["test"]:34s} {_fmt(row["value"], row["error"], nd)}{exp}')

summary = '\n'.join(lines)
print(summary)

sF = scalars['F']
sH = scalars.get('H', sF)
_wF = _fits[SPLIT_PAIR['F'][0]]
_wH = _fits[SPLIT_PAIR['H'][0]] if 'H' in SPLIT_PAIR else _wF
para = (
    f"A type II radio burst was observed by NenuFAR on "
    f"{pd.Timestamp(EVENT_DATE).strftime('%d %B %Y')} between {ta} and {tb}~UT, in both the "
    f"fundamental and the harmonic of plasma emission. The fundamental drifted through the "
    f"observing band from "
    f"{(t0 + pd.Timedelta(seconds=float(_wF['tmin']))).strftime('%H:%M')}~UT and the harmonic, at "
    f"twice its frequency, entered the top of the band from "
    f"{(t0 + pd.Timedelta(seconds=float(_wH['tmin']))).strftime('%H:%M')}~UT and followed the same "
    f"shock to the end of the event; over the interval in which both were visible their frequency "
    f"ratio is "
    f"{_tex(float(fh_test.iloc[0]['value']), float(fh_test.iloc[0]['error']), 2) if len(fh_test) else 'n/a'}, "
    f"confirming the identification. The relative drift rate is "
    f"{_tex(*sF['rel_drift_s'], min_dp=4)}~s$^{{-1}}$. Both bands are separately band-split, "
    f"giving relative bandwidths of {_tex(*sF['rel_bandwidth'], min_dp=2)} (F) and "
    f"{_tex(*sH['rel_bandwidth'], min_dp=2)} (H). Interpreting the split as emission from the "
    f"upstream and downstream sides of the shock front gives, for the "
    f"{BAND_NAME[_bref].lower()} band, a density jump of "
    f"$X = {_tex(X0, X0e, 2)[1:-1]}$ and, through the Rankine-Hugoniot relation for a perpendicular "
    f"shock at $\\gamma=5/3$ and $\\beta=0$, an Alfv\\'en Mach number of "
    f"$M_A = {_tex(MA0, MA0e, 2)[1:-1]}$; neither depends on the assumed coronal density model. "
    f"Each band was then converted with its own harmonic number and each lane analysed separately"
    + (f", and the two were also combined into the single track {JOINT}" if MAKE_JOINT else
       "; the two bands were not merged into a joint track") + f". Adopting a "
    f"{REF_MODEL_NAME.replace(' x', ', fold-')} electron-density profile places {REF_TRACK} at "
    f"$r = {_tex(r0, r0e, 2)[1:-1]}\\,R_\\odot$ with a shock speed of "
    f"$v_{{\\rm sh}} = {_tex(v0, v0e, 0)[1:-1]}$~km~s$^{{-1}}$, both averaged over its full traced "
    f"span. Repeating the analysis over a grid "
    f"of five density models at folds 1--4 shifts the height to "
    f"{np.nanmin(sweep['r_Rsun']):.2f}--{np.nanmax(sweep['r_Rsun']):.2f}~$R_\\odot$ and the speed "
    f"to {np.nanmin(sweep['v_kms']):.0f}--{np.nanmax(sweep['v_kms']):.0f}~km~s$^{{-1}}$, which we "
    f"adopt as the systematic uncertainty. Over the {_w0['split_window']} interval in which both "
    f"branches of the band are present, at $r = {_tex(rS, rSe, 2)[1:-1]}\\,R_\\odot$ and an "
    f"upstream density of {_sci(ne0)}~cm$^{{-3}}$, the implied upstream Alfv\\'en speed is "
    f"$v_A = v_{{\\rm sh}}/M_A = {_tex(vA0, vA0e, 0)[1:-1]}$~km~s$^{{-1}}$ and the coronal magnetic "
    f"field is $B = {_tex(B0, B0e, 2)[1:-1]}$~G "
    f"({np.nanmin(sweep['B_G']):.2f}--{np.nanmax(sweep['B_G']):.2f}~G across the model grid). "
    + _B_context(rS, B0)
)

with open(os.path.join(OUTDIR, 'typeii_results_text.md'), 'w') as fh:
    fh.write(f'# Type II results, NenuFAR {EVENT_DATE}\n\n'
             '## Measured values\n\n```\n' + summary + '\n```\n\n'
             '## Summary paragraph\n\n' + para + '\n')

print('\n' + '=' * 78 + '\nSUMMARY PARAGRAPH\n' + '=' * 78)
print(textwrap.fill(para, width=95))
print('\nsaved', os.path.join(OUTDIR, 'typeii_results_text.md'))

> **Output 2 &mdash; Summary text.** `typeii_results_text.md`
>
> The measured values laid out per band and per lane, followed by a summary paragraph with the
> numbers already substituted, errors quoted to two significant figures with the value matched, and
> `\citet{}` keys in place. Also written: `typeii_picks.pkl`, holding the raw traces, the passes,
> the lane roles and the configuration, so any figure can be regenerated without re-tracing.

## A.11 &middot; Height-time tracks for every density model

&sect;A.6 and &sect;A.8 answer *what the shock did* under one reference model and *how much that
answer moves* across the model grid. This section writes the tracks themselves out, so the radio
height-time curve can be put on the same axes as an EUV or coronagraph track and the comparison can
decide which density model suits this event.

The logic is worth stating, because it is the only leverage available on a quantity the radio data
cannot determine on its own. A type II gives $f(t)$, and $f\rightarrow n_e$ is exact; the step that
is not exact is $n_e\rightarrow r$, which needs an assumed density profile. An independently
measured height &mdash; a front in EUV, a leading edge in a coronagraph &mdash; supplies that missing
information from outside. **It is not proof that the chosen model is correct**, only that it is the
one consistent with the other instruments for this event, which is the strongest statement available.

Three files are written:

| file | contents |
|---|---|
| `height_time_all_models.csv` | $r(t)$ for every lane, every model, every fold, at `EXPORT_DT_S` cadence |
| `height_time_model_summary.csv` | one row per model &times; fold &times; lane: height range, mean speed, acceleration, $B$ |
| `model_independent_scalars.csv` | $X$ and $M_A$ per band &mdash; the quantities that need **no** density model |

### Two height conventions, and why the distinction is not pedantry

Radio heights are almost always heliocentric: $r$ measured from Sun centre, so the photosphere is
at $r=1\,R_\odot$. Coronagraph and EUV heights are often quoted as **height above the limb**,
$h=r-1\,R_\odot$. Mixing them displaces one track against the other by exactly $1\,R_\odot$, which
is **larger than the entire spread between the five density models** &mdash; so a convention slip
would look like a model problem and could be "fixed" by choosing the wrong model.

The export therefore carries `r_heliocentric_Rsun` and `h_above_limb_Rsun` as separate columns and
never a bare "height". Check which convention the comparison track uses before overplotting.

In [ ]:
# ---- r(t) for every lane through every model x fold ----------------------------------------
tg_exp = np.arange(tg.min(), tg.max() + EXPORT_DT_S, EXPORT_DT_S)
rows, summary = [], []

for mf_name, model in tqdm(MODEL_GRID.items(), desc='exporting model x fold'):
    base, fold = mf_name.rsplit(' x', 1)
    agg = aggregate_lanes(passes, tg_exp, model, n_mc=EXPORT_N_MC)
    for lab in TRACED:
        d = agg.get(lab)
        if d is None:
            continue
        # keep only grid points where EVERY realisation contributed. At a partly covered endpoint
        # the mean is taken over a subset of the passes and jumps by more than its own error bar,
        # which shows up as the height track briefly running backwards.
        ok = np.isfinite(d['r_mean'])
        if 'r_n' in d:
            ok &= (d['r_n'] == d['r_n'].max())
        if not ok.any():
            continue
        band = LANE_BAND[lab]
        t_utc = [t0 + pd.Timedelta(seconds=float(x)) for x in tg_exp[ok]]
        rows.append(pd.DataFrame({
            'model': base, 'fold': int(fold), 'model_fold': mf_name,
            'band': band, 'lane': lab, 'role': LANE_ROLE.get(lab, ''),
            'harmonic_s': HARM[band],
            # keep the fractional second: the export grid is anchored on the first traced
            # sample, not on a whole second, so truncating drops a constant ~0.6 s from every row
            'time_UT': [x.strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3] for x in t_utc],
            't_sec_from_window_start': tg_exp[ok],
            'f_MHz': d['f_mean'][ok],
            'r_heliocentric_Rsun': d['r_mean'][ok],
            'r_err_Rsun': d['r_sd'][ok],
            'h_above_limb_Rsun': d['r_mean'][ok] - R_LIMB_RSUN,
            'v_kms': d['v_mean'][ok],
            'B_G': d['B_mean'][ok]}))
        v_, ev_ = grid_scalar(d, 'v')
        a_, ea_ = grid_scalar(d, 'a')
        B_, eB_ = grid_scalar(d, 'B')
        summary.append({
            'model': base, 'fold': int(fold), 'model_fold': mf_name, 'band': band, 'lane': lab,
            'role': LANE_ROLE.get(lab, ''),
            't_start_UT': t_utc[0].strftime('%H:%M:%S'),
            't_end_UT': t_utc[-1].strftime('%H:%M:%S'),
            'r_start_Rsun': d['r_mean'][ok][0], 'r_end_Rsun': d['r_mean'][ok][-1],
            'h_start_above_limb_Rsun': d['r_mean'][ok][0] - R_LIMB_RSUN,
            'h_end_above_limb_Rsun': d['r_mean'][ok][-1] - R_LIMB_RSUN,
            'v_mean_kms': v_, 'v_err_kms': ev_,
            'v_chord_kms': ((d['r_mean'][ok][-1] - d['r_mean'][ok][0]) * R_SUN_M / 1e3
                            / (tg_exp[ok][-1] - tg_exp[ok][0])),
            'a_mean_ms2': a_, 'a_err_ms2': ea_, 'B_mean_G': B_, 'B_err_G': eB_})

track_export = pd.concat(rows, ignore_index=True)
model_summary = pd.DataFrame(summary)
track_export.to_csv(os.path.join(OUTDIR, 'height_time_all_models.csv'), index=False)
model_summary.to_csv(os.path.join(OUTDIR, 'height_time_model_summary.csv'), index=False)

# the two quantities no density model touches, written on their own so they cannot be confused
# with anything that does depend on one
mi = []
for b in BANDS_TRACED:
    s = scalars[b]
    lo, up = SPLIT_PAIR[b]
    mi.append({'band': b, 'band_name': BAND_NAME[b], 'harmonic_s': HARM[b],
               'upstream_lane': lo, 'downstream_lane': up,
               'X': s['X'][0], 'X_err': s['X'][1],
               'X_min_along_overlap': s['X_range'][0], 'X_max_along_overlap': s['X_range'][1],
               'M_A': s['M_A'][0], 'M_A_err': s['M_A'][1],
               'relative_bandwidth': s['rel_bandwidth'][0],
               'relative_bandwidth_err': s['rel_bandwidth'][1],
               # Both drift columns are averaged over THIS BAND'S OWN traced span, which is a
               # different stretch of the burst for F and for H. Do not compare them across bands:
               # the textbook "harmonic drifts twice as fast" applies at a common instant, and the
               # ratio of these two numbers is not 2 because they cover different intervals. The
               # cross-band test belongs on the overlap, and that value is the last column.
               'drift_own_span_MHz_s': s['drift_MHz_s'][0],
               'drift_own_span_err_MHz_s': s['drift_MHz_s'][1],
               'rel_drift_own_span_per_s': s['rel_drift_s'][0],
               'rel_drift_own_span_err_per_s': s['rel_drift_s'][1],
               'rel_drift_over_FH_overlap_per_s': (rdF if b == 'F' else rdH)
               if OVERLAP is not None else np.nan})
model_independent = pd.DataFrame(mi)
model_independent.to_csv(os.path.join(OUTDIR, 'model_independent_scalars.csv'), index=False)

print(f'{len(track_export)} rows over {track_export.model_fold.nunique()} model x fold '
      f'combinations x {track_export.lane.nunique()} lanes, at {EXPORT_DT_S} s cadence')
print('  height_time_all_models.csv      r(t) per lane per model, BOTH height conventions')
print('  height_time_model_summary.csv   one row per model x fold x lane')
print('  model_independent_scalars.csv   X and M_A, no density model involved')
# quote the upstream branches, which is what a height-time comparison should use, and say so -
# taking min/max over the whole table mixes in the downstream branches and widens the speed range
# from 281-903 to 240-903 km/s under a label that says "upstream"
_ups = model_summary[model_summary.role.str.startswith('upstream')]
_lanes = ', '.join(sorted(_ups.lane.unique()))
print(f'\nacross the grid the UPSTREAM lanes ({_lanes}) span '
      f'{_ups.h_start_above_limb_Rsun.min():.2f} to '
      f'{_ups.h_end_above_limb_Rsun.max():.2f} Rsun above the limb, '
      f'mean speed {_ups.v_mean_kms.min():.0f}-{_ups.v_mean_kms.max():.0f} km/s')
print(f'  all four lanes together span {model_summary.h_start_above_limb_Rsun.min():.2f} to '
      f'{model_summary.h_end_above_limb_Rsun.max():.2f} Rsun and '
      f'{model_summary.v_mean_kms.min():.0f}-{model_summary.v_mean_kms.max():.0f} km/s')
print('  these are means over each lane\'s FULL traced span, which is the right window for a '
      'height-time comparison')
model_summary.head(8).round(3)

> **Table 9 &mdash; Height-time tracks for the full model grid.**
> `height_time_all_models.csv`, `height_time_model_summary.csv`, `model_independent_scalars.csv`
>
> $r(t)$ for every traced lane through all five density models at folds 1&ndash;4, at `EXPORT_DT_S`
> cadence, written for overplotting against tracks from other instruments. Both height conventions
> are carried as separate columns: `r_heliocentric_Rsun` measured from Sun centre, and
> `h_above_limb_Rsun` $= r - 1\,R_\odot$. The 1 $R_\odot$ between them exceeds the whole spread
> between density models, so the distinction decides the comparison.
>
> *What to look for:* which model &times; fold brings the radio track closest to an independently
> measured height. That is the only external constraint on a choice the radio data cannot make for
> itself. `model_independent_scalars.csv` is kept separate because $X$ and $M_A$ need no density
> model at all and are unaffected by whatever the comparison concludes.

In [ ]:
# one panel per band: every model x fold track for that band's UPSTREAM lane, which is the shock
# front proper. Plotted as height ABOVE THE LIMB, the convention coronagraph tracks usually use.
fig, axes = plt.subplots(1, 2, figsize=[14, 5.6], sharey=True)
_styles = {1: ':', 2: '-', 3: '--', 4: '-.'}
_cols = {m: c for m, c in zip(BASE_MODELS, plt.cm.tab10.colors)}

for ax, b in zip(axes, BANDS_TRACED):
    lo = SPLIT_PAIR[b][0]
    sub = track_export[track_export.lane == lo]
    for mf, g in sub.groupby('model_fold'):
        base, fold = mf.rsplit(' x', 1)
        ax.plot(pd.to_datetime(g.time_UT), g.h_above_limb_Rsun, lw=1.3,
                color=_cols[base], ls=_styles[int(fold)], alpha=0.85,
                label=(base if fold == '1' else None))
    ref = sub[sub.model_fold == REF_MODEL_NAME]
    if len(ref):
        ax.plot(pd.to_datetime(ref.time_UT), ref.h_above_limb_Rsun, lw=3.2, color='k',
                alpha=0.85, label=f'{REF_MODEL_NAME} (reference)')
    ax.set_title(f'{BAND_NAME[b]} (s = {HARM[b]}), {lo}')
    ax.set_xlabel(f'Time (UT) on {EVENT_DATE}')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r'height above the limb  $r - R_\odot$  [$R_\odot$]')
axes[0].legend(fontsize=8, ncol=2, title='line style = fold 1-4')
fig.suptitle('Type II height-time track under every density model x fold\n'
             'heights are ABOVE THE LIMB; add 1 Rsun for heliocentric r', y=1.02, fontsize=11)
fig.tight_layout()
save_fig(fig, 'height_time_all_models')
plt.show()

> **Figure 10 &mdash; Every density model on one set of axes.**
> `height_time_all_models.png`
>
> The upstream branch of each band, drawn once per model &times; fold: colour is the model, line
> style is the fold, and the reference model is the heavy black line. Heights are **above the
> limb**, matching the convention coronagraph tracks normally use.
>
> *What to look for:* the vertical spread at fixed time is the density-model systematic, drawn
> rather than tabulated. An external height measurement laid on these axes selects the tracks it is
> consistent with. Expect the shock to run ahead of a CME leading edge, so exact coincidence is not
> the test &mdash; a track that is parallel and slightly ahead is the physically sensible outcome.

## A.12 &middot; Conventions and caveats

- **One burst, one shock, two harmonics.** The bands occupy different intervals of the observing
  window only because $f_H=2f_F$ carries the harmonic across the top of the band while the
  fundamental is on its way out of the bottom; they are simultaneous emission from the same shock.
- **Lanes are named, not declared.** Each trace is recorded as `F lane 1`, `H lane 1` and so on in
  the order you make it. &sect;A.4 then sorts the lanes of each band in frequency and takes the
  lowest as the **upstream (unshocked)** branch and the next as the **downstream (compressed)**
  branch, so $X=(f_U/f_L)^2\ge1$. Heights, $n_e$ and $B$ are quoted from the upstream branch. Extra
  lanes of a band are kept and reported but do not enter the split.
- **$X$ and $M_A$ carry no density model** and are measured twice, once from each band's split;
  they should agree, and &sect;A.5 checks that they do. Height, $v_{\rm sh}$, $v_A$ and $B$ all do
  depend on the model, and the 5&times;4 grid in &sect;A.8 is the systematic error on them. Quote
  the reference-model value with the grid range beside it.
- **Every lane is quoted separately; the bands are not combined.** `MAKE_JOINT` is `False`, so
  there is no `F+H joint` track in a default run and nothing in the tables or the summary text is
  an average of the two bands. Setting it `True` builds one &mdash; averaging the two bands'
  *upstream* heights where they overlap and taking whichever exists elsewhere &mdash; which buys
  the longer time baseline a second derivative wants, at the cost of merging two independent
  measurements. The two bands add no independent constraint on the density model either way, since
  $f_H/2$ and $f_F$ map to the same density.
- **$B$ and $v_A$ are quoted for the upstream branch only.** $v_A=v_{\rm sh}/M_A$ from
  Rankine-Hugoniot is the *upstream* Alfv&eacute;n speed, so pairing it with the downstream
  branch's density would give $B_1\sqrt{X}$ &mdash; neither the upstream field nor the
  downstream one.
- **The Rankine-Hugoniot form assumes cold upstream plasma**, $\beta=0$, as well as a
  perpendicular shock and $\gamma=5/3$. Finite $\beta$ lowers $M_A$ at fixed $X$ and so lowers
  $B$: $\beta=0.1$ costs about 4%.
- **$M_A$ assumes a perpendicular shock** with $\gamma=5/3$. That is the standard band-split
  assumption but it is an assumption; an oblique geometry lowers the inferred $M_A$ at fixed $X$.
- **Lane fits are in $\log_{10}f$** (`FIT_IN_LOGF`). Over an octave of frequency a quadratic in $f$
  is much too stiff and biases the drift rate and the split; the relative drift then follows as
  $\ln 10\,\mathrm{d}\log_{10}f/\mathrm{d}t$.
- **Every quantity is quoted on one of two windows, and each row says which.** A lane's **full
  traced span** is where $r$, $v_{\rm sh}$, $a$ and $n_e$ belong: that is the measurement of the
  burst. $X$ and $M_A$ need both branches of a band and therefore exist only on the **band-split
  window** where the branches overlap in time, and so do $v_A=v_{\rm sh}/M_A$ and $B$. The two are
  not interchangeable: on this event `F lane 1` averages 628&nbsp;km&nbsp;s$^{-1}$ over its whole
  596&nbsp;s and 771 over the 214&nbsp;s split window, because it is accelerating. Every table
  repeats $r$, $v_{\rm sh}$ and $n_e$ on the split window beside $v_A$ and $B$, so a reader can
  check $v_A=v_{\rm sh}/M_A$ from the printed numbers. `lane_windows()` computes both, and &sect;A.4,
  &sect;A.9 and &sect;A.10 all call it rather than each deriving its own.
- **Error bars combine two sources**: the spread between your repeats of a lane and the polynomial
  coefficient covariance. `_sd` is the total spread; `_se` divides it by $\sqrt{N_{\rm pass}}$
  **only when `REPEATS_INDEPENDENT` is true**, which requires the repeats to have been placed
  independently. Auto-jittered B&eacute;zier repeats are one curve with its control points moved,
  so &sect;A.3 sets that flag false and no $\sqrt{N}$ division happens &mdash; which is the case for
  this event. Tracing a lane three times by hand, with auto-repeats unticked, is what buys a real
  reproducibility term; tracing it three times by jitter does not.
- **$\sigma_f$ is an assumed scale, not a measured one.** It is the rms offset between a trace and
  the brightest pixel within $\pm$`half` channels, and a brightest-pixel search has no intrinsic
  width: &sect;A.3 sweeps the window and the rms grows with it. Every statistical error bar here is
  proportional to $\sigma_f$; no central value is. The control in &sect;A.3 &mdash; repeating the
  measurement with each lane displaced onto blank spectrum &mdash; is what establishes that the
  traces follow emission at all.
- **The relative drift on the F/H overlap is measured on the traced samples, not on the lane
  fits.** The overlap is the last third of the fundamental's fit and the first eighth of the
  harmonic's, where a polynomial's derivative is least constrained; taking it from the fits opened
  a 15% gap between two bands of the same shock. The fit-derived value is still printed, labelled
  as a diagnostic.
- **Acceleration is in m&nbsp;s$^{-2}$** everywhere (speed in km&nbsp;s$^{-1}$, height in
  $R_\odot$). It is a second derivative of a fitted curve, so &sect;A.7 exists precisely to show how
  much of it depends on the fitting choice.
- **Kinematics are gated**, but on the lane rather than on the shared grid: a model must resolve a
  height over more than `KIN_MIN_FRAC` of **that lane's own traced span** and supply at least
  `KIN_MIN_PTS` points, and only outward speeds below 3000&nbsp;km&nbsp;s$^{-1}$ are kept. The
  derivatives are blanked outside the span each track was actually traced over. Model&times;fold
  combinations that put the source at or below the surface drop out rather than producing a
  spurious number. There is deliberately **no cut on traced duration**: `kin_degree()` is set by
  the point count alone, and whether a curvature counts as measured is decided afterwards by
  `accel_is_measured()` against its own error bar and the method's bias floor. Gating on duration
  made a four-second difference decide whether a lane had an acceleration at all.
- **The "bulk exciter energy"** is $(\gamma-1)m_ec^2$ evaluated at the shock speed. For a type II
  this is the kinetic energy of an electron moving with the shock, not the energy of the
  beam-accelerated electrons that produce the emission, and it is firmly non-relativistic.
- **Polarisation is reported, and on this event it is not evidence of anything.** Fundamental
  emission escapes in the o-mode and is expected to be the more strongly circularly polarised of
  the two, but that expectation can only be tested where the contrast clears the noise. &sect;A.5
  compares the band-to-band difference against the point-to-point scatter along a lane, undivided
  by $\sqrt{N}$, and requires a factor of two before calling it a contrast; here it reaches 1.3.
  `polarisation_caveats()` separately withdraws the claim when two lanes agree far more closely
  than either is individually determined, the signature of a common instrumental offset rather than
  of lane-specific polarisation &mdash; which is what happens here. The F/H identification
  therefore rests on the frequency ratio, the relative drift and the height agreement, and not on
  polarisation at all. Note also that V/I is a ratio that has already been formed: averaging it
  where Stokes&nbsp;I is near background pulls the result toward the noise mean, so these values
  are an upper bound on a contrast rather than a detection.
- **Display stretch.** `LAYER_PLO` clips the bottom of the colour scale and `LAYER_GAMMA` stretches
  the faint end; the comparison figure in &sect;A.1 exists because how much fine structure you can
  see, and therefore trace, depends on both. Nothing is averaged in frequency.
- **Fine structure** (herringbones, striae) can be traced as extra lanes of a band, but the fits
  are low-order curves through whatever you click, so they describe the envelope rather than the
  individual striae. Zoom in with the toolbar before tracing them, and raise `LAYER_MAX_T` if the
  time decimation is smoothing them away.
- **All plots use `pcolormesh`**, never `imshow`, and no tick labels are rotated.

**Outputs** (in `OUTDIR`): `typeii_characteristics.csv` / `.tex`, `typeii_polarisation.csv`,
`fundamental_harmonic_tests.csv`, `height_time_fit_comparison.csv`, `model_grid_sweep.csv`,
`typeii_results_text.md`, `typeii_picks.pkl`, and every figure as a 300&nbsp;dpi PNG.

## A.13 &middot; Methods, in the order they are applied

A compact description of the whole chain, in the order the steps are applied.

**Observations and pre-processing.** NenuFAR high-resolution Stokes&nbsp;I and V/I dynamic spectra
are used. Stokes&nbsp;I is converted to decibels on load; the tracing layer is that frame minus the
time median of each frequency channel, which removes the instrumental bandpass and the quiet-Sun
background and leaves the burst as an excess above zero. The layer is restricted to the type II
window and block-averaged in time only, never in frequency, since the band splitting and fine
structure are frequency-domain features.

**Lane tracing.** The emission lanes are traced by hand on a single rendering of the dynamic
spectrum, either by clicking points along a lane or by placing a B&eacute;zier curve on it with
sliders. Each lane is traced several times so the spread between repeats supplies a reproducibility
error; repeats produced by jittering one B&eacute;zier's control points are recorded as such,
because their spread measures that jitter rather than the reproducibility of the tracing. Densely
sampled curves are thinned to one point per `FIT_MIN_DT_S` seconds so that correlated samples do
not inflate the apparent information content.

Each traced point is assigned a frequency uncertainty $\sigma_f$ equal to the rms offset between
the trace and the brightest pixel within a fixed search window of $\pm$`half` channels. This is an
assumed scale rather than a measured one, and the notebook says so where it is computed: a
brightest-pixel search carries no intrinsic width, and on a spectrum carrying this much fine
structure the rms offset keeps growing as the window is widened, so $\sigma_f$ reflects the window
as much as the emission. Every statistical uncertainty quoted here is proportional to it, while no
central value depends on it at all. What the window sweep does establish, by repeating the same
measurement with each lane displaced onto blank spectrum, is that the offsets are smaller on the
lanes than off them &mdash; that is, the traced curves are following emission and not noise.

**Lane fits.** Each lane is fitted with a low-order polynomial in $\log_{10}f$ against time, the
coefficient covariance being computed from the assigned frequency uncertainties rather than from
the residual scatter. Log space is used because a type II lane spans close to an octave, over which
a polynomial in $f$ is too stiff. The drift rate follows analytically as
$\dot f=f\ln10\;\mathrm{d}\log_{10}f/\mathrm{d}t$, and the relative drift $\dot f/f$ is independent
of the harmonic number and therefore directly comparable between bands.

**Band identification.** The lanes of each band are ordered by frequency over the interval in which
they overlap in time, the lower being taken as the upstream (unshocked) branch of the split. The
fundamental/harmonic assignment is then tested against the data: the frequency ratio over the
overlap must be 2, the relative drifts must agree, the two bands must map to the same height, and
the degree of circular polarisation is compared between them.

**Shock diagnostics.** The band split gives the density jump $X=(f_U/f_L)^2$ and, through the
Rankine-Hugoniot relation for a perpendicular shock with $\gamma=5/3$,
$M_A=\sqrt{X(X+5)/[2(4-X)]}$. Neither involves a density model, and each band provides an
independent determination. Frequencies are converted to local electron density through
$n_e=(f/s)^2/\mathrm{const}$ with $s$ the harmonic number, and to height by numerically inverting
an assumed $n_e(r)$; out-of-range densities return no height rather than an extrapolation.
Differentiating $r(t)$ gives the shock speed and acceleration, and
$v_A=v_{\rm sh}/M_A$, $B=v_A\sqrt{\mu_0\rho}$ with $\rho=\mu m_p n_e$ complete the chain.

**Uncertainties.** Statistical errors are propagated by Monte Carlo: for each traced repeat, the
lane's polynomial coefficients are drawn many times from their covariance and each draw is pushed
through the entire chain, so the fitting and reproducibility errors combine. The height-time fits
carry a separate parametric bootstrap in which every height is displaced by a draw from its own
$1\sigma$ before refitting. Systematic error is assessed by recomputing everything for five
electron-density models at folds 1&ndash;4; for height-dependent quantities that systematic exceeds
the statistical error by one to two orders of magnitude and is the one that should be quoted.

**Checks applied before any number is used.** Traces are verified to sit on emission rather than on
blank spectrum; quantities that must agree are compared with their combined errors and flagged
beyond a set significance; accelerations are reported only where the traced baseline can constrain
a curvature, and are labelled as model-dominated; and each height-time model is tested for whether
its functional form can represent the track at all before its kinematics are quoted.